# CIFAR-10 ResNet under 5M parameters: Scheme E student with knowledge distillation (T05)

NYU Tandon Deep Learning (Fall 2026), Project 1 ([Kaggle competition](https://www.kaggle.com/competitions/cifar-10-res-net-architecture-challenge-dl)).
A residual network is trained **from scratch** on the 50,000 provided CIFAR-10 training images and predicts the
2,000 hidden test images. No pretrained weights and no external data are used. The frozen teachers are earlier
models of this project, trained from scratch on the same 45k training split; they are not submitted.

| | |
|---|---|
| Student (submitted) | Scheme E, PyramidNet-110 basic, α = 98, exactly as `E_pyramid.ipynb` (section 4 is a verbatim copy): **4,963,991** trainable parameters (budget 5,000,000), random initialization |
| Teachers (not submitted) | `KD_TEACHERS` in section 1b, any A/B/C/E/F/G checkpoint of this project; placeholder default E ver-1/T18 + C ver-2 until Q08's ensemble-diversity analysis selects them. Rebuilt from their saved architecture with verbatim copies of the six model definitions, checked (dataset, split, architecture, fingerprint, parameter count, SHA-256, reproduced clean validation accuracy) and frozen in eval mode |
| Objective | `L = α · T² · KL(p_teacher^T ‖ p_student^T) + (1 − α) · CE(student, label)`, α = 0.9, T = 2; `p_teacher^T` = mean of the teachers' softmax(logits/T), also over the flipped batch (`KD_FLIP_AVERAGE`); no label smoothing |
| Training | E ver-1/T18's recipe (R0, 300 epochs): SGD Nesterov 0.9, lr 0.1, batch 128, weight decay 5e-4, 5 warmup epochs + cosine, EMA 0.999, crop + flip + Cutout 16, bf16 on an A100. Each batch is augmented once; teachers and student see the identical tensor; teacher outputs are computed on the fly |
| Headline (pre-declared) | Final epoch, EMA weights, flip TTA on the 5k validation split; best/final × raw/EMA × plain/flip are also saved |
| Other notebooks | `E_pyramid.ipynb` (same student without distillation), A/B/C/F/G, `Q08_shiftval.ipynb`. Apart from this cell, sections 1/1b, section 5b, the trainer cell (marked `KD:`) and section 7b, every cell is identical to `E_pyramid.ipynb` |

**Usage.** Set the teachers in section 1b (see `README.md`), then *Run all* on a Colab A100. Use `RESUME_RUN` to continue
an interrupted run; resuming with another teacher set or distillation setting is refused.

**Sections.** 1 Configuration · 1b Teachers · 2 Environment · 3 Data · 4 Model · 5 Training · 5b Distillation ·
6 Training curves · 7 Prediction and submission · 7b T05 headline · 8 Final results

**Main references.** Hinton, Vinyals & Dean, *Distilling the Knowledge in a Neural Network*, 2015 (soft targets,
T² scaling); Han, Kim & Kim, *Deep Pyramidal Residual Networks*, CVPR 2017; DeVries & Taylor, *Cutout*, 2017.
The full list is in `README.md`.

## 1. Configuration

Edit this cell and the teacher cell (section 1b), then Run all. Every new experiment receives a New York timestamp and
a separate artifact tree. Set `RESUME_RUN` to the exact previous run ID to resume; changing a training or distillation
setting during resume is rejected. Leave `RUN_NAME=None` when resuming. There is no destructive overwrite option.

**T05 preset.** Everything except the objective follows E ver-1/T18 (R0 with 300 epochs): `EPOCHS=300`, `LR=0.1`,
`BATCH_SIZE=128`, `WEIGHT_DECAY=5e-4`, `WARMUP_EPOCHS=5`, `EMA_DECAY=0.999`, `AUG="cutout"`, `SEED=0`,
`SPLIT_SEED=0`, `VAL_SIZE=5000`, `MIXUP=False`, `DROP_PATH=0.0`, `AMP="auto"` (bf16 on an A100).
`LABEL_SMOOTHING=0.0` is required: the soft teacher target replaces R0's smoothing 0.1, so T05 differs from T18 in
the objective only. The pre-declared headline is the final epoch with EMA weights and flip TTA, so the submission
uses `CHECKPOINT="last"`, `WEIGHTS="ema"`, `TTA=True`; the best checkpoint and raw weights are still evaluated.

A two-epoch smoke test needs `EPOCHS=2`, `WARMUP_EPOCHS=1` and `TRAIN_SUBSET >= BATCH_SIZE` (for example 5120); the
teachers are still loaded and verified on the full 5k validation split. Smoke-test accuracy is not a result.

Names use `MM-DD-HH-mm_E_pyramid-ver-<version>-T05-<features>_<role>.<extension>` in `America/New_York`: a distilled
student is a scheme E model, so it shares E's notebook stem and version sequence (`MODEL_VERSION=None` takes the next
unused E version in this result root). The feature words must start with `distill` or `kd`; the default is
`distill-ema` (`distill-mixup` with Mixup). Resume uses the exact existing `RESUME_RUN` and does not create a version.
The student checkpoint keeps E's model fingerprint, so `E_pyramid.ipynb`'s evaluation code and `Q08_shiftval.ipynb`
can load it, while E's trainer refuses to resume it (the distillation settings differ).

In [ ]:
# T05 knowledge distillation: scheme E student (PyramidNet-110 basic, alpha=98, section 4) trained from scratch
# against frozen teachers (section 1b). Edit the values below and section 1b, then Run all.

RUN_NAME = None          # optional 1-2 feature words starting with distill or kd; default: distill-ema
RESUME_RUN = None        # exact saved run ID to resume, e.g. "10-30-08-40_A_resnet-ver-1-T00-cutout-ema"
EPOCHS = 300             # E ver-1/T18's 300-epoch schedule (warmup + cosine built for 300)
AUG = "cutout"           # "basic": crop + flip | "cutout": basic + Cutout | "autoaug": basic + AutoAugment + AUTOAUG_ERASE
SEED = 0                 # model initialization, data order and augmentation draws
VAL_SIZE = 5000          # class-balanced validation hold-out (multiple of 10); 0 = train on all 50,000 images
EMA = True               # also keep (and evaluate) an exponential moving average of the weights
AMP = "auto"             # CUDA only: "auto" = bf16 on GPUs with native bf16 (e.g. A100), else fp16 | "off" | "bf16" | "fp16"
CKPT_EVERY = None        # write last-ckpt.pt every N epochs; None = 5 on Colab (Drive writes are slow), else 1
TRAIN_SUBSET = 0         # > 0: train on this many images only (quick tests; multiple of 10)
# New runs never overwrite previous outputs. Set RESUME_RUN explicitly to continue a run.

# Optimization
BATCH_SIZE = 128
LR = 0.1
MOMENTUM = 0.9
NESTEROV = True
WEIGHT_DECAY = 5e-4
WD_SKIP_BN_BIAS = False  # True: no weight decay on BN parameters and biases
WARMUP_EPOCHS = 5
LABEL_SMOOTHING = 0.0    # distillation: R0's 0.1 is replaced by the soft teacher target; must stay 0.0
EMA_DECAY = 0.999

# Data and augmentation
SPLIT_SEED = 0           # seed of the train/validation split; keep it fixed so runs are comparable
CUTOUT_SIZE = 16
AUTOAUG_ERASE = "cutout" # occlusion after AutoAugment: "cutout" | "erasing" (RandomErasing) | "none"
ERASING_P = 0.5          # RandomErasing probability

# Training-only Mixup option (P13): off for T05. When on, teachers and student see the same mixed batch (KD target =
# teacher output on the mixed image, hard CE on both labels); the shared Mixup rule also requires AUG="basic".
MIXUP = False            # True enables P13 batch Mixup (with AUG="basic" and LABEL_SMOOTHING=0.0)
MIXUP_ALPHA = 0.2        # positive Beta distribution parameter; one lambda per triggered batch
MIXUP_PROB = 0.5         # probability of mixing a training batch; validation/test never mix

# Hardware (None = automatic: on for CUDA, off for MPS / CPU)
DEVICE = None            # "cuda" | "mps" | "cpu"; None = $CIFAR_DEVICE, else cuda > mps > cpu
CHANNELS_LAST = None
CUDNN_BENCHMARK = None
PIN_MEMORY = None
WORKERS = None           # DataLoader workers for AUG = "autoaug" (always 0 on macOS / Windows, see section 2)
EVAL_BATCH_SIZE = 1000
LOG_INTERVAL = 0         # also print every N steps within an epoch (0 = off)
# Optional speed switches (README "Compilation"); neither is part of the recipe or of any fingerprint.
COMPILE = "off"          # torch.compile on CUDA: "off" | "default" | "reduce-overhead" | "max-autotune" | "max-autotune-no-cudagraphs"
BENCHMARK_STEPS = 0      # > 0: only time this many training steps (real data, current COMPILE); no run folder; then stop

# Prediction
# Pre-declared T05 rule: the final-epoch EMA weights with flip TTA are the headline and the submission;
# best/final x raw/EMA x plain/flip are still evaluated and saved for comparison.
CHECKPOINT = "last"      # checkpoint used for the submission: "last" (final epoch) | "best" (highest validation accuracy)
WEIGHTS = "ema"          # "ema" | "model" | "auto" (choose on validation using TTA)
TTA = True               # average the softmax of each test image and its horizontal flip

# Runtime host, independent of the editor: auto | colab | kaggle | local.
# For a VS Code kernel connected to Colab, set "colab" if auto detection is ambiguous.
RUNTIME_ENV = "auto"     # explicit value overrides CIFAR_ENV; auto uses CIFAR_ENV then detection

# Paths (None = $CIFAR_DATA_DIR / $CIFAR_CACHE_DIR / $CIFAR_RESULT_DIR, else the default of the environment)
AUTO_DOWNLOAD = False     # Kaggle download/login code is commented out; use uploaded data
SAVE_CACHE_TO_DRIVE = True # on Colab, automatically save/verify MyDrive/DL_Project1/cache.zip
DATA_DIR = None          # Colab: MyDrive/DL_Project1/data/ (folder or override with a ZIP path)
CACHE_DIR = None
RESULT_DIR = None
SHOW_SUMMARY = True      # torchinfo layer table in section 4 (skipped if torchinfo is not installed)

# Architecture switches (requirement 15): the student is scheme E; stochastic depth is its only switch; off for T05.
# A later E variant (e.g. E + IBN) is plugged in by copying E_pyramid.ipynb's updated section 4 cells here and
# adding its switch below; tests/test_distill.py keeps section 4 identical to E_pyramid.ipynb.
TASK_ID = "T05"          # checklist task
MODEL_VERSION = None     # assign next sequential version at actual launch; no advance reservations
DROP_PATH = 0.0          # scheme E stochastic depth: block l of 54 drops its residual branch with prob DROP_PATH*l/54; 0 = off

### 1b. Teachers and distillation loss

List the teachers in `KD_TEACHERS`. **The default list is a placeholder** (E ver-1/T18 and C ver-2, both in
`MyDrive/DL_Project1/result/checkpoints/`); the final set comes from the ensemble-diversity analysis in section 8 of
`Q08_shiftval.ipynb`. By default each teacher uses the checkpoint and raw/EMA weights of its recorded submission CSV;
set `"checkpoint"` and `"weights"` to override. A teacher in another Drive folder is given by `"root"` (its `result`
folder) or by an explicit checkpoint `"path"`.

Before the first training step every teacher is rebuilt exactly as the resume/evaluation code rebuilds a checkpoint
(dataset fingerprint, model family, saved architecture options, resolved topology, model fingerprint, parameter count),
must use this student's 5k validation split, must not have trained on it, and must reproduce every stored clean
validation accuracy of that epoch and weights (training log row, validation export, prediction report) within
`KD_REPRO_TOLERANCE_PP`. Its checkpoint SHA-256, role, epoch and weights form the recipe identity together with
`KD_ALPHA`, `KD_TEMPERATURE`, `KD_FLIP_AVERAGE` and the Mixup settings.

In [ ]:
# ==== 1b. Distillation teachers and loss (T05). Edit here, then Run all. ====
# PLACEHOLDER TEACHER SET: the final teachers are chosen from Q08_shiftval.ipynb's ensemble-diversity analysis
# (section 8). Until then this list is the suggested default; replace it with the selected run IDs before launching.
#
# Each entry is a run ID string or a dict with these keys (only "run" or "path" is required):
#   "run":        exact teacher run ID (legacy IDs are translated). Its checkpoint is read from
#                 <root>/checkpoints/<run>/<run>_<checkpoint>-ckpt.pt
#   "root":       result folder holding that run; only this folder is then searched. Default: KD_TEACHER_ROOT
#                 (else this notebook's RESULT_DIR), then each KD_EXTRA_TEACHER_ROOTS folder, first match wins
#   "path":       explicit checkpoint file instead of run/root/checkpoint
#   "checkpoint": "best" | "last"; default: the checkpoint of the run's recorded submission CSV
#   "weights":    "model" (raw) | "ema"; default: the weights of the run's recorded submission CSV
# The recorded submission is read from <root>/evaluations/test/<run>/*_prediction-report.json (first searched root
# that has it). If it is missing or
# ambiguous (e.g. an explicit "path" outside a result folder), set "checkpoint"/"weights" explicitly.
# Every teacher must have trained on the same 45k split (checked), must reproduce its stored clean validation
# accuracy within KD_REPRO_TOLERANCE_PP before training starts, and is frozen (eval mode, no grad). Teachers are
# not submitted, so they need not be E; any A/B/C/E/F/G checkpoint of this project works.
KD_TEACHERS = [
    "10-30-21-05_E_pyramid-ver-1-T18-basic110-ema",  # E ver-1/T18: submission = best251 EMA (flip 97.22%)
    "10-30-09-50_C_seresnet-ver-2-T00-cutout-ema",   # C ver-2: submission = best191 raw (flip 97.00%)
]
KD_TEACHER_ROOT = None       # default result folder of the teacher runs; None = RESULT_DIR (Colab: MyDrive/DL_Project1/result)
# Read-only result folders searched after KD_TEACHER_ROOT for teachers without their own "root" (checkpoints and their
# prediction/validation records). Example for H.W.'s shared folder, added to MyDrive as the shortcut HW_DL_Project1:
#   KD_EXTRA_TEACHER_ROOTS = ["/content/drive/MyDrive/HW_DL_Project1/result"]
KD_EXTRA_TEACHER_ROOTS = []
KD_EXTRA_ROOT_WAIT_S = 60    # Colab only: a Drive shortcut can appear a little after mounting; wait this long
KD_ALPHA = 0.9               # weight of the soft term: alpha*T^2*KL(teacher || student) + (1 - alpha)*CE(hard labels)
KD_TEMPERATURE = 2.0         # softmax temperature T for teachers and student
KD_FLIP_AVERAGE = True       # each teacher also sees the horizontally flipped batch; probabilities are averaged
KD_REPRO_TOLERANCE_PP = 0.10 # allowed |reproduced - stored| clean validation accuracy, percentage points
KD_COMPILE_TEACHERS = "off" # torch.compile of the teachers' training-step forward (CUDA only): same values as COMPILE;
                            # speed only, not part of the recipe fingerprint (README "Compilation")

## 2. Environment, paths and device

| Environment | Data | Cache | Results |
|---|---|---|---|
| Local | `project1/data` | `project1/cache` | `project1/result`, beside `Archive` |
| Colab | `MyDrive/DL_Project1/data` (manual upload), or explicit `DATA_DIR` | `/content/cache` | `MyDrive/DL_Project1/result` |
| Kaggle | Competition directory under `/kaggle/input` | `/kaggle/working/cache` | `/kaggle/working/result` |

Set `DATA_DIR`, `CACHE_DIR`, `RESULT_DIR` explicitly when needed. Their `CIFAR_*_DIR` environment variables
are fallback overrides. On Colab select an A100 GPU. Automatic precision uses bf16 on an A100;
channels-last and cuDNN benchmarking are enabled. AutoAugment uses forked workers on Linux and zero workers
on macOS/Windows. Keep the same hardware settings across architecture comparisons.

Artifacts are grouped by type under `result`: `checkpoints`, `configs`, `logs`, `evaluations/validation`,
`evaluations/test`, `plots`, `progress`, `manifests`, and `exports`. Each uses the same timestamped run ID.
The local default assumes the kernel starts in `project1` or `project1/Archive`; otherwise set paths explicitly.

Detection follows the remote Python kernel, even when the editor is VS Code. Runtime markers take priority
over directory names. Set `RUNTIME_ENV="colab"` in section 1 to explicitly select Colab paths; this overrides
`CIFAR_ENV`. It does not connect or move a local kernel to Colab. Restart the kernel and rerun from section 1
after changing it. Check the printed environment, kernel executable, GPU and paths.


In [ ]:
import copy
import csv
import hashlib
import importlib.util
import importlib.metadata
import io
import json
import math
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import time
import tempfile
import zlib
import zipfile
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from datetime import datetime
from zoneinfo import ZoneInfo
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from matplotlib.ticker import MaxNLocator
from PIL import Image

ENVS = ("local", "colab", "kaggle")
KAGGLE_SLUG = "cifar-10-res-net-architecture-challenge-dl"
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/DL_Project1"
WORK_DIRS = {"colab": "/content", "kaggle": "/kaggle/working"}
COLAB_CKPT_EVERY = 5  # Drive writes are slow; best-ckpt.pt and the final epoch are always saved
PROJECT_DIR = Path.cwd().resolve()
LOCAL_PROJECT_DIR = PROJECT_DIR.parent if PROJECT_DIR.name == "Archive" else PROJECT_DIR
# Start method for the AutoAugment DataLoader workers: "fork" lets them use the Dataset class defined in this
# notebook. Under "spawn" (macOS, Windows) they cannot import it, so the pipeline runs in the main process there.
MP_CONTEXT = "fork" if sys.platform.startswith("linux") else None

_warned = set()


def _getenv(name):
    value = os.environ.get(name, "").strip()
    return value or None


def warn_once(msg):
    if msg not in _warned:
        _warned.add(msg)
        print(f"Warning: {msg}", flush=True)


def _module_exists(name):
    try:
        return importlib.util.find_spec(name) is not None
    except (ImportError, ValueError):
        return False


def detect_env():
    """Detect the kernel host, independent of the notebook editor (browser or VS Code)."""
    configured = str(globals().get("RUNTIME_ENV") or "auto").strip().lower()
    if configured not in ("auto",) + ENVS:
        raise ValueError(f"RUNTIME_ENV={configured!r}; expected auto or one of {ENVS}")
    requested = configured if configured != "auto" else (_getenv("CIFAR_ENV") or "auto").lower()
    if requested not in ("auto",) + ENVS:
        raise ValueError(f"CIFAR_ENV={requested!r}; expected auto or one of {ENVS}")
    if requested != "auto":
        return requested
    # Runtime markers beat directory names: /kaggle/input may also exist on Colab.
    if any(_getenv(key) for key in ("COLAB_RELEASE_TAG", "COLAB_BACKEND_VERSION")):
        return "colab"
    if _getenv("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    if _module_exists("google.colab"):
        return "colab"
    # A directory alone is weak evidence; use the full layout only as a fallback.
    if os.path.isdir("/kaggle/input") and os.path.isdir("/kaggle/working"):
        return "kaggle"
    return "local"


def work_dir(env=None):
    env = env or detect_env()
    return Path(_getenv("CIFAR_WORK_DIR") or WORK_DIRS.get(env, PROJECT_DIR))


def drive_dir():
    return Path(_getenv("CIFAR_DRIVE_DIR") or DRIVE_PROJECT_DIR)


def drive_mounted():
    return drive_dir().parent.is_dir()  # .../MyDrive exists once Drive is mounted


def kaggle_input_dir():
    return Path(_getenv("CIFAR_KAGGLE_INPUT") or "/kaggle/input")


def find_competition_data(root=None, max_depth=6):
    """Directories under `root` that contain train_labels.csv, best match first.

    Breadth-first, follows symlinks, skips *_images folders (50k files) and hidden folders.
    """
    root = Path(root or kaggle_input_dir())
    found, frontier = [], [(root, 0)]
    while frontier:
        d, depth = frontier.pop(0)
        try:
            entries = sorted(os.scandir(d), key=lambda e: e.name)
        except OSError:
            continue
        if any(e.name == "train_labels.csv" and e.is_file() for e in entries):
            found.append(d)
        if depth < max_depth:
            frontier += [(Path(e.path), depth + 1) for e in entries
                         if e.is_dir() and not e.name.endswith("_images") and not e.name.startswith(".")]

    def rank(d):
        complete = (d / "test.csv").is_file() and (d / "sample_submission.csv").is_file()
        return (not complete, KAGGLE_SLUG not in str(d), len(d.parts), str(d))

    return sorted(found, key=rank)


@dataclass
class Paths:
    env: str
    data_dir: Path
    cache_dir: Path
    result_dir: Path


def resolve_paths(data_dir=None, cache_dir=None, result_dir=None, env=None):
    """Data, cache and runs directories: argument > environment variable > environment default."""
    env = env or detect_env()
    data = data_dir or _getenv("CIFAR_DATA_DIR")
    cache = cache_dir or _getenv("CIFAR_CACHE_DIR")
    runs = result_dir or _getenv("CIFAR_RESULT_DIR")
    if env == "local":
        data = data or LOCAL_PROJECT_DIR / "data"
        cache = cache or LOCAL_PROJECT_DIR / "cache"
        runs = runs or LOCAL_PROJECT_DIR / "result"
    elif env == "colab":
        cache = cache or work_dir(env) / "cache"
        data = data or drive_dir() / "data"  # manually uploaded ZIPs or extracted competition files
        if runs is None:
            if drive_mounted():
                runs = drive_dir() / "result"
            else:
                runs = work_dir(env) / "result"
                warn_once(f"Google Drive is not mounted ({drive_dir().parent} missing); writing results to "
                          f"{runs}, which is lost when the runtime is recycled")
    else:  # kaggle
        cache = cache or work_dir(env) / "cache"
        runs = runs or work_dir(env) / "result"
        if data is None:
            found = find_competition_data()
            if found:
                data = found[0]
            else:
                data = work_dir(env) / "data"
                warn_once(f"no train_labels.csv found under {kaggle_input_dir()}; checking cache or manually supplied data")
    return Paths(env, Path(data), Path(cache), Path(runs))


def default_ckpt_every(env=None):
    return COLAB_CKPT_EVERY if (env or detect_env()) == "colab" else 1


def available_cpus():
    try:
        return len(os.sched_getaffinity(0))
    except AttributeError:  # macOS
        return os.cpu_count() or 1


def default_workers(device_type):
    """DataLoader workers (used by the per-image AutoAugment pipeline only)."""
    n = available_cpus()
    if device_type == "cuda":
        return min(8, max(2, n - 1))
    return min(4, max(1, n // 2))  # 4 on the 12-core M4 Pro, as before


def get_device(name=None):
    """`name` (DEVICE), else $CIFAR_DEVICE, else cuda > mps > cpu."""
    name = name or _getenv("CIFAR_DEVICE") or "auto"
    if name == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    return torch.device(name)


def cuda_bf16_native(device):
    return torch.cuda.get_device_capability(device) >= (8, 0)  # Ampere or newer (L4, A100, ...)


def resolve_amp(requested, device):
    """'bf16', 'fp16' or 'off'. requested: auto | on | off | bf16 | fp16 (bools accepted)."""
    req = {True: "on", False: "off", None: "auto"}.get(requested, requested)
    req = str(req).lower()
    if req == "off":
        return "off"
    if device.type != "cuda":
        if req != "auto":
            # MPS: fp16 autocast gave no speed-up on an M4 Pro, and torch 2.5.1 GradScaler fails on MPS.
            print(f"Warning: AMP is only used on CUDA; training in fp32 on {device.type}.")
        return "off"
    if req == "fp16":
        return "fp16"
    if cuda_bf16_native(device):
        return "bf16"
    if req == "bf16":
        print("Warning: this GPU has no native bf16; using fp16 autocast + GradScaler instead.")
    return "fp16"


# torch.compile modes for the training step (README "Compilation"); a speed option, never part of the recipe.
COMPILE_MODES = ("off", "default", "reduce-overhead", "max-autotune", "max-autotune-no-cudagraphs")


def resolve_compile(requested, device, name="COMPILE"):
    """One of COMPILE_MODES. Compilation is used on CUDA only; elsewhere it warns and runs uncompiled ("off")."""
    if not isinstance(requested, str) or requested not in COMPILE_MODES:
        raise ValueError(f"{name} = {requested!r}; choose one of {COMPILE_MODES}")
    if requested != "off" and device.type != "cuda":
        print(f"Warning: {name}={requested!r} is only used on CUDA; running uncompiled on {device.type}.")
        return "off"
    return requested


def resolve_hardware(args, device):
    """Fill in the automatic (None) speed settings on `args` for `device`, in place."""
    cuda = device.type == "cuda"
    args.amp = resolve_amp(getattr(args, "amp", "auto"), device)
    args.compile = resolve_compile(getattr(args, "compile", "off"), device)
    if args.channels_last is None:
        args.channels_last = cuda
    elif args.channels_last and device.type == "mps":
        # torch 2.5.1 MPS: backward through conv/BN/ReLU in channels_last raises a stride error.
        print("Warning: CHANNELS_LAST is not supported on MPS with this PyTorch; disabling it.")
        args.channels_last = False
    if getattr(args, "cudnn_benchmark", None) is None:
        args.cudnn_benchmark = cuda
    if getattr(args, "pin_memory", None) is None:
        args.pin_memory = cuda
    if getattr(args, "workers", None) is None:
        args.workers = default_workers(device.type)
    if MP_CONTEXT is None:
        args.workers = 0  # see MP_CONTEXT
    return args


def device_name(device):
    if device.type == "cuda":
        props = torch.cuda.get_device_properties(device)
        return f"cuda ({props.name}, {props.total_memory / 2**30:.0f} GB)"
    if device.type == "cpu":
        return f"cpu ({available_cpus()} cores)"
    return f"{device.type} ({platform.machine()})"


def _on(x):
    return "on" if x else "off"


def summary_line(paths, device=None, args=None):
    """One line: environment, device, torch, speed settings and the output directories."""
    import torch

    parts = [f"env {paths.env}"]
    if device is not None:
        parts.append(device_name(device))
    parts.append(f"torch {torch.__version__}")
    if args is not None:
        parts.append(f"amp {args.amp} | channels_last {_on(args.channels_last)} | "
                     f"cudnn.benchmark {_on(args.cudnn_benchmark)} | workers {args.workers} | "
                     f"pin_memory {_on(args.pin_memory)}")
        if getattr(args, "compile", "off") != "off":
            parts.append(f"compile {args.compile}")
    parts.append(f"data {paths.data_dir} | cache {paths.cache_dir} | result {paths.result_dir}")
    return "[env] " + " | ".join(parts)

def run_timestamp():
    return datetime.now(ZoneInfo("America/New_York")).strftime("%m-%d-%H-%M")


# Exact historical aliases: checkpoint bytes and signed export manifests remain unchanged.
RUN_ID_ALIASES = {'10-30-08-40_A_resnet-ver-1-cutout-ema': '10-30-08-40_A_resnet-ver-1-T00-cutout-ema', '10-30-09-44_B_wrn-ver-1-cutout-ema': '10-30-09-44_B_wrn-ver-1-T00-cutout-ema', '10-30-09-46_C_seresnet-ver-1-cutout-ema': '10-30-09-46_C_seresnet-ver-1-T00-cutout-ema', '10-30-09-50_C_seresnet-ver-2-cutout-ema': '10-30-09-50_C_seresnet-ver-2-T00-cutout-ema', '10-30-19-41_B_wrn-ver-2-wide-ema': '10-30-19-41_B_wrn-ver-2-T08-wide-ema', '10-30-20-03_A_resnet-ver-6-eca-ema': '10-30-20-03_A_resnet-ver-2-T48-eca-ema', '10-30-20-04_C_seresnet-ver-8-se8-ema': '10-30-20-04_C_seresnet-ver-3-T06-se8-ema', '10-30-21-05_E_pyramid-ver-1-basic110-ema': '10-30-21-05_E_pyramid-ver-1-T18-basic110-ema', '10-30-21-07_G_resnext-ver-1-grouped-ema': '10-30-21-07_G_resnext-ver-1-T42-grouped-ema', '10-30-21-17_A_resnet-ver-7-mixup-ema': '10-30-21-17_A_resnet-ver-3-T49-mixup-ema', '10-30-21-18_B_wrn-ver-8-mixup-ema': '10-30-21-18_B_wrn-ver-3-T50-mixup-ema', '10-30-21-42_F_shake-ver-1-shake40': '10-30-21-42_F_shake-ver-1-T35-shake40', '10-30-21-47_C_seresnet-ver-31-mixup-ema': '10-30-21-47_C_seresnet-ver-4-T51-mixup-ema', '10-30-23-09_A_resnet-ver-4-early-ema': '10-30-23-09_A_resnet-ver-4-T22-early-ema', '10-30-23-10_B_wrn-ver-6-deep-ema': '10-30-23-10_B_wrn-ver-4-T24-deep-ema', '10-30-23-43_C_seresnet-ver-3-autoaug-cutout': '10-30-23-43_C_seresnet-ver-5-T01-autoaug-cutout', '10-30-23-56_A_resnet-ver-5-mid-ema': '10-30-23-56_A_resnet-ver-5-T26-mid-ema', '10-31-00-42_B_wrn-ver-5-balanced-ema': '10-31-00-42_B_wrn-ver-5-T23-balanced-ema', '10-31-00-50_C_seresnet-ver-19-reflect-ema': '10-31-00-50_C_seresnet-ver-6-T31-reflect-ema', '10-30-22-34_F_shake-ver-2-shake40-1800ep': '10-30-22-34_F_shake-ver-2-T35-shake40-1800ep', '10-31-20-06_E_pyramid-ver-3-T61-autoaug-only': '10-31-20-06_E_pyramid-ver-3-T69-autoaug-only'}


def canonical_run_id(run_id):
    return RUN_ID_ALIASES.get(run_id, run_id)


def run_task_id(run_id):
    match = re.search(r"-ver-\d+-(T\d{2,})-", canonical_run_id(run_id))
    return match.group(1) if match else "T00"


def run_paths(result_root, run_id, create=False):
    """One timestamped ID across directories grouped by artifact type."""
    if not run_id or Path(run_id).name != run_id or run_id in (".", ".."):
        raise ValueError("Run ID must be a single directory name")
    run_id = canonical_run_id(run_id)
    root = Path(result_root)
    folders = {"checkpoints": "checkpoints", "configs": "configs", "logs": "logs",
               "validation": "evaluations/validation", "test": "evaluations/test",
               "plots": "plots", "progress": "progress", "manifests": "manifests", "exports": "exports"}
    paths = SimpleNamespace(root=root, run_id=run_id,
                            **{key: root / folder / run_id for key, folder in folders.items()})
    if not paths.checkpoints.exists():
        for legacy, canonical in RUN_ID_ALIASES.items():
            old_dir = root / "checkpoints" / legacy
            if canonical == run_id and old_dir.is_dir():
                paths.checkpoints = old_dir
                break
    if create:
        for key in folders:
            getattr(paths, key).mkdir(parents=True, exist_ok=True)
    return paths


def paths_for_run(checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir)
    if checkpoint_dir.parent.name != "checkpoints":
        raise ValueError("Expected result/checkpoints/<RUN_ID>; legacy flat runs must be migrated explicitly")
    return run_paths(checkpoint_dir.parent.parent, checkpoint_dir.name)


def artifact_path(artifacts, category, role, extension, event=False, unique=False):
    """Stable training paths; dated evaluation/export paths with collision-safe role suffixes."""
    folder = getattr(artifacts, category)
    prefix = artifacts.run_id
    if event:
        identifier = artifacts.run_id.split('_', 1)[1] if '_' in artifacts.run_id else artifacts.run_id
        prefix = f"{run_timestamp()}_{identifier}"
    stem = f"{prefix}_{role}"
    path = folder / (stem + extension)
    if unique:
        index = 1
        while any(folder.glob(path.stem + '*')):
            index += 1
            path = folder / (f"{stem}-{index}" + extension)
    return path


def checkpoint_path(run_dir, which):
    if which not in ('best', 'last'):
        raise ValueError("Checkpoint role must be best or last")
    run_dir = Path(run_dir)
    canonical = canonical_run_id(run_dir.name)
    candidates = [run_dir.parent / canonical / f"{canonical}_{which}-ckpt.pt",
                  run_dir / f"{run_dir.name}_{which}-ckpt.pt"]
    for legacy, target in RUN_ID_ALIASES.items():
        if target == canonical:
            candidates.extend([run_dir.parent / canonical / f"{legacy}_{which}-ckpt.pt",
                               run_dir.parent / legacy / f"{legacy}_{which}-ckpt.pt"])
    return next((p for p in candidates if p.is_file()), candidates[0])


In [ ]:
ENV = detect_env()
print(f"[runtime] detected {ENV} | kernel {sys.executable} | cwd {Path.cwd()} | "
      f"RUNTIME_ENV={RUNTIME_ENV!r} | CIFAR_ENV={_getenv('CIFAR_ENV')!r}")
if ENV == "colab":
    try:
        from google.colab import drive
    except ImportError:  # CIFAR_ENV=colab outside Google Colab (testing)
        print("google.colab is not available: skipping drive.mount")
    else:
        drive.mount("/content/drive")

PATHS = resolve_paths(DATA_DIR, CACHE_DIR, RESULT_DIR, ENV)
DEV = get_device(DEVICE)
if ENV == "colab" and DEV.type != "cuda" and (DEVICE or _getenv("CIFAR_DEVICE")) != "cpu":
    raise RuntimeError("No GPU: choose Runtime > Change runtime type > A100 GPU, then Runtime > Run all again")
HW = resolve_hardware(SimpleNamespace(amp=AMP, channels_last=CHANNELS_LAST, cudnn_benchmark=CUDNN_BENCHMARK,
                                      pin_memory=PIN_MEMORY, workers=WORKERS), DEV)
print(summary_line(PATHS, DEV, HW))
print(f"python {platform.python_version()} | torchvision {torchvision.__version__} | numpy {np.__version__} | "
      f"{platform.platform()} | {available_cpus()} CPUs")
if ENV == "colab":
    print(f"Google Drive folder {drive_dir()}: {'found' if drive_dir().is_dir() else 'NOT FOUND'}")
if ENV == "kaggle":
    print(f"Kaggle input {kaggle_input_dir()}: train_labels.csv in "
          f"{[str(d) for d in find_competition_data()] or 'none'}")
if DEV.type == "cuda" and shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 3. Data

### 3.1 Image cache

All PNGs are decoded once into uint8 arrays. The cache describes itself, so training and prediction need neither
the PNGs nor the data folder once it exists:

| File | Content |
|---|---|
| `train_images.npy`, `train_labels.npy`, `train_ids.npy` | uint8 images (50000, 32, 32, 3), labels and ids in `train_labels.csv` order |
| `test_images.npy`, `test_ids.npy` | test images and ids in `test.csv` order |
| `train_labels.csv`, `test.csv`, `sample_submission.csv` | copies of the competition CSVs |
| `meta.json` | format 2: image counts, shape, sha256 of every file and a fingerprint over all of them |

The data cell verifies the local cache, then tries a compatible Drive cache ZIP, then reads
your manually uploaded competition data. Kaggle download, installation, login and token prompts
are commented out; no Kaggle credentials are needed.

On Colab, upload the downloaded competition ZIP (any filename) into **MyDrive/DL_Project1/data/**.
You can instead place the extracted CSVs and image folders there, or the CSVs plus image ZIPs.
Keep just one dataset in that folder. Wrapper folders and nested image ZIPs are supported.
Leave `DATA_DIR=None` to use this folder, or set it to another folder or a specific ZIP file.
ZIPs are copied/extracted on local disk at `/content/data_extracted/`; Drive originals are preserved.
The processed arrays stay at `/content/cache/` for training.

With `SAVE_CACHE_TO_DRIVE=True`, the notebook saves and verifies **MyDrive/DL_Project1/cache.zip**.
Later runs restore this cache without downloading anything. Results remain under **MyDrive/DL_Project1/result/**.
Keep `AUTO_DOWNLOAD=False`; changing it to True is rejected while the download code is disabled.
Google Drive mounting is still required. For VS Code connected to Colab, set `RUNTIME_ENV="colab"`.
Source CSV changes invalidate incompatible caches. If replacing only a ZIP or changing image contents
without changing CSVs, remove the old local cache and Drive cache backups before rebuilding.


In [ ]:
NUM_CLASSES = 10
CLASSES = ("airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck")
CACHE_FORMAT = 2
SOURCE_CSVS = ("train_labels.csv", "test.csv", "sample_submission.csv")
CACHE_ARRAYS = ("train_images.npy", "train_labels.npy", "train_ids.npy", "test_images.npy", "test_ids.npy")


def sha256_file(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def _decode_pngs(paths):
    def load(p):
        with Image.open(p) as im:
            return np.asarray(im.convert("RGB"), dtype=np.uint8)

    with ThreadPoolExecutor(max_workers=min(8, os.cpu_count() or 1)) as pool:
        return np.stack(list(pool.map(load, paths, chunksize=256)))


def _save_npy(path, arr):
    tmp = path.with_suffix(".tmp")
    with open(tmp, "wb") as f:
        np.save(f, arr)
    os.replace(tmp, path)


def _copy_csvs(data_dir, cache_dir):
    if data_dir.resolve() == cache_dir.resolve():
        return
    for name in SOURCE_CSVS:
        tmp = cache_dir / (name + ".tmp")
        shutil.copyfile(data_dir / name, tmp)
        os.replace(tmp, cache_dir / name)


def _write_meta(cache_dir, n_train, n_test, shape, note):
    files = {name: {"sha256": sha256_file(cache_dir / name), "bytes": (cache_dir / name).stat().st_size}
             for name in CACHE_ARRAYS + SOURCE_CSVS}
    fingerprint = hashlib.sha256("".join(f"{n} {files[n]['sha256']}\n" for n in sorted(files)).encode())
    meta = {"format": CACHE_FORMAT, "n_train": n_train, "n_test": n_test, "shape": list(shape),
            "fingerprint": fingerprint.hexdigest(), "files": files, "note": note,
            "written": datetime.now().isoformat(timespec="seconds")}
    tmp = cache_dir / "meta.json.tmp"
    tmp.write_text(json.dumps(meta, indent=2))
    os.replace(tmp, cache_dir / "meta.json")  # written last: a cache without it is incomplete
    return meta


def read_meta(cache_dir):
    path = Path(cache_dir) / "meta.json"
    return json.loads(path.read_text()) if path.exists() else None


def build_cache(data_dir, cache_dir):
    """Decode all PNGs once into uint8 arrays (N, 32, 32, 3) and save them with the CSVs and meta.json."""
    data_dir, cache_dir = Path(data_dir), Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    train = pd.read_csv(data_dir / "train_labels.csv")
    print(f"[cache] decoding {len(train):,} training PNGs ...", flush=True)
    x = _decode_pngs([data_dir / "train_images" / f for f in train["filename"]])
    _save_npy(cache_dir / "train_images.npy", x)
    _save_npy(cache_dir / "train_labels.npy", train["label"].to_numpy(np.int64))
    _save_npy(cache_dir / "train_ids.npy", train["id"].to_numpy(np.int64))

    test = pd.read_csv(data_dir / "test.csv")
    print(f"[cache] decoding {len(test):,} test PNGs ...", flush=True)
    x = _decode_pngs([data_dir / "test_images" / f for f in test["filename"]])
    _save_npy(cache_dir / "test_images.npy", x)
    _save_npy(cache_dir / "test_ids.npy", test["id"].to_numpy(np.int64))

    _copy_csvs(data_dir, cache_dir)
    validate_cache_arrays(cache_dir)
    meta = _write_meta(cache_dir, len(train), len(test), x.shape[1:], f"built from {data_dir}")
    print(f"[cache] wrote {cache_dir} (fingerprint {meta['fingerprint'][:12]})", flush=True)
    return meta


def _upgrade_format1(data_dir, cache_dir):
    """Add CSV copies and hashes to a format-1 cache if its arrays match the CSVs. Returns success."""
    train = pd.read_csv(data_dir / "train_labels.csv")
    test = pd.read_csv(data_dir / "test.csv")
    try:
        x_train = np.load(cache_dir / "train_images.npy", mmap_mode="r")
        x_test = np.load(cache_dir / "test_images.npy", mmap_mode="r")
        ok = (x_train.shape[0] == len(train) and x_test.shape[0] == len(test)
              and x_train.shape[1:] == x_test.shape[1:]
              and np.array_equal(np.load(cache_dir / "train_labels.npy"), train["label"].to_numpy(np.int64))
              and np.array_equal(np.load(cache_dir / "train_ids.npy"), train["id"].to_numpy(np.int64))
              and np.array_equal(np.load(cache_dir / "test_ids.npy"), test["id"].to_numpy(np.int64)))
    except (OSError, ValueError):
        ok = False
    if not ok:
        return False
    shape = x_train.shape[1:]
    del x_train, x_test
    _copy_csvs(data_dir, cache_dir)
    validate_cache_arrays(cache_dir)
    meta = _write_meta(cache_dir, len(train), len(test), shape, f"format 1 cache upgraded with {data_dir}")
    print(f"[cache] upgraded {cache_dir} to format {CACHE_FORMAT} (arrays unchanged, "
          f"fingerprint {meta['fingerprint'][:12]})", flush=True)
    return True


def _cache_problem(meta, cache_dir, data_dir, has_csvs):
    """None if the format-2 cache is complete and matches the CSVs in data_dir, else the reason."""
    for name, info in meta["files"].items():
        path = cache_dir / name
        if not path.exists() or path.stat().st_size != info["bytes"]:
            return f"{name} is missing or has the wrong size"
    if has_csvs and data_dir.resolve() != cache_dir.resolve():
        for name in SOURCE_CSVS:
            if sha256_file(data_dir / name) != meta["files"][name]["sha256"]:
                return f"{data_dir / name} differs from the cached copy"
    return None


def ensure_cache(data_dir, cache_dir):
    """Return the cache dir, building, upgrading or rebuilding the cache only when needed.

    A complete format-2 cache is used as is; if data_dir also holds the CSVs, they must match the
    cached copies (otherwise the cache is rebuilt from the PNGs).
    """
    data_dir, cache_dir = Path(data_dir), Path(cache_dir)
    has_csvs = all((data_dir / n).is_file() for n in SOURCE_CSVS)
    has_pngs = has_csvs and (data_dir / "train_images").is_dir() and (data_dir / "test_images").is_dir()
    meta = read_meta(cache_dir)
    if meta is not None and meta.get("format") == CACHE_FORMAT:
        problem = _cache_problem(meta, cache_dir, data_dir, has_csvs)
        if problem is None:
            return cache_dir
        print(f"[cache] {problem}; rebuilding", flush=True)
    elif meta is not None and has_csvs and all((cache_dir / n).exists() for n in CACHE_ARRAYS):
        if _upgrade_format1(data_dir, cache_dir):
            return cache_dir
        print("[cache] format 1 cache does not match the CSVs; rebuilding", flush=True)
    if not has_pngs and meta is not None and meta.get("format") is None:
        raise FileNotFoundError(
            f"{cache_dir} is a format-1 cache and {data_dir} has no CSVs to upgrade it with; "
            f"set DATA_DIR to the folder with the competition CSVs")
    if not has_pngs:
        raise FileNotFoundError(
            f"no usable cache in {cache_dir} and no PNG images in {data_dir} to build one; set DATA_DIR "
            f"(folder with train_labels.csv, train_images/, ...) or CACHE_DIR, or on Colab upload cache.zip "
            f"to {drive_dir()}")
    build_cache(data_dir, cache_dir)
    return cache_dir


def verify_cache(cache_dir):
    """Check every cached file against the sha256 in meta.json; return meta (raises on a mismatch)."""
    cache_dir = Path(cache_dir)
    meta = read_meta(cache_dir)
    _check_cache_manifest(meta)
    bad = [n for n, info in meta["files"].items()
           if not (cache_dir / n).exists() or sha256_file(cache_dir / n) != info["sha256"]]
    if bad:
        raise RuntimeError(f"cache files do not match meta.json: {bad}")
    n_train, n_test = validate_cache_arrays(cache_dir)
    if (meta["n_train"], meta["n_test"], meta["shape"]) != (n_train, n_test, [32, 32, 3]):
        raise RuntimeError("Cache metadata does not match its arrays")
    return meta


def validate_cache_arrays(cache_dir):
    """Validate shapes, labels, IDs, and the correspondence between arrays and CSVs."""
    cache_dir = Path(cache_dir)
    frames = {name: pd.read_csv(cache_dir / name) for name in SOURCE_CSVS}
    for name, columns in (("train_labels.csv", ("id", "filename", "label")),
                          ("test.csv", ("id", "filename")),
                          ("sample_submission.csv", ("id", "label"))):
        frame = frames[name]
        if not set(columns).issubset(frame.columns) or frame.empty:
            raise ValueError(f"{name}: missing columns or no rows")
        if not pd.api.types.is_integer_dtype(frame["id"]) or not frame["id"].is_unique:
            raise ValueError(f"{name}: IDs must be unique integers")
        if "filename" in frame and (frame["filename"].isna().any() or not frame["filename"].is_unique):
            raise ValueError(f"{name}: filenames must be present and unique")
    train, test = frames["train_labels.csv"], frames["test.csv"]
    if not pd.api.types.is_integer_dtype(train["label"]) or not train["label"].between(0, 9).all():
        raise ValueError("Training labels must be integers in 0..9")
    for split, frame in (("train", train), ("test", test)):
        images = np.load(cache_dir / f"{split}_images.npy", mmap_mode="r", allow_pickle=False)
        ids = np.load(cache_dir / f"{split}_ids.npy", allow_pickle=False)
        if images.dtype != np.uint8 or images.shape != (len(frame), 32, 32, 3):
            raise ValueError(f"{split} images must be uint8 NHWC with shape (N,32,32,3)")
        if not np.array_equal(ids, frame["id"].to_numpy()):
            raise ValueError(f"{split} array IDs differ from CSV order")
    labels = np.load(cache_dir / "train_labels.npy", allow_pickle=False)
    if not np.issubdtype(labels.dtype, np.integer) or not np.array_equal(labels, train["label"].to_numpy()):
        raise ValueError("Cached labels differ from training CSV")
    if set(frames["sample_submission.csv"]["id"]) != set(test["id"]):
        raise ValueError("Sample submission and test IDs differ")
    return len(train), len(test)


def cache_fingerprint(cache_dir):
    meta = read_meta(cache_dir) or {}
    return meta.get("fingerprint")


def load_train(data_dir, cache_dir, mmap=False):
    cache_dir = ensure_cache(data_dir, cache_dir)
    images = np.load(cache_dir / "train_images.npy", mmap_mode="r" if mmap else None)
    return images, np.load(cache_dir / "train_labels.npy"), np.load(cache_dir / "train_ids.npy")


def load_test(data_dir, cache_dir):
    cache_dir = ensure_cache(data_dir, cache_dir)
    return np.load(cache_dir / "test_images.npy"), np.load(cache_dir / "test_ids.npy")


def _cache_matches_source(meta, source_dir):
    """An explicitly supplied set of CSVs must agree with a reused cache."""
    source_dir = Path(source_dir)
    return all(not (source_dir / name).is_file() or
               sha256_file(source_dir / name) == meta['files'][name]['sha256'] for name in SOURCE_CSVS)


def _check_cache_manifest(meta):
    required = set(CACHE_ARRAYS + SOURCE_CSVS)
    if (not isinstance(meta, dict) or meta.get('format') != CACHE_FORMAT
            or not isinstance(meta.get('files'), dict) or set(meta['files']) != required):
        raise ValueError('Cache needs a format-2 manifest with all required files')
    for info in meta['files'].values():
        if (not isinstance(info, dict) or not isinstance(info.get('bytes'), int)
                or info['bytes'] <= 0 or not isinstance(info.get('sha256'), str)
                or not re.fullmatch(r'[0-9a-f]{64}', info['sha256'])):
            raise ValueError('Invalid cache file size or SHA256 in manifest')
    fingerprint = hashlib.sha256(''.join(
        f"{name} {meta['files'][name]['sha256']}\n" for name in sorted(required)).encode()).hexdigest()
    if fingerprint != meta.get('fingerprint'):
        raise ValueError('Cache ZIP fingerprint differs from its manifest')
    return required


def verify_cache_zip(path):
    """Check ZIP contents and every uncompressed file hash, not just the ZIP header."""
    with zipfile.ZipFile(path) as z:
        names = z.namelist()
        if len(names) != len(set(names)) or set(names) != set(CACHE_ARRAYS + SOURCE_CSVS + ('meta.json',)):
            raise ValueError('Cache ZIP must contain exactly the required files at its root')
        meta = json.loads(z.read('meta.json'))
        for name in _check_cache_manifest(meta):
            info = meta['files'][name]
            digest = hashlib.sha256()
            with z.open(name) as f:
                for block in iter(lambda: f.read(1 << 22), b''):
                    digest.update(block)
            if z.getinfo(name).file_size != info['bytes'] or digest.hexdigest() != info['sha256']:
                raise ValueError(f'Cache ZIP checksum mismatch: {name}')
    return meta


def extract_zip_checked(archive, destination):
    """Extract inside a staging folder, rejecting escaping paths and symlink entries."""
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            target = (destination / member.filename).resolve()
            if ('\\' in member.filename or not target.is_relative_to(root)
                    or (member.external_attr >> 16) & 0o170000 == 0o120000):
                raise ValueError(f'Unsafe ZIP member: {member.filename}')
        z.extractall(destination)  # decompression also checks each member's CRC


def publish_data_directory(staged, destination):
    """Keep the previous directory until the validated replacement can be published."""
    staged, destination = Path(staged), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    previous = destination.with_name(f'.{destination.name}.previous-{time.time_ns()}')
    if destination.exists():
        os.replace(destination, previous)
    try:
        os.replace(staged, destination)
    except BaseException:
        if previous.exists():
            try:
                os.replace(previous, destination)
            except OSError:
                # Keep recovery files even if the filesystem also refuses rollback.
                print(f'[data] previous files preserved at {previous}; restore them before retrying')
        raise
    else:
        if previous.is_dir() and not previous.is_symlink():
            shutil.rmtree(previous)
        else:
            previous.unlink(missing_ok=True)


def drive_cache_candidates(folder):
    folder = Path(folder)
    if not folder.is_dir():
        return []
    pattern = re.compile(r'cache(?: \(\d+\))?\.zip$')
    return sorted((p for p in folder.iterdir() if p.is_file() and pattern.fullmatch(p.name)),
                  key=lambda p: (p.stat().st_mtime_ns, p.name), reverse=True)


def fetch_drive_cache(folder, cache_dir, source_dir=None):
    """Try Drive cache copies newest first; never replace a cache with an unverified ZIP."""
    cache_dir = Path(cache_dir)
    cache_dir.parent.mkdir(parents=True, exist_ok=True)
    for archive in drive_cache_candidates(folder):
        try:
            with tempfile.TemporaryDirectory(prefix='.restore-cache-', dir=cache_dir.parent) as tmp:
                local_zip, staged = Path(tmp) / 'cache.zip', Path(tmp) / 'cache'
                shutil.copyfile(archive, local_zip)
                meta = verify_cache_zip(local_zip)
                if source_dir is not None and not _cache_matches_source(meta, source_dir):
                    print(f'[cache] skipping {archive.name}: source CSVs differ')
                    continue
                extract_zip_checked(local_zip, staged)
                verify_cache(staged)
                publish_data_directory(staged, cache_dir)
            print(f'[cache] restored {archive} -> {cache_dir}')
            return cache_dir
        except (OSError, ValueError, KeyError, RuntimeError, zipfile.BadZipFile, zlib.error, EOFError) as exc:
            print(f'[cache] cannot reuse {archive.name}: {type(exc).__name__}; trying another source')
    raise FileNotFoundError('No valid compatible Drive cache was found')


def save_drive_cache(cache_dir, folder):
    """Publish a verified cache.zip once; skip writing an identical valid backup."""
    cache_dir, folder = Path(cache_dir), Path(folder)
    meta = verify_cache(cache_dir)
    folder.mkdir(parents=True, exist_ok=True)
    destination = folder / 'cache.zip'
    if destination.is_file():
        try:
            if verify_cache_zip(destination)['fingerprint'] == meta['fingerprint']:
                print(f'[cache] Drive backup already verified: {destination}')
                return destination
        except (OSError, ValueError, KeyError, RuntimeError, zipfile.BadZipFile, zlib.error, EOFError):
            print('[cache] replacing an invalid Drive backup after verifying its replacement')
    partial = folder / f'.cache-{time.time_ns()}.zip.tmp'
    try:
        with tempfile.TemporaryDirectory(prefix='.export-cache-', dir=cache_dir.parent) as tmp:
            archive = Path(tmp) / 'cache.zip'
            with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED, compresslevel=1) as z:
                for name in CACHE_ARRAYS + SOURCE_CSVS + ('meta.json',):
                    z.write(cache_dir / name, arcname=name)
            verify_cache_zip(archive)
            shutil.copyfile(archive, partial)
            verify_cache_zip(partial)  # check the Drive copy before replacing the old backup
            os.replace(partial, destination)
    finally:
        partial.unlink(missing_ok=True)
    print(f'[cache] saved reusable Drive backup: {destination}')
    return destination


def find_raw_data(root, require_images=False):
    """Find CSVs under a wrapper folder, optionally checking every referenced PNG exists."""
    for folder in find_competition_data(root):
        if not all((folder / name).is_file() for name in SOURCE_CSVS):
            continue
        if not require_images:
            return folder
        complete = True
        for split, csv_name in (('train', 'train_labels.csv'), ('test', 'test.csv')):
            frame = pd.read_csv(folder / csv_name)
            if 'filename' not in frame or frame.empty or frame['filename'].isna().any():
                complete = False
                break
            image_root = (folder / f'{split}_images').resolve()
            for filename in frame['filename']:
                image = (image_root / str(filename)).resolve()
                if not image.is_relative_to(image_root) or not image.is_file():
                    complete = False
                    break
            if not complete:
                break
        if complete:
            return folder
    return None


# Kaggle download and login disabled: use manually uploaded data instead.
# def run_kaggle_download():
#     """Use KaggleHub in this notebook process and honor its returned cache/input path."""
#     if not _getenv('KAGGLE_CONFIG_DIR') and (Path('/content') / 'kaggle.json').is_file():
#         os.environ['KAGGLE_CONFIG_DIR'] = '/content'
#     try:
#         kagglehub = importlib.import_module('kagglehub')
#     except ModuleNotFoundError as exc:
#         if exc.name != 'kagglehub':
#             raise RuntimeError('A KaggleHub dependency is missing. Run %pip install -U "kagglehub>=1,<2", '
#                                'restart the kernel, and rerun from the configuration cell.') from None
#         print('[download] installing KaggleHub in this kernel environment')
#         installed = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
#                                     '--disable-pip-version-check', 'kagglehub>=1,<2'],
#                                    capture_output=True, text=True)
#         if installed.returncode:
#             raise RuntimeError('Cannot install KaggleHub. Check internet access and use Python 3.11+; '
#                                'alternatively provide extracted data or a saved cache.')
#         importlib.invalidate_caches()
#         kagglehub = importlib.import_module('kagglehub')
#     print(f'[download] KaggleHub competition_download({KAGGLE_SLUG!r})', flush=True)
#     try:
#         # Do not assume a ZIP filename or pass newer optional API arguments.
#         # This also preserves credentials set by kagglehub.login() in this kernel.
#         downloaded = Path(kagglehub.competition_download(KAGGLE_SLUG))
#     except Exception as exc:
#         response = getattr(exc, 'response', None)
#         status = getattr(response, 'status_code', None)
#         if not isinstance(status, int):
#             status = getattr(exc, 'error_code', None)
#         status_text = f', HTTP {status}' if isinstance(status, int) and 100 <= status <= 599 else ''
#         kind = type(exc).__name__
#         if status == 401 or kind in ('UnauthenticatedError', 'CredentialError'):
#             print('[auth] Kaggle login is needed for this download. Enter your own API token in the login prompt.')
#             try:
#                 kagglehub.login()
#             except Exception as login_error:
#                 raise RuntimeError(
#                     f'Cannot open/complete Kaggle login ({type(login_error).__name__}). '
#                     'For notebook widgets, install ipywidgets and enable widget support in your editor, '
#                     'or use a masked getpass prompt to set KAGGLE_API_TOKEN (see README.md). '
#                     'Then rerun this data cell.') from None
#             # Notebook login displays an asynchronous widget. The kernel must become idle
#             # for its Login button to work; do not block/poll or download again immediately.
#             raise RuntimeError(
#                 f'Data setup paused for Kaggle login ({kind}{status_text}). '
#                 'Complete the login prompt above, then rerun THIS DATA CELL and continue the remaining cells. '
#                 'Do not restart the kernel after login. Existing data and valid caches need no login.') from None
#         if status == 403:
#             advice = ('Your account may need competition access or rule acceptance. '
#                       'If you need to switch credentials, use kagglehub.login() or set KAGGLE_API_TOKEN. '
#                       f'Confirm competition access and accepted rules at https://www.kaggle.com/competitions/{KAGGLE_SLUG}/rules.')
#         elif status == 404:
#             advice = f'Check the competition name {KAGGLE_SLUG!r} and that your account can access it.'
#         elif status == 429:
#             advice = 'Kaggle rate-limited the request. Wait before retrying.'
#         elif kind in ('ConnectionError', 'ConnectTimeout', 'ReadTimeout', 'Timeout'):
#             advice = 'Check the runtime internet connection and retry the data cell.'
#         elif isinstance(exc, OSError):
#             advice = 'Check local free space and write permissions for the KaggleHub cache.'
#         else:
#             advice = ('Check KaggleHub installation and account access; use kagglehub.login() or '
#                       'KAGGLE_API_TOKEN if authentication is required.')
#         # Keep useful diagnostics without echoing request URLs/tokens from exception text.
#         raise RuntimeError(f'KaggleHub download failed ({kind}{status_text}). {advice} '
#                            'Rerun the data cell after correcting the issue.') from None
#     if not downloaded.exists():
#         raise FileNotFoundError(f'KaggleHub returned a path that does not exist: {downloaded}')
#     print(f'[download] KaggleHub returned: {downloaded}')
#     return downloaded
#
#
# def download_competition_data(data_dir):
#     """Normalize a KaggleHub folder/ZIP or a legacy downloaded ZIP into the raw data directory."""
#     data_dir = Path(data_dir)
#     data_dir.parent.mkdir(parents=True, exist_ok=True)
#     legacy_archive = data_dir.parent / 'downloads' / KAGGLE_SLUG / f'{KAGGLE_SLUG}.zip'
#     downloaded = None
#     if legacy_archive.is_file():
#         try:
#             with zipfile.ZipFile(legacy_archive) as z:
#                 if z.testzip() is None:
#                     downloaded = legacy_archive
#         except (OSError, zipfile.BadZipFile, zlib.error, EOFError):
#             pass
#     if downloaded is None:
#         downloaded = run_kaggle_download()
#     else:
#         print(f'[download] reusing completed legacy archive: {downloaded}')
#     if downloaded.is_dir() and (downloaded.resolve().is_relative_to(data_dir.resolve())
#                                 or data_dir.resolve().is_relative_to(downloaded.resolve())):
#         raw = find_raw_data(downloaded, require_images=True)
#         if raw is not None:
#             return raw  # reuse overlapping complete data without moving or modifying the provider cache
#         raise ValueError('KaggleHub cache overlaps DATA_DIR. Choose a separate DATA_DIR for extraction.')
#     with tempfile.TemporaryDirectory(prefix='.extract-data-', dir=data_dir.parent) as tmp:
#         extracted = Path(tmp) / 'extracted'
#         if downloaded.is_dir():
#             # KaggleHub normally already extracted the competition archive. Work on a copy:
#             # its cache (or Kaggle's read-only input mount) must not be moved or unpacked in place.
#             shutil.copytree(downloaded, extracted, copy_function=shutil.copyfile)
#             for directory, _, _ in os.walk(extracted):
#                 Path(directory).chmod(0o700)  # only the working copy must allow nested ZIP extraction
#         elif zipfile.is_zipfile(downloaded):
#             extract_zip_checked(downloaded, extracted)
#         else:
#             raise ValueError(f'Expected a competition folder or ZIP from KaggleHub, received {downloaded}')
#         unpacked = set()
#         for _ in range(4):
#             raw = find_raw_data(extracted, require_images=True)
#             if raw is not None:
#                 publish_data_directory(raw, data_dir)
#                 print(f'[data] ready: {data_dir}')
#                 return data_dir
#             nested = [p for p in extracted.rglob('*.zip') if p not in unpacked]
#             if not nested:
#                 break
#             for path in nested:
#                 target = path.parent
#                 if path.stem in ('train_images', 'test_images'):
#                     with zipfile.ZipFile(path) as z:
#                         if not any(name.startswith(path.stem + '/') for name in z.namelist()):
#                             target = target / path.stem
#                 extract_zip_checked(path, target)
#                 unpacked.add(path)
#     raise ValueError(f'Downloaded files at {downloaded} lack complete train_images/, test_images/ and '
#                      'the three CSVs. Inspect that folder or set DATA_DIR to complete extracted data. '
#                      'If the KaggleHub cache is incomplete, refresh it with '
#                      'kagglehub.competition_download(..., force_download=True).')
#
#

def prepare_uploaded_data(uploaded, data_dir):
    """Read a user-supplied folder/ZIP; unpack only into local staging, without network access."""
    uploaded, data_dir = Path(uploaded), Path(data_dir)
    if not uploaded.exists():
        raise FileNotFoundError(
            f'No complete data or valid cache. Upload the competition ZIP or extracted files to '
            f'{uploaded}. On Colab the default is MyDrive/DL_Project1/data/. '
            'Kaggle downloads and login are disabled.')
    if (uploaded.resolve().is_relative_to(data_dir.resolve())
            or data_dir.resolve().is_relative_to(uploaded.resolve())):
        raise ValueError('Uploaded source and local extraction directory must be separate.')
    data_dir.parent.mkdir(parents=True, exist_ok=True)
    print(f'[data] reading manually uploaded files: {uploaded}')
    with tempfile.TemporaryDirectory(prefix='.extract-data-', dir=data_dir.parent) as tmp:
        extracted = Path(tmp) / 'extracted'
        if uploaded.is_dir():
            shutil.copytree(uploaded, extracted, copy_function=shutil.copyfile)
            for directory, _, _ in os.walk(extracted):
                Path(directory).chmod(0o700)
        elif zipfile.is_zipfile(uploaded):
            extract_zip_checked(uploaded, extracted)
        else:
            raise ValueError(f'Expected an uploaded competition folder or ZIP, received {uploaded}')
        unpacked = set()
        for _ in range(4):
            raw = find_raw_data(extracted, require_images=True)
            if raw is not None:
                publish_data_directory(raw, data_dir)
                print(f'[data] ready: {data_dir}')
                return data_dir
            nested = [p for p in extracted.rglob('*.zip') if p not in unpacked]
            if not nested:
                break
            for path in nested:
                target = path.parent
                if path.stem in ('train_images', 'test_images'):
                    with zipfile.ZipFile(path) as z:
                        if not any(name.startswith(path.stem + '/') for name in z.namelist()):
                            target = target / path.stem
                extract_zip_checked(path, target)
                unpacked.add(path)
    raise ValueError(
        f'Uploaded files at {uploaded} lack complete train_images/, test_images/, '
        'train_labels.csv, test.csv and sample_submission.csv. Upload all competition files; '
        'ZIP and extracted-folder layouts are supported.')


def prepare_data(paths, auto_download=False, save_to_drive=True):
    """Valid local cache -> compatible Drive cache -> manually uploaded folder/ZIP."""
    if auto_download:
        raise ValueError("Kaggle download/login is disabled. Keep AUTO_DOWNLOAD=False and upload data manually.")
    cache = paths.cache_dir
    persist = paths.env == 'colab' and save_to_drive
    if persist and not drive_mounted():
        raise RuntimeError('Mount Google Drive before data setup to save cache.zip, or explicitly set '
                           'SAVE_CACHE_TO_DRIVE=False for temporary storage only.')
    source = find_raw_data(paths.data_dir) or paths.data_dir
    meta = None
    if (cache / 'meta.json').is_file():
        try:
            candidate = verify_cache(cache)
            if _cache_matches_source(candidate, source):
                meta = candidate
                print(f'[cache] reusing verified local cache: {cache}')
            else:
                print('[cache] source CSVs changed; rebuilding from the requested data')
        except (OSError, ValueError, KeyError, RuntimeError, EOFError) as exc:
            print(f'[cache] local cache is incomplete or invalid ({type(exc).__name__})')
    if meta is None and paths.env == 'colab' and drive_mounted():
        try:
            fetch_drive_cache(drive_dir(), cache, source)
            meta = verify_cache(cache)
        except FileNotFoundError:
            pass
    if meta is None:
        if paths.data_dir.resolve() == cache.resolve():
            raise ValueError('Use separate DATA_DIR and CACHE_DIR when building from uploaded data')
        raw = find_raw_data(paths.data_dir, require_images=True)
        if raw is None:
            raw = prepare_uploaded_data(paths.data_dir, cache.parent / 'data_extracted')
        paths.data_dir = raw
        cache.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='.build-cache-', dir=cache.parent) as tmp:
            staged = Path(tmp) / 'cache'
            build_cache(raw, staged)
            meta = verify_cache(staged)
            publish_data_directory(staged, cache)
    else:
        paths.data_dir = source
    if persist:
        save_drive_cache(cache, drive_dir())
    return cache, meta


In [ ]:
CACHE, META = prepare_data(PATHS, auto_download=AUTO_DOWNLOAD, save_to_drive=SAVE_CACHE_TO_DRIVE)
# Keep later training/evaluation cells on the resolved source folder (including wrapper directories).
DATA_DIR = str(PATHS.data_dir)
CACHE_DIR = str(CACHE)
print(f"[cache] {CACHE}: format {META['format']} | {META['n_train']:,} train / {META['n_test']:,} test images "
      f"{tuple(META['shape'])} | fingerprint {META['fingerprint'][:12]} | sha256 verified")
counts = np.bincount(np.load(CACHE / "train_labels.npy"), minlength=NUM_CLASSES)
print("Training images per class: " + ", ".join(f"{c} {n:,}" for c, n in zip(CLASSES, counts)))

### 3.2 Split, normalization and augmentation

- **Validation split**: `VAL_SIZE` images are held out, the same number from every class, with the fixed
  `SPLIT_SEED`, so every run sees the same split.
- **Normalization**: per-channel mean and standard deviation of the *training split only*; they are stored in the
  checkpoint and reused for validation and test images. Nothing is computed from the test images.
- **Augmentation** (`AUG`):
  - `basic`: random crop to 32×32 with 4-pixel zero padding, plus horizontal flip;
  - `cutout`: `basic` plus one 16×16 Cutout square per image. As in the original Cutout code, the square's center
    is uniform over the image (so it can be clipped at the border), and it is applied after normalization, so the
    patch takes the training-set mean colour;
  - `autoaug`: `basic` plus AutoAugment (CIFAR-10 policy), then Cutout or RandomErasing (`AUTOAUG_ERASE`);
  - `trivialaug` (E only): `basic` plus TrivialAugmentWide (Müller & Hutter, 2021; torchvision defaults: one random
    operation per image, uniform magnitude from 31 bins, nearest interpolation, fill 0, like AutoAugment here), then
    the same `AUTOAUG_ERASE` occlusion (`"cutout"`, `"erasing"` or `"none"`).
- For `basic` and `cutout`, the uint8 training split stays on the accelerator and crop, flip and Cutout are applied to
  the whole batch at once, with an independent random draw for every image. torchvision's `AutoAugment` picks one
  sub-policy per call, so `autoaug` applies it to one image at a time on the CPU (in DataLoader workers on Linux);
  `trivialaug` uses the same uint8 CPU path, normalization order and worker handling (0 workers on macOS / Windows).
- All randomness of the data order and augmentation comes from generators seeded by `(SEED, epoch)`; AutoAugment and
  TrivialAugment seed each image from its position in the epoch. A resumed run therefore sees exactly the same data as an
  uninterrupted one, independent of the device and the number of workers.

In [ ]:
# ---- split and normalization statistics

def _per_class(n, what):
    if not isinstance(n, (int, np.integer)) or isinstance(n, (bool, np.bool_)) or n < 0:
        raise ValueError(f"{what} must be a nonnegative integer, got {n!r}")
    if n % NUM_CLASSES:
        raise ValueError(f"{what} must be a multiple of {NUM_CLASSES} for a class-balanced split, got {n}")
    return n // NUM_CLASSES


def stratified_split(labels, val_size, seed=0):
    """Hold out `val_size` images, the same number from every class. Returns sorted (train_idx, val_idx)."""
    labels = np.asarray(labels)
    if labels.ndim != 1 or not np.issubdtype(labels.dtype, np.integer) or not len(labels):
        raise ValueError("labels must be a nonempty one-dimensional integer array")
    if np.any((labels < 0) | (labels >= NUM_CLASSES)):
        raise ValueError("labels must be in 0..9")
    k = _per_class(val_size, "val_size")
    if val_size >= len(labels):
        raise ValueError("Validation size must leave at least one training image")
    if np.any(np.bincount(labels, minlength=NUM_CLASSES) < k):
        raise ValueError("Not enough examples per class for the requested validation split")
    all_idx = np.arange(len(labels))
    if val_size == 0:
        return all_idx, np.array([], dtype=np.int64)
    k = _per_class(val_size, "val_size")
    rng = np.random.default_rng(seed)
    val = np.concatenate([rng.permutation(np.flatnonzero(labels == c))[:k] for c in range(NUM_CLASSES)])
    val = np.sort(val)
    return np.setdiff1d(all_idx, val), val


def stratified_subset(indices, labels, size, seed=0):
    """Class-balanced subset of `indices` (used for quick smoke tests)."""
    k = _per_class(size, "train_subset")
    if size == 0 or size > len(indices) or np.any(np.bincount(labels[indices], minlength=NUM_CLASSES) < k):
        raise ValueError("Subset size must be positive and available in every training class")
    rng = np.random.default_rng(seed + 1)
    sub = [rng.permutation(indices[labels[indices] == c])[:k] for c in range(NUM_CLASSES)]
    return np.sort(np.concatenate(sub))


def channel_stats(images):
    """Per-channel mean/std in [0, 1] units over uint8 NHWC images (computed in chunks)."""
    s = np.zeros(3)
    ss = np.zeros(3)
    n = 0
    for i in range(0, len(images), 5000):
        chunk = np.asarray(images[i:i + 5000], dtype=np.float64) / 255.0
        s += chunk.sum(axis=(0, 1, 2))
        ss += (chunk ** 2).sum(axis=(0, 1, 2))
        n += chunk.shape[0] * chunk.shape[1] * chunk.shape[2]
    mean = s / n
    std = np.sqrt(ss / n - mean ** 2)
    return mean.tolist(), std.tolist()


# ---- batch augmentation on the accelerator

def _to_dev(t, dev):
    """Copy a small CPU tensor to `dev`; on CUDA via pinned memory without a host sync."""
    if dev.type == "cuda":
        return t.pin_memory().to(dev, non_blocking=True)
    return t.to(dev)


def epoch_generator(seed, epoch, stream):
    """CPU generator for a given (seed, epoch, stream); device independent and resumable."""
    state = np.random.SeedSequence([seed, epoch, stream]).generate_state(1, dtype=np.uint64)[0]
    return torch.Generator().manual_seed(int(state))


def to_float_nchw(x_u8, mean, std, channels_last=False):
    """uint8 NHWC on device -> normalized float NCHW (channels_last strides are free from NHWC)."""
    x = x_u8.permute(0, 3, 1, 2).float().div_(255.0)
    x = (x - mean) / std
    return x.contiguous(memory_format=torch.channels_last if channels_last else torch.contiguous_format)


def random_crop_flip(x_u8, gen, pad=4):
    """Per-sample random crop with zero padding plus horizontal flip on a uint8 NHWC batch."""
    b, h, w, _ = x_u8.shape
    dev = x_u8.device
    padded = F.pad(x_u8, (0, 0, pad, pad, pad, pad))
    offs = _to_dev(torch.randint(0, 2 * pad + 1, (b, 2), generator=gen), dev)
    flip = _to_dev(torch.rand(b, generator=gen) < 0.5, dev)
    rows = offs[:, 0, None] + torch.arange(h, device=dev)
    cols = torch.arange(w, device=dev).expand(b, w)
    cols = torch.where(flip[:, None], cols.flip(1), cols) + offs[:, 1, None]
    bidx = torch.arange(b, device=dev)[:, None, None]
    return padded[bidx, rows[:, :, None], cols[:, None, :]]


def cutout(x, gen, size=16):
    """Zero a size x size square per sample, center uniform over the image (DeVries & Taylor, 2017).

    Applied after normalization, so zero equals the training-set mean colour, as in the paper's code.
    """
    b, _, h, w = x.shape
    dev = x.device
    cy = _to_dev(torch.randint(0, h, (b,), generator=gen), dev)
    cx = _to_dev(torch.randint(0, w, (b,), generator=gen), dev)
    ys = torch.arange(h, device=dev)
    xs = torch.arange(w, device=dev)
    in_y = (ys[None] >= (cy - size // 2)[:, None]) & (ys[None] < (cy + size - size // 2)[:, None])
    in_x = (xs[None] >= (cx - size // 2)[:, None]) & (xs[None] < (cx + size - size // 2)[:, None])
    mask = ~(in_y[:, :, None] & in_x[:, None, :])
    return x * mask[:, None].to(x.dtype)


def random_erasing(x, gen, p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.3), attempts=10):
    """Per-sample RandomErasing (Zhong et al., 2020) with value 0, same sampling as torchvision."""
    b, _, h, w = x.shape
    area = h * w
    target = area * torch.empty(b, attempts).uniform_(scale[0], scale[1], generator=gen)
    log_r = torch.empty(b, attempts).uniform_(np.log(ratio[0]), np.log(ratio[1]), generator=gen)
    r = torch.exp(log_r)
    eh = torch.sqrt(target * r).round().long()
    ew = torch.sqrt(target / r).round().long()
    valid = (eh < h) & (ew < w)
    first = valid.float().argmax(1)  # first attempt that fits (torchvision tries up to 10)
    pick = torch.arange(b)
    eh, ew = eh[pick, first], ew[pick, first]
    apply = valid.any(1) & (torch.rand(b, generator=gen) < p)
    top = (torch.rand(b, generator=gen) * (h - eh + 1)).long()
    left = (torch.rand(b, generator=gen) * (w - ew + 1)).long()
    params = _to_dev(torch.stack([top, left, eh, ew, apply.long()], 1), x.device)
    ys = torch.arange(h, device=x.device)[None]
    xs = torch.arange(w, device=x.device)[None]
    t, l, hh, ww, a = params.unbind(1)
    in_y = (ys >= t[:, None]) & (ys < (t + hh)[:, None])
    in_x = (xs >= l[:, None]) & (xs < (l + ww)[:, None])
    mask = ~(in_y[:, :, None] & in_x[:, None, :] & a.bool()[:, None, None])
    return x * mask[:, None].to(x.dtype)


# ---- AutoAugment / TrivialAugment, one image at a time on the CPU

# AUG values whose policy runs per image on the CPU; AUTOAUG_ERASE then chooses the occlusion after normalization.
CPU_POLICY_AUGS = ("autoaug", "trivialaug")


class AutoAugmentDataset(torch.utils.data.Dataset):
    """Applies RandomCrop(32, pad 4) + flip + a per-image policy to one uint8 image at a time.

    `policy` "autoaug" is AutoAugment (CIFAR10 policy); "trivialaug" is TrivialAugmentWide (Mueller & Hutter, 2021)
    with torchvision's defaults (31 magnitude bins, nearest interpolation, fill 0), the same defaults as AutoAugment.
    Indices arrive as `epoch * len(self) + position` from `EpochSampler`; the per-image RNG is seeded
    from that number, so results do not depend on which worker handles which image.
    """

    def __init__(self, images_path, indices, labels, seed, policy="autoaug"):
        from torchvision.transforms import v2

        if policy not in CPU_POLICY_AUGS:
            raise ValueError(f"policy = {policy!r}; choose one of {CPU_POLICY_AUGS}")
        self.images_path = str(images_path)
        self.indices = np.asarray(indices)
        self.labels = torch.as_tensor(labels[self.indices], dtype=torch.long)
        self.seed = int(seed)
        self.policy = policy
        self.images = None  # opened lazily as a memmap inside each worker
        self.transform = v2.Compose([
            v2.RandomCrop(32, padding=4),
            v2.RandomHorizontalFlip(),
            v2.AutoAugment(v2.AutoAugmentPolicy.CIFAR10) if policy == "autoaug" else v2.TrivialAugmentWide(),
        ])

    def describe(self):
        """The installed policy and its settings, as recorded in the run configuration."""
        op = self.transform.transforms[-1]
        return {"policy": self.policy, "transform": repr(self.transform), "operation": type(op).__name__,
                "num_magnitude_bins": getattr(op, "num_magnitude_bins", None),
                "interpolation": str(op.interpolation.value), "fill": op.fill,
                "order": "uint8 crop/flip -> policy (CPU) -> float/normalize -> AUTOAUG_ERASE occlusion"}

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, g):
        if self.images is None:
            self.images = np.load(self.images_path, mmap_mode="r")
        pos = g % len(self.indices)
        torch.manual_seed((self.seed * 10**10 + g) % 2**63)
        img = torch.from_numpy(np.array(self.images[self.indices[pos]])).permute(2, 0, 1)
        return self.transform(img).permute(1, 2, 0).contiguous(), self.labels[pos]  # back to HWC


class EpochSampler(torch.utils.data.Sampler):
    """Shuffles positions with a (seed, epoch) generator and encodes the epoch into each index."""

    def __init__(self, n, seed):
        self.n, self.seed, self.epoch = n, seed, 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __iter__(self):
        perm = torch.randperm(self.n, generator=epoch_generator(self.seed, self.epoch, 0))
        return iter((perm + self.epoch * self.n).tolist())

    def __len__(self):
        return self.n


# ---- training batches for one epoch

class TrainBatches:
    """Yields normalized, augmented (x, y) training batches on `device` for one epoch at a time."""

    def __init__(self, images, labels, train_idx, args, mean, std, device, images_path=None):
        self.args, self.device = args, device
        self.mean = torch.tensor(mean, device=device).view(1, 3, 1, 1)
        self.std = torch.tensor(std, device=device).view(1, 3, 1, 1)
        self.n = len(train_idx)
        self.batch_size = args.batch_size
        if self.batch_size <= 0 or self.n < self.batch_size:
            raise ValueError(f"Need at least one complete training batch: {self.n} images, batch {self.batch_size}")
        self.steps_per_epoch = self.n // self.batch_size  # drop the last partial batch
        self.loader = None
        if args.aug in CPU_POLICY_AUGS:
            ds = AutoAugmentDataset(images_path, train_idx, labels, args.seed, policy=args.aug)
            self.sampler = EpochSampler(self.n, args.seed)
            batch_sampler = torch.utils.data.BatchSampler(self.sampler, self.batch_size, drop_last=True)
            kw = dict(num_workers=args.workers, pin_memory=bool(getattr(args, "pin_memory", device.type == "cuda")))
            if args.workers > 0:
                kw.update(persistent_workers=True, prefetch_factor=4, multiprocessing_context=MP_CONTEXT)
            else:
                name = "AutoAugment" if args.aug == "autoaug" else "TrivialAugment"
                print(f"{name} runs in the main process (workers = 0; see section 2)")
            self.loader = torch.utils.data.DataLoader(ds, batch_sampler=batch_sampler, **kw)
        else:
            self.x = torch.from_numpy(np.ascontiguousarray(images[train_idx])).to(device)
            self.y = torch.from_numpy(labels[train_idx]).to(device)

    def _finish(self, x_u8, gen):
        a = self.args
        x = to_float_nchw(x_u8, self.mean, self.std, a.channels_last)
        if a.aug == "cutout" or (a.aug in CPU_POLICY_AUGS and a.autoaug_erase == "cutout"):
            x = cutout(x, gen, a.cutout_size)
        elif a.aug in CPU_POLICY_AUGS and a.autoaug_erase == "erasing":
            x = random_erasing(x, gen, p=a.erasing_p)
        if a.channels_last:  # keep NHWC strides after the masking ops (no-op if already)
            x = x.contiguous(memory_format=torch.channels_last)
        return x

    def epoch(self, epoch):
        seed = self.args.seed
        gen = epoch_generator(seed, epoch, 1)
        if self.loader is not None:
            self.sampler.set_epoch(epoch)
            for x_u8, y in self.loader:
                x_u8 = x_u8.to(self.device, non_blocking=True)
                yield self._finish(x_u8, gen), y.to(self.device, non_blocking=True)
            return
        perm = torch.randperm(self.n, generator=epoch_generator(seed, epoch, 0)).to(self.device)
        for i in range(self.steps_per_epoch):
            idx = perm[i * self.batch_size:(i + 1) * self.batch_size]
            x_u8 = random_crop_flip(self.x[idx], gen) if self.args.aug in ("basic", "cutout") else self.x[idx]
            yield self._finish(x_u8, gen), self.y[idx]

## 4. Model: E, PyramidNet-110 (basic blocks, α = 98)

A PyramidNet (Han, Kim & Kim, 2017) with 54 basic blocks, 18 per stage. Instead of doubling at each downsampling,
the width grows additively by α/54 = 98/54 ≈ 1.81 after every block, from 16 to 114 channels. As in the
[reference implementation](https://github.com/dyhan0920/PyramidNet-PyTorch), the width is accumulated as a float and
each block uses `int(round(width))` channels; the stages end at 49, 81 and 114 channels. Each block computes
BN-conv3×3-BN-ReLU-conv3×3-BN and adds a parameter-free shortcut: the identity, a 2×2 average pool (stride 2) in the
first block of stages 2 and 3, and zero padding for the extra channels. There is no ReLU after the addition; a final
BN-ReLU precedes global average pooling and the linear classifier. This block differs from the course's
ReLU(S(x) + F(x)) form, as B's pre-activation block already does; the project rules allow custom residual blocks.
α = 98 is our choice, made to fill the parameter budget; it is not a configuration from the paper. The paper's
basic-block PyramidNet-110 with α = 84 has ≈3.8M parameters (3,904,446 in this implementation), and α = 99 would
have 5,039,793, over the 5M limit. Widening is the direction the paper found helpful: at depth 110, raising α from 48 to
84 to 270 lowered the error. This implementation (rounded widths, bias-free convolutions) has exactly 4,963,991
parameters. Convolutions use Kaiming-normal initialization (fan-out), BatchNorm starts at γ = 1, β = 0.

`DROP_PATH` in the configuration cell (default 0 = off) enables stochastic depth (Huang et al., 2016) with the
linear decay rule: block l of 54 drops its residual branch with probability `DROP_PATH` · l / 54, independently per
image during training, and surviving branches are scaled by 1 / (1 − p); evaluation uses every block. The masks come
from the global torch RNG, so the seed and the checkpointed RNG state cover them. The parameter count does not
change. `DROP_PATH` is this notebook's architecture switch (option `drop_path`): the rate is saved with the
architecture options in the run configuration and checkpoints, appears in the model description and enters the
architecture fingerprint. Resuming a run with a different `DROP_PATH` is rejected, and evaluation rebuilds the
saved rate whatever the current setting. A positive rate gives the default feature label `stochdepth`.

`MIXSTYLE` in the configuration cell (default off) enables MixStyle (Zhou et al., ICLR 2021, "random" mixing) after
stages 1 and 2 (`MIXSTYLE_AFTER_STAGES`). During training only, with probability `MIXSTYLE_P` per batch and layer, each
image's per-channel feature mean and standard deviation over the spatial positions (float32, unbiased variance +
`MIXSTYLE_EPS`, detached as in the reference code) are replaced by a Beta(`MIXSTYLE_ALPHA`, `MIXSTYLE_ALPHA`)-weighted
mix with those of a partner from a random permutation of the batch. It is parameter-free and adds no module: the
parameter count, state_dict keys, model description and fingerprint are unchanged, the training forward merely accepts
the draws (`forward(x, mixstyle=...)`), and evaluation is exactly the plain network. MixStyle is therefore a training
setting of section 5, not an architecture option; its checkpoints load anywhere, also with the switch off.

In [ ]:
MODEL = "E"
MODEL_DESCRIPTION = "PyramidNet-110 basic, alpha=98 (widths 16->114)"
EXPECTED_PARAMS = 4_963_991


class DropPath(nn.Module):
    """Stochastic depth (Huang et al., 2016) for one residual branch, per sample.

    In training, each image's branch is dropped with probability p and survivors are scaled by 1 / (1 - p);
    in evaluation it is the identity. The mask comes from the global torch RNG of the input's device.
    """

    def __init__(self, p=0.0):
        super().__init__()
        self.p = float(p)

    def forward(self, x):
        if not self.training or self.p == 0.0:
            return x
        keep = 1.0 - self.p
        mask = torch.rand(x.shape[0], 1, 1, 1, device=x.device) < keep
        return x * mask.to(x.dtype) / keep

    def extra_repr(self):
        return f"p={self.p:.4f}"


MIXSTYLE_AFTER_STAGES = (1, 2)  # training-only MixStyle (MIXSTYLE, section 1) mixes the outputs of stages 1 and 2


def mix_feature_styles(x, apply, lam, perm, eps):
    """MixStyle (Zhou et al., ICLR 2021), "random" variant, on one training batch; parameter-free.

    Each image's per-channel mean mu and standard deviation sigma over the spatial positions (unbiased variance + eps,
    computed in float32 and detached, as in the reference code) are replaced by lam * stat + (1 - lam) * stat of its
    partner image perm[i]; the result is cast back to x's dtype. `apply` (0-dim bool tensor) selects the mixed batch
    or exactly x; lam has shape (B, 1, 1, 1). The draws come from the trainer's MixStyleSampler (section 5).
    """
    x32 = x.float()
    mu = x32.mean(dim=(2, 3), keepdim=True).detach()
    sigma = (x32.var(dim=(2, 3), keepdim=True) + eps).sqrt().detach()
    mixed_mu = lam * mu + (1 - lam) * mu[perm]
    mixed_sigma = lam * sigma + (1 - lam) * sigma[perm]
    mixed = ((x32 - mu) / sigma * mixed_sigma + mixed_mu).to(x.dtype)
    return torch.where(apply, mixed, x)


class PyramidBlock(nn.Module):
    """BN-conv-BN-ReLU-conv-BN (Han et al., 2017), added to a parameter-free shortcut; no ReLU after the addition."""

    def __init__(self, in_ch, out_ch, stride=1, drop_path=0.0):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_ch)
        self.drop_path = DropPath(drop_path)
        # Shortcut: identity, 2x2 average pool where the block downsamples, zero padding for the extra channels.
        self.pool = nn.AvgPool2d(2, 2, ceil_mode=True) if stride != 1 else nn.Identity()
        self.extra_ch = out_ch - in_ch

    def forward(self, x):
        out = self.conv1(self.bn1(x))
        out = self.conv2(F.relu(self.bn2(out)))
        out = self.drop_path(self.bn3(out))
        shortcut = self.pool(x)
        if self.extra_ch:
            shortcut = F.pad(shortcut, (0, 0, 0, 0, 0, self.extra_ch))
        return out + shortcut


class PyramidNet(nn.Module):
    """3x3 stem (16 channels, BN), 3 stages of (depth-2)/6 pyramidal blocks, BN-ReLU, global average pool, linear.

    The width grows by alpha / N after each of the N blocks. As in the reference implementation, the width is
    accumulated as a float and each block uses int(round(width)) channels. Block l of N drops its residual branch
    with probability drop_path * l / N (linear decay rule of Huang et al., 2016). A training forward may receive
    MixStyle draws for the points after stages MIXSTYLE_AFTER_STAGES; this adds no module, parameter or buffer.
    """

    mixstyle_layers = len(MIXSTYLE_AFTER_STAGES)  # number of MixStyle draws a training forward accepts

    def __init__(self, depth=110, alpha=98, drop_path=0.0, num_classes=10):
        super().__init__()
        assert (depth - 2) % 6 == 0, "basic-block PyramidNet depth must be 6n+2"
        n = (depth - 2) // 6
        total = 3 * n
        add_rate = alpha / total
        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, 1, 1, bias=False), nn.BatchNorm2d(16))
        stages, in_ch, width, index = [], 16, 16.0, 0
        for i in range(3):
            layers = []
            for j in range(n):
                index += 1
                width += add_rate
                out_ch = int(round(width))
                stride = 2 if (i > 0 and j == 0) else 1
                layers.append(PyramidBlock(in_ch, out_ch, stride, drop_path * index / total))
                in_ch = out_ch
            stages.append(nn.Sequential(*layers))
        self.stages = nn.Sequential(*stages)
        self.bn = nn.BatchNorm2d(in_ch)
        self.fc = nn.Linear(in_ch, num_classes)

    def forward(self, x, mixstyle=None):
        """`mixstyle` = (apply, lam, perm, eps) from MixStyleSampler.draw, or None. Without draws, and always in
        evaluation mode, this is exactly the plain PyramidNet forward."""
        x = self.stem(x)
        if mixstyle is None or not self.training:
            x = self.stages(x)
        else:
            apply, lam, perm, eps = mixstyle
            layer = 0
            for index, stage in enumerate(self.stages, 1):
                x = stage(x)
                if index in MIXSTYLE_AFTER_STAGES:
                    x = mix_feature_styles(x, apply[layer], lam[layer], perm[layer], eps)
                    layer += 1
        x = F.relu(self.bn(x))
        return self.fc(torch.flatten(F.adaptive_avg_pool2d(x, 1), 1))


ARCH_DEFAULTS = {"implementation_version": 1, "drop_path": 0.0}


def architecture_from_config():
    return {"drop_path": DROP_PATH}


def validate_architecture(options):
    rate = options["drop_path"]
    if isinstance(rate, bool) or not isinstance(rate, (int, float)) or not 0 <= rate < 1:
        raise ValueError(f"DROP_PATH = {rate!r}; choose a rate in [0, 1) (0 = no stochastic depth)")


def describe_architecture(options):
    # The rate is saved with the options and enters the architecture fingerprint: resuming with another rate is
    # rejected, and evaluation rebuilds the saved rate whatever the current DROP_PATH.
    return f"{MODEL_DESCRIPTION}, stochastic depth DROP_PATH={float(options['drop_path'])!r}"


def architecture_feature(options):
    return "stochdepth" if options["drop_path"] > 0 else None


def expected_param_count(options):
    return EXPECTED_PARAMS  # stochastic depth adds no parameters


def architecture_topology(options):
    """Stem width, output channels of each stage (the width grows by alpha/54 in every block) and blocks per stage."""
    return {"stem_channels": 16, "stage_channels": [49, 81, 114], "blocks": [18, 18, 18]}


def make_model(num_classes=10, architecture=None):
    options = resolve_architecture(architecture)
    return PyramidNet(depth=110, alpha=98, drop_path=options["drop_path"], num_classes=num_classes)

In [ ]:
PARAM_BUDGET = 5_000_000


def init_weights(model):
    """Kaiming-normal (fan_out) convs and unit/zero BN, as in torchvision's ResNet."""
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def count_params(model):
    """Number of trainable parameters (conv/linear weights and biases, BN gamma/beta)."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


ARCHITECTURE_IMPLEMENTATION_VERSION = 1  # Bump when the meaning of a switch/forward pass changes.


def resolve_architecture(options=None):
    """None reads UI switches; an explicit dict (including {}) uses saved/baseline options."""
    options = architecture_from_config() if options is None else options
    if not isinstance(options, dict):
        raise ValueError("Architecture options must be a dictionary")
    unknown = set(options) - set(ARCH_DEFAULTS)
    if unknown:
        raise ValueError(f"Unknown architecture options for model {MODEL}: {sorted(unknown)}")
    resolved = dict(ARCH_DEFAULTS, **options)
    version = resolved["implementation_version"]
    if type(version) is not int or version != ARCHITECTURE_IMPLEMENTATION_VERSION:
        raise ValueError("Unsupported architecture implementation_version; use the matching notebook code")
    validate_architecture(resolved)
    return resolved


def saved_architecture(args):
    """Missing legacy fields mean baseline; malformed saved options must never read UI globals."""
    options = args.get("architecture", ARCH_DEFAULTS)
    if not isinstance(options, dict):
        raise ValueError("Saved architecture options must be a dictionary")
    return resolve_architecture(options)


def architecture_details(options):
    """Resolved topology accompanies options in configs, checkpoints and evaluation reports."""
    details = resolve_architecture(options)
    if MODEL not in ("A", "B", "C"):  # E/F/G define architecture_topology next to their model (section 4)
        return dict(details, model=MODEL, **architecture_topology(details), num_classes=NUM_CLASSES)
    channels = WIDTH_PRESETS[details["width_preset"]] if MODEL == "B" else (64, 128, 256)
    return dict(details, model=MODEL, stem_channels=16 if MODEL == "B" else 64,
                stage_channels=list(channels), blocks=[3, 3, 3] if MODEL == "B" else [3, 5, 3],
                num_classes=NUM_CLASSES)


def build_model(num_classes=NUM_CLASSES, architecture=None):
    """Build this notebook's model, check the 5M budget, and return (model, n_trainable_params)."""
    options = resolve_architecture(architecture)
    model = init_weights(make_model(num_classes, architecture=options))
    # Plain attributes do not alter legacy state_dict keys or baseline repr/fingerprints.
    model._project_architecture = options
    model._project_description = describe_architecture(options)
    n_params = count_params(model)
    assert n_params <= PARAM_BUDGET, f"model {MODEL} has {n_params:,} trainable params > {PARAM_BUDGET:,}"
    return model, n_params


_model, N_PARAMS = build_model()
print(f"Model {MODEL}: {_model._project_description}")
print(f"Architecture options: {_model._project_architecture}")
print(f"Trainable parameters: {N_PARAMS:,} (budget {PARAM_BUDGET:,})")
assert N_PARAMS <= PARAM_BUDGET
assert N_PARAMS == expected_param_count(_model._project_architecture), "Unexpected architecture parameter count"
if SHOW_SUMMARY:
    try:
        from torchinfo import summary
    except ImportError:
        print("torchinfo is not installed; layer summary skipped (the parameter count above is complete)")
    else:
        print(summary(_model, input_size=(1, 3, 32, 32), depth=3, col_names=("output_size", "num_params"),
                      device="cpu", verbose=0))
del _model

## 5. Training

The recipe is SGD with Nesterov momentum, warmup and cosine decay, optional Mixup, and EMA.
The current P13 preset explicitly uses basic crop/flip and zero label smoothing. Mixup is applied after
normalization to training batches only: a probability0.5 trigger, one Beta(0.2,0.2) lambda and shuffled partners.
The loss and augmented training accuracy weight both target labels by lambda and1−lambda. Validation/test
remain unmixed. A dedicated PCG64 stream seeded by `[SEED, 0x4D495855]` is saved in every full checkpoint;
missing or incompatible Mixup state is rejected on resume. Pre-Mixup checkpoints imply disabled Mixup.
Training precision defaults to bf16 on an A100; fp16 uses a gradient scaler and only successful optimizer
updates advance the scheduler and EMA. Validation uses fp32 and ordinary cross-entropy.

**SAM (E, `SAM=True`, default off).** Sharpness-aware minimization (Foret et al., ICLR 2021) wraps the same SGD.
Each update runs a first forward/backward at the weights w, moves to w + e with e = ρ·g/‖g‖ (`SAM_RHO`, default
0.05; `SAM_ADAPTIVE=True` gives ASAM, Kwon et al. 2021, e = ρ·w²g/‖wg‖ elementwise, typically ρ ≈ 2.0), runs a second
forward/backward there on the same augmented batch (same Mixup partners and MixStyle draws), restores w exactly and
lets SGD (momentum, Nesterov and weight decay unchanged) step with the second gradient. The perturbation uses the loss
gradient only (weight decay stays in SGD), as in the reference implementation `davda54/sam`. BatchNorm normalizes the
second pass with its batch statistics, but its running statistics are restored to their first-pass update. Stochastic
depth draws independent masks in the two passes. The LR schedule and EMA advance once per update; `train_loss` /
`train_acc` come from the first pass, the CSV adds `sam_loss` (second pass) and `sam_grad_norm`. SAM keeps no state
between updates. It doubles the forward/backward cost (about 2× s/epoch). SAM rejects fp16 (use bf16 or fp32) and the
CUDA-graph compile modes `"reduce-overhead"` / `"max-autotune"` (not verified with the two-pass step; use
`COMPILE="default"`, `"max-autotune-no-cudagraphs"` or `"off"`).

**MixStyle (E, `MIXSTYLE=True`, default off; section 4).** The draws (mix or not with probability `MIXSTYLE_P`, one
Beta(α, α) weight per image, a partner permutation; per step and layer) come from a dedicated PCG64 stream seeded by
`[SEED, 0x4D495853]`, independent of the global RNG, and its state is saved in every full checkpoint; missing state
is rejected on resume. The CSV adds `mixstyle_fraction`. Runs saved before these switches existed mean SAM and
MixStyle off; resuming with either switch or its settings changed is rejected. Settings of a disabled switch are
ignored. With `RUN_NAME=None` the default feature words become `sam-ema`, `mixstyle-ema` or `sam-mixstyle`.

An improved `<RUN_ID>_best-ckpt.pt` is saved immediately. `<RUN_ID>_last-ckpt.pt` is saved every `CKPT_EVERY` epochs and at completion.
Each is a complete resume point. Resume selects the newest readable compatible checkpoint and checks its
dataset fingerprint, architecture, exact split/subset, and training settings. Interrupted file writes never
fall back to overwriting an existing checkpoint with a partial copy. Resume is at a saved epoch boundary;
unsaved epochs are rerun. Numerical results may differ across devices or library versions.

Logs, configuration and checkpoints are in separate folders under the same run ID. The final report includes
compute time, end-to-end session time and peak CUDA allocation. Keep the Colab results on mounted Drive.

Mixup CSV logs add `mixup_batches`, `mixup_fraction` and `mixup_lambda_mean` (triggered batches only;1.0 when none trigger). The saved `train_acc` is lambda-weighted augmented accuracy, not clean training accuracy. Disabling Mixup retains the old CSV schema and ordinary hard-label objective.


In [ ]:
# Training-only Mixup; the original model fingerprint is unchanged.
MIXUP_DEFAULTS = {"mixup": False, "mixup_alpha": 0.2, "mixup_prob": 0.5,
                  "mixup_implementation_version": 1}
MIXUP_SEED_DOMAIN = 0x4D495855


def mixup_metadata(args):
    settings = {key: args.get(key, value) for key, value in MIXUP_DEFAULTS.items()}
    return {**settings, "rng": "numpy.PCG64", "seed_sequence": [args["seed"], MIXUP_SEED_DOMAIN],
            "application": "training only, after crop/flip and normalization",
            "train_accuracy": "lambda-weighted augmented accuracy" if settings["mixup"] else "hard-label augmented accuracy",
            "lambda_mean": "triggered batches only; 1.0 when no batch triggers"}


class BatchMixup:
    """One Beta draw and shuffled partner batch per trigger, with an independent RNG."""

    def __init__(self, alpha, probability, seed):
        if (isinstance(alpha, bool) or not isinstance(alpha, (int, float)) or
                not math.isfinite(alpha) or alpha <= 0):
            raise ValueError("MIXUP_ALPHA must be finite and positive")
        if (isinstance(probability, bool) or not isinstance(probability, (int, float)) or
                not math.isfinite(probability) or not 0 <= probability <= 1):
            raise ValueError("MIXUP_PROB must be finite and in [0,1]")
        if not isinstance(seed, int) or isinstance(seed, bool) or seed < 0:
            raise ValueError("Mixup requires a nonnegative integer SEED")
        self.alpha, self.probability, self.seed = float(alpha), float(probability), seed
        self.rng = np.random.Generator(np.random.PCG64(np.random.SeedSequence([seed, MIXUP_SEED_DOMAIN])))

    def apply(self, x, y):
        if self.probability == 0 or self.rng.random() >= self.probability:
            return x, y, 1.0, False
        lam = float(self.rng.beta(self.alpha, self.alpha))
        order = torch.as_tensor(self.rng.permutation(len(y)), device=y.device, dtype=torch.long)
        return lam * x + (1.0 - lam) * x[order], y[order], lam, True

    def state_dict(self):
        return {"implementation_version": 1, "alpha": self.alpha, "probability": self.probability,
                "seed": self.seed, "seed_domain": MIXUP_SEED_DOMAIN,
                "rng_state": copy.deepcopy(self.rng.bit_generator.state)}

    def load_state_dict(self, state):
        expected = self.state_dict()
        if not isinstance(state, dict) or set(state) != set(expected):
            raise ValueError("Checkpoint Mixup state is missing or malformed; cannot reproduce resume")
        if any(state[k] != expected[k] for k in expected if k != "rng_state"):
            raise ValueError("Checkpoint Mixup settings or implementation version differ")
        try:
            if state["rng_state"]["bit_generator"] != "PCG64":
                raise ValueError("Expected PCG64")
            self.rng.bit_generator.state = copy.deepcopy(state["rng_state"])
        except (KeyError, TypeError, ValueError) as exc:
            raise ValueError("Checkpoint Mixup RNG state is invalid") from exc


def mixup_loss_and_correct(logits, y, partner_y, lam, criterion):
    """Weighted hard-target CE equals soft-target CE, without allocating a one-hot target."""
    loss = criterion(logits, y)
    if lam != 1.0:
        loss = lam * loss + (1.0 - lam) * criterion(logits, partner_y)
    with torch.no_grad():
        prediction = logits.argmax(1)
        correct = lam * (prediction == y).float().sum() + (1.0 - lam) * (prediction == partner_y).float().sum()
    return loss, correct


# Optional SAM / MixStyle training switches, used by E's trainer (section 1 there); off by default. A run saved
# without these fields trained without them. Neither changes the model, its parameters or the model fingerprint.
SAM_DEFAULTS = {"sam": False, "sam_rho": 0.05, "sam_adaptive": False, "sam_implementation_version": 1}
MIXSTYLE_DEFAULTS = {"mixstyle": False, "mixstyle_p": 0.5, "mixstyle_alpha": 0.1, "mixstyle_eps": 1e-6,
                     "mixstyle_implementation_version": 1}
MIXSTYLE_SEED_DOMAIN = 0x4D495853  # "MIXS"
SAM_LOG_FIELDS = ["sam_loss", "sam_grad_norm"]  # epoch means of the second-pass loss and of ||T g||
MIXSTYLE_LOG_FIELDS = ["mixstyle_fraction"]  # fraction of (step, MixStyle layer) pairs that mixed


def _positive_finite(value):
    return not isinstance(value, bool) and isinstance(value, (int, float)) and math.isfinite(value) and value > 0


def validate_sam_mixstyle_args(args):
    """Fill in (absent = off) and check the SAM / MixStyle settings in place. Settings of a disabled switch are reset
    to their defaults, so they never block resuming a run saved before the switch existed."""
    for key, default in {**SAM_DEFAULTS, **MIXSTYLE_DEFAULTS}.items():
        if not hasattr(args, key):
            setattr(args, key, default)
    for key in ("sam", "sam_adaptive", "mixstyle"):
        if not isinstance(getattr(args, key), bool):
            raise ValueError(f"{key.upper()} must be True or False")
    for key in ("sam_implementation_version", "mixstyle_implementation_version"):
        if type(getattr(args, key)) is not int or getattr(args, key) != 1:
            raise ValueError(f"Unsupported {key.split('_')[0]} implementation version")
    if args.sam:
        if not _positive_finite(args.sam_rho):
            raise ValueError(f"SAM_RHO = {args.sam_rho!r}; choose a finite positive radius (SAM 0.05; ASAM about 2.0)")
        if args.sam_adaptive and args.sam_rho < 0.1:
            raise ValueError(f"SAM_ADAPTIVE=True with SAM_RHO={args.sam_rho!r}: ASAM scales the step by |w|, so plain-SAM "
                             "radii are far too small; the reference implementation suggests SAM_RHO about 2.0")
        if getattr(args, "compile", "off") in CUDAGRAPH_COMPILE_MODES:
            raise ValueError(f"SAM with COMPILE={args.compile!r} is not supported: the two-pass SAM step has not been "
                             "verified with CUDA graphs; use COMPILE='default', 'max-autotune-no-cudagraphs' or 'off'")
        if getattr(args, "amp", None) == "fp16":
            raise ValueError("SAM needs bf16 or fp32 training (AMP='bf16', 'auto' on a bf16 GPU, or 'off'); "
                             "the fp16 gradient scaler is not supported with two backward passes")
    elif args.sam_adaptive:
        raise ValueError("SAM_ADAPTIVE=True selects ASAM, which also needs SAM=True")
    else:
        args.sam_rho, args.sam_adaptive = SAM_DEFAULTS["sam_rho"], SAM_DEFAULTS["sam_adaptive"]
    if args.mixstyle:
        MixStyleSampler(args.mixstyle_p, args.mixstyle_alpha, args.mixstyle_eps, args.seed, 1)  # checks the values
    else:
        for key in ("mixstyle_p", "mixstyle_alpha", "mixstyle_eps"):
            setattr(args, key, MIXSTYLE_DEFAULTS[key])
    return args


def sam_mixstyle_label(args):
    """Default feature words of a SAM and/or MixStyle run when RUN_NAME is None (None otherwise)."""
    words = [word for word, on in (("sam", args.sam), ("mixstyle", args.mixstyle)) if on]
    if not words:
        return None
    return "-".join(words) if len(words) == 2 else words[0] + ("-ema" if args.ema else "")


def sam_metadata(args):
    settings = {key: args.get(key, value) for key, value in SAM_DEFAULTS.items()}
    if not settings["sam"]:
        return settings
    return {**settings, "variant": "ASAM (T = |w|)" if settings["sam_adaptive"] else "SAM (T = 1)",
            "perturbation": "e = rho * T^2 g / (||T g|| + 1e-12) from the loss gradient g only (weight decay stays "
                            "in SGD), all parameters, as in github.com/davda54/sam",
            "second_pass": "same augmented batch, Mixup and MixStyle draws; independent stochastic-depth masks; "
                           "BatchNorm normalizes with batch statistics, its running statistics are restored after",
            "update": "base SGD step with the second-pass gradient at the exactly restored weights; LR schedule and "
                      "EMA advance once per update",
            "train_metrics": "train_loss/train_acc from the first pass (unperturbed weights); sam_loss from the second",
            "cost": "two forward/backward passes per update (about 2x s/epoch)"}


def mixstyle_metadata(args, after_stages=()):
    settings = {key: args.get(key, value) for key, value in MIXSTYLE_DEFAULTS.items()}
    if not settings["mixstyle"]:
        return settings
    return {**settings, "after_stages": list(after_stages), "mixing": "random partner (batch permutation)",
            "statistics": "per image and channel over spatial positions; float32, unbiased variance + eps, detached",
            "draws": "per step and layer: apply with probability p, Beta(alpha, alpha) lam per image, permutation",
            "rng": "numpy.PCG64", "seed_sequence": [args["seed"], MIXSTYLE_SEED_DOMAIN],
            "application": "training forward only (both SAM passes share one draw); identity in evaluation"}


class SharpnessAwareMinimization:
    """SAM (Foret et al., ICLR 2021) around the existing optimizer; ASAM (Kwon et al., ICML 2021) if adaptive.

    As in the reference implementation github.com/davda54/sam: e = rho * T^2 g / (||T g|| + 1e-12), with T = 1 (SAM)
    or |w| elementwise (ASAM) for every parameter, where g is the loss gradient of the first pass (weight decay is
    added inside the base SGD step, never to the perturbation). second_pass(loss_fn) saves w exactly, moves to w + e,
    backpropagates loss_fn() there, then restores w bit for bit and the BatchNorm running statistics of the first
    pass. The caller then steps the base optimizer, which sees only the second-pass gradient. No state is kept
    between updates, so checkpoints and resume need nothing extra.
    """

    def __init__(self, model, optimizer, rho, adaptive=False):
        if not _positive_finite(rho):
            raise ValueError("SAM_RHO must be finite and positive")
        self.optimizer, self.rho, self.adaptive = optimizer, float(rho), bool(adaptive)
        self.params = [p for group in optimizer.param_groups for p in group["params"]]
        self.saved = [torch.empty_like(p) for p in self.params]
        self.bn_buffers = [b for m in model.modules() if isinstance(m, nn.modules.batchnorm._BatchNorm)
                           for b in (m.running_mean, m.running_var, m.num_batches_tracked) if b is not None]
        self.bn_saved = [torch.empty_like(b) for b in self.bn_buffers]

    @torch.no_grad()
    def ascend(self):
        """Save w and the BN running statistics, move to w + e; returns ||T g|| (0-dim tensor, no host sync)."""
        params = [p for p in self.params if p.grad is not None]
        if not params:
            raise RuntimeError("SAM: no parameter has a first-pass gradient")
        torch._foreach_copy_(self.saved, self.params)
        if self.bn_buffers:
            torch._foreach_copy_(self.bn_saved, self.bn_buffers)
        grads = [p.grad for p in params]
        if self.adaptive:
            scales = torch._foreach_abs(params)
            grads = torch._foreach_mul(grads, scales)  # T g
        norm = torch.linalg.vector_norm(torch.stack(torch._foreach_norm(grads)))
        steps = torch._foreach_mul(grads, scales) if self.adaptive else [g.clone() for g in grads]  # T^2 g
        torch._foreach_mul_(steps, self.rho / (norm + 1e-12))
        torch._foreach_add_(params, steps)
        self.optimizer.zero_grad(set_to_none=True)
        return norm

    @torch.no_grad()
    def restore(self):
        """Back to the exact first-pass weights and BN running statistics; .grad keeps the second-pass gradient."""
        torch._foreach_copy_(self.params, self.saved)
        if self.bn_buffers:
            torch._foreach_copy_(self.bn_buffers, self.bn_saved)

    def second_pass(self, loss_fn):
        """Gradient at w + e from loss_fn() (forward + loss on the same batch); returns (||T g||, detached loss)."""
        norm = self.ascend()
        loss = loss_fn()
        loss.backward()
        self.restore()
        return norm, loss.detach().float()


class MixStyleSampler:
    """MixStyle training draws (Zhou et al., ICLR 2021) from an independent RNG, so the global RNG, the data order and
    the model are unaffected (like BatchMixup).

    Per step and MixStyle layer: whether the layer mixes (probability p), one Beta(alpha, alpha) weight per image and
    a random partner permutation ("random" mixing). Weights and partners are drawn even when a layer does not mix,
    so the stream advances by the same amount every step. The draws are passed to the model's training forward.
    """

    def __init__(self, p, alpha, eps, seed, layers):
        if isinstance(p, bool) or not isinstance(p, (int, float)) or not math.isfinite(p) or not 0 <= p <= 1:
            raise ValueError(f"MIXSTYLE_P = {p!r}; choose a probability in [0, 1] (paper 0.5)")
        if not _positive_finite(alpha):
            raise ValueError(f"MIXSTYLE_ALPHA = {alpha!r}; choose a finite positive Beta parameter (paper 0.1)")
        if not _positive_finite(eps):
            raise ValueError(f"MIXSTYLE_EPS = {eps!r}; choose a small finite positive value (paper 1e-6)")
        if not isinstance(seed, int) or isinstance(seed, bool) or seed < 0:
            raise ValueError("MixStyle requires a nonnegative integer SEED")
        if not isinstance(layers, int) or isinstance(layers, bool) or layers < 1:
            raise ValueError("MixStyle needs a model with at least one MixStyle layer")
        self.p, self.alpha, self.eps, self.seed, self.layers = float(p), float(alpha), float(eps), seed, layers
        self.rng = np.random.Generator(np.random.PCG64(np.random.SeedSequence([seed, MIXSTYLE_SEED_DOMAIN])))
        self.applied = 0  # mixed (step, layer) pairs since the last pop_applied()

    def draw(self, batch_size, device):
        """(apply, lam, perm, eps): apply (layers,) bool, lam (layers, B, 1, 1, 1) float32, perm (layers, B) int64."""
        apply = np.empty(self.layers, dtype=bool)
        lam = np.empty((self.layers, batch_size), dtype=np.float32)
        perm = np.empty((self.layers, batch_size), dtype=np.int64)
        for layer in range(self.layers):
            apply[layer] = self.rng.random() < self.p
            lam[layer] = self.rng.beta(self.alpha, self.alpha, size=batch_size)
            perm[layer] = self.rng.permutation(batch_size)
        self.applied += int(apply.sum())
        lam_t = torch.from_numpy(lam).view(self.layers, batch_size, 1, 1, 1)
        return (_to_dev(torch.from_numpy(apply), device), _to_dev(lam_t, device),
                _to_dev(torch.from_numpy(perm), device), self.eps)

    def pop_applied(self):
        applied, self.applied = self.applied, 0
        return applied

    def state_dict(self):
        return {"implementation_version": 1, "p": self.p, "alpha": self.alpha, "eps": self.eps, "seed": self.seed,
                "seed_domain": MIXSTYLE_SEED_DOMAIN, "layers": self.layers,
                "rng_state": copy.deepcopy(self.rng.bit_generator.state)}

    def load_state_dict(self, state):
        expected = self.state_dict()
        if not isinstance(state, dict) or set(state) != set(expected):
            raise ValueError("Checkpoint MixStyle state is missing or malformed; cannot reproduce resume")
        if any(state[k] != expected[k] for k in expected if k != "rng_state"):
            raise ValueError("Checkpoint MixStyle settings or implementation version differ")
        try:
            if state["rng_state"]["bit_generator"] != "PCG64":
                raise ValueError("Expected PCG64")
            self.rng.bit_generator.state = copy.deepcopy(state["rng_state"])
        except (KeyError, TypeError, ValueError) as exc:
            raise ValueError("Checkpoint MixStyle RNG state is invalid") from exc


def synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)  # also seeds CUDA and MPS


def get_rng_state():
    state = {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state()}
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    if torch.backends.mps.is_available():
        state["mps"] = torch.mps.get_rng_state()
    return state


def set_rng_state(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if "cuda" in state and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state["cuda"])
    if "mps" in state and torch.backends.mps.is_available():
        torch.mps.set_rng_state(state["mps"])


AMP_DTYPES = {"bf16": torch.bfloat16, "fp16": torch.float16}


def autocast(device, mode):
    """bf16 or fp16 autocast on CUDA (mode 'bf16' / 'fp16'; True means fp16); a no-op otherwise."""
    mode = "fp16" if mode is True else mode
    enabled = device.type == "cuda" and mode in AMP_DTYPES
    return torch.autocast(device.type if device.type == "cuda" else "cpu",
                          dtype=AMP_DTYPES.get(mode, torch.float16), enabled=enabled)


class ModelEMA:
    """Exponential moving average of weights and BN running stats (cf. timm's ModelEmaV2).

    The effective decay is min(decay, (1 + n) / (10 + n)) after n updates, so early on the average
    follows the model closely instead of being dominated by the random initialization.
    """

    def __init__(self, model, decay=0.999):
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay = decay
        self.updates = 0

    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        d = min(self.decay, (1 + self.updates) / (10 + self.updates))
        torch._foreach_lerp_(list(self.module.parameters()), list(model.parameters()), 1.0 - d)
        for eb, b in zip(self.module.buffers(), model.buffers()):
            if eb.dtype.is_floating_point:
                eb.lerp_(b, 1.0 - d)
            else:
                eb.copy_(b)  # num_batches_tracked

    def state_dict(self):
        return {"module": self.module.state_dict(), "updates": self.updates, "decay": self.decay}

    def load_state_dict(self, state):
        self.module.load_state_dict(state["module"])
        self.updates = state["updates"]
        self.decay = state["decay"]


# Optional speed switches (README "Compilation"): recorded with each run, never part of the recipe, the model
# fingerprint or a resume check. Compiled training is not bit-identical to uncompiled training.
PERFORMANCE_DEFAULTS = {"compile": "off", "benchmark_steps": 0}
CUDAGRAPH_COMPILE_MODES = ("reduce-overhead", "max-autotune")  # torch.compile modes that replay CUDA graphs
BENCHMARK_WARMUP_STEPS = 10  # untimed first benchmark steps: compilation, CUDA graph recording, cuDNN autotuning


def compile_module(module, mode):
    """Callable for training-step forwards: `module` itself for "off", else a torch.compile wrapper sharing its weights.

    Shapes are static (dynamic=False): every training batch is full because the last partial batch is dropped. Only
    the training forward/backward goes through the wrapper; checkpoints, EMA, evaluation, TTA and prediction use the
    original module, so saved state_dict keys never carry the wrapper's "_orig_mod." prefix.
    """
    if mode == "off":
        return module
    return torch.compile(module, mode=None if mode == "default" else mode, dynamic=False)


def compile_step_begin(*modes):
    """Start of a training step in a CUDA graph mode: the previous step's graph outputs may now be overwritten."""
    if any(mode in CUDAGRAPH_COMPILE_MODES for mode in modes):
        mark = getattr(getattr(torch, "compiler", None), "cudagraph_mark_step_begin", None)
        if mark is not None:
            mark()


class EpochTimer:
    """The [timing] lines: this session's first epoch (with compilation) and its steady-state median epoch time."""

    def __init__(self, label, compiled, steady_after=5):
        self.label, self.compiled, self.steady_after, self.times = label, compiled, steady_after, []

    def add(self, seconds):
        """Record one epoch (training + validation); returns a line after the first and the steady_after-th epoch."""
        self.times.append(seconds)
        if len(self.times) == 1:
            return (f"[timing] {self.label} | first epoch of this session {seconds:.1f}s"
                    + (" (includes compilation)" if self.compiled else ""))
        return self.summary() if len(self.times) == self.steady_after else None

    def summary(self):
        if len(self.times) < 2:
            return None
        return (f"[timing] {self.label} | first epoch {self.times[0]:.1f}s | steady state "
                f"{float(np.median(self.times[1:])):.1f}s/epoch (median of the {len(self.times) - 1} later epochs)")

    def final(self):
        """The summary at the end of training, unless add() has just returned the same one."""
        return None if len(self.times) == self.steady_after else self.summary()


def warmup_cosine_lambda(total_steps, warmup_steps):
    """LR multiplier per optimizer step: linear warmup to 1, then cosine decay to 0."""

    def f(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    return f


def build_optimizer(model, args):
    if args.wd_skip_bn_bias:  # no weight decay on BN gamma/beta and biases (all 1-D tensors)
        decay = [p for p in model.parameters() if p.requires_grad and p.ndim > 1]
        no_decay = [p for p in model.parameters() if p.requires_grad and p.ndim <= 1]
        groups = [{"params": decay, "weight_decay": args.weight_decay},
                  {"params": no_decay, "weight_decay": 0.0}]
    else:
        groups = [{"params": [p for p in model.parameters() if p.requires_grad],
                   "weight_decay": args.weight_decay}]
    return torch.optim.SGD(groups, lr=args.lr, momentum=args.momentum, nesterov=args.nesterov)


def replace_file(src, dst):
    """Publish a completed file by rename; never overwrite a good checkpoint by partial copy."""
    try:
        os.replace(src, dst)
    except OSError as exc:
        raise OSError(f"Could not atomically publish {dst}; the previous file is preserved. "
                      "Use a filesystem that supports same-directory rename.") from exc


def atomic_save(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    replace_file(tmp, path)


def write_bytes_atomic(data, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_bytes(data)
    replace_file(tmp, path)


def load_checkpoint(path):
    state = torch.load(path, map_location="cpu", weights_only=False)
    if not isinstance(state, dict) or "args" not in state:
        return state
    original = state["args"].get("run_name")
    canonical = canonical_run_id(original)
    if original != canonical:
        state.setdefault("original_identity", {"run_id": original,
                         "model_version": state["args"].get("model_version")})
        state["args"] = dict(state["args"], run_name=canonical,
                             model_version=int(re.search(r"-ver-(\d+)-", canonical).group(1)),
                             task_id=run_task_id(canonical))
    return state


class CSVLogger:
    """Appends one row per epoch; on resume, drops rows past the checkpoint epoch."""

    FIELDS = ["epoch", "lr", "train_loss", "train_acc", "val_loss", "val_acc",
              "ema_val_loss", "ema_val_acc", "epoch_time_s", "elapsed_min", "skipped_updates"]

    def __init__(self, path, resume_epoch=None, mixup=False, extra=()):
        self.fields = (self.FIELDS + (["mixup_batches", "mixup_fraction", "mixup_lambda_mean"] if mixup else [])
                       + list(extra))  # extra: SAM_LOG_FIELDS / MIXSTYLE_LOG_FIELDS of enabled switches
        self.path = Path(path)
        if resume_epoch is not None and self.path.exists():
            with open(self.path, newline="") as f:
                reader = csv.DictReader(f)
                if reader.fieldnames != self.fields:
                    raise ValueError("Saved CSV columns differ from the requested training recipe")
                rows = [r for r in reader if int(r["epoch"]) <= resume_epoch]
            self._write(rows, "w")
        else:
            self._write([], "w")

    def _write(self, rows, mode):
        with open(self.path, mode, newline="") as f:
            w = csv.DictWriter(f, fieldnames=self.fields)
            if mode == "w":
                w.writeheader()
            w.writerows(rows)

    def log(self, row):
        self._write([{k: row.get(k, "") for k in self.fields}], "a")


@torch.inference_mode()
def predict_logits(model, images_u8, mean, std, device, batch_size=1000, flip=False, channels_last=False):
    """Logits for uint8 NHWC images (numpy or tensor); returns a CPU float tensor.

    With flip=True, returns the logits of horizontally flipped images instead.
    """
    model.eval()
    mean_t = torch.tensor(mean, device=device).view(1, 3, 1, 1)
    std_t = torch.tensor(std, device=device).view(1, 3, 1, 1)
    out = []
    for i in range(0, len(images_u8), batch_size):
        xb = torch.as_tensor(np.asarray(images_u8[i:i + batch_size])).to(device)
        x = to_float_nchw(xb, mean_t, std_t, channels_last)
        if flip:
            x = x.flip(3)
        out.append(model(x).float().cpu())
    return torch.cat(out)


def predict_probs(model, images_u8, mean, std, device, batch_size=1000, tta=True, channels_last=False):
    """Softmax probabilities; with tta=True, averages the original and horizontally flipped view."""
    kw = dict(batch_size=batch_size, channels_last=channels_last)
    probs = F.softmax(predict_logits(model, images_u8, mean, std, device, **kw), 1)
    if tta:
        probs = 0.5 * (probs + F.softmax(predict_logits(model, images_u8, mean, std, device, flip=True, **kw), 1))
    return probs


def evaluate(model, images_u8, labels, mean, std, device, batch_size=1000, channels_last=False):
    """Plain cross-entropy (no label smoothing) and accuracy in percent, without TTA."""
    logits = predict_logits(model, images_u8, mean, std, device, batch_size, channels_last=channels_last)
    y = torch.as_tensor(labels)
    loss = F.cross_entropy(logits, y).item()
    acc = (logits.argmax(1) == y).float().mean().item() * 100
    return loss, acc

CHECKPOINT_FORMAT = 3


def write_json_atomic(value, path):
    write_bytes_atomic((json.dumps(value, indent=2) + "\n").encode(), path)


def model_fingerprint(model):
    options = getattr(model, "_project_architecture", ARCH_DEFAULTS)
    definition = getattr(model, "_project_description", MODEL_DESCRIPTION) + "\n" + repr(model)
    # Preserve the exact legacy baseline hash. New variants also bind options and implementation version.
    if options != ARCH_DEFAULTS:
        definition += "\n" + json.dumps(options, sort_keys=True)
    return hashlib.sha256(definition.encode()).hexdigest()


def validate_checkpoint(ckpt, fingerprint, expected_model_hash=None, run_id=None):
    if ckpt.get("format") != CHECKPOINT_FORMAT or "cache_fingerprint" not in ckpt:
        raise ValueError("Legacy checkpoint lacks verified dataset identity; start a fresh run with this notebook")
    if ckpt["cache_fingerprint"] != fingerprint:
        raise ValueError("Checkpoint dataset fingerprint differs from the current cache; refusing resume/evaluation")
    if ckpt["args"]["model"] != MODEL:
        raise ValueError("Checkpoint belongs to another model")
    saved_options = saved_architecture(ckpt["args"])
    if "architecture" in ckpt and ckpt["architecture"] != architecture_details(saved_options):
        raise ValueError("Checkpoint resolved architecture differs from its saved options")
    if expected_model_hash is not None and ckpt.get("model_fingerprint") != expected_model_hash:
        raise ValueError("Checkpoint architecture differs from this notebook")
    if run_id is not None and canonical_run_id(ckpt["args"]["run_name"]) != canonical_run_id(run_id):
        raise ValueError("Checkpoint belongs to another run ID")


def recover_checkpoints(run_dir, fingerprint, expected_model_hash):
    """Best is independently durable and is also a complete resume point."""
    valid = {}
    for name in ("last", "best"):
        path = checkpoint_path(run_dir, name)
        if not path.exists():
            continue
        try:
            ckpt = load_checkpoint(path)
        except Exception as exc:
            print(f"Warning: unreadable {path.name}: {exc}; trying the other checkpoint")
            continue
        validate_checkpoint(ckpt, fingerprint, expected_model_hash, Path(run_dir).name)
        valid[name] = ckpt
    if not valid:
        raise FileNotFoundError("No readable, compatible checkpoint to resume")
    selected = max(valid, key=lambda name: (valid[name]["epoch"], name == "last"))
    ckpt = valid[selected]
    if ckpt["args"]["val_size"]:
        best_ckpt = valid.get("best")
        if best_ckpt is None:
            # Repairable only when a readable checkpoint itself contains the winning weights.
            if ckpt["best"]["epoch"] != ckpt["epoch"]:
                raise RuntimeError("Best checkpoint is missing/corrupt and its weights are not recoverable from last")
            atomic_save(ckpt, checkpoint_path(run_dir, "best"))
            best_ckpt = ckpt
        if best_ckpt["best"]["epoch"] != best_ckpt["epoch"]:
            raise RuntimeError("best-ckpt.pt does not contain its recorded best epoch")
        if ckpt["best"]["acc"] > best_ckpt["best"]["acc"]:
            if ckpt["best"]["epoch"] != ckpt["epoch"]:
                raise RuntimeError("Checkpoint metadata refers to unavailable best weights")
            atomic_save(ckpt, checkpoint_path(run_dir, "best"))
            best_ckpt = ckpt
        ckpt["best"] = dict(best_ckpt["best"])
    print(f"Resume checkpoint: {checkpoint_path(run_dir, selected).name}, epoch {ckpt['epoch']}")
    return ckpt


def save_epoch_checkpoints(state, run_dir, is_best, save_last):
    # Publish best FIRST, independently of the last-checkpoint interval.
    # If interrupted between writes, resume can use the newer complete best-ckpt.pt.
    if is_best:
        atomic_save(state, checkpoint_path(run_dir, "best"))
    if save_last:
        atomic_save(state, checkpoint_path(run_dir, "last"))


class BenchmarkFinished(Exception):
    """Raised by train() after a BENCHMARK_STEPS timing run: nothing was saved, so Run all deliberately stops here."""

    def __init__(self, result):
        self.result = result
        super().__init__(f"BENCHMARK_STEPS = {result['steps']} finished (COMPILE = {result['compile']!r}); no run "
                         "folder or file was created. Set BENCHMARK_STEPS = 0 to train.")


def benchmark_training(args, model, images, labels, train_idx, val_idx, mean, std, device, images_path,
                       target_fn=None, loss_fn=None, target_timer=None, target_compile="off"):
    """Time args.benchmark_steps real training steps with the chosen COMPILE mode; nothing is written or kept.

    Builds the EMA, batches (real augmented data), optimizer, schedule and loss as train() does and repeats its step:
    Mixup, autocast forward through compile_module(model), loss, finite check, backward, optimizer/scheduler step,
    EMA update; with args.sam / args.mixstyle (E) also the SAM second pass and the MixStyle draws.
    The first BENCHMARK_WARMUP_STEPS steps (compilation, CUDA graph recording) are timed separately;
    validation of the model and EMA (uncompiled) is then timed after one untimed pass. The distillation notebook
    passes target_fn(x) (the teacher forward, timed by target_timer), loss_fn(logits, target, y, criterion,
    partner_y, lam) -> (loss, correct) and the teachers' compile mode. Returns the measurements as a dict.
    """
    steps, warmup = args.benchmark_steps, BENCHMARK_WARMUP_STEPS
    model.to(device)
    if args.channels_last:
        model.to(memory_format=torch.channels_last)
    ema = ModelEMA(model, args.ema_decay) if args.ema else None
    batches = TrainBatches(images, labels, train_idx, args, mean, std, device, images_path=images_path)
    optimizer = build_optimizer(model, args)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, warmup_cosine_lambda(
        args.epochs * batches.steps_per_epoch, round(args.warmup_epochs * batches.steps_per_epoch)))
    criterion = nn.CrossEntropyLoss(label_smoothing=args.label_smoothing)
    scaler = torch.amp.GradScaler("cuda", enabled=args.amp == "fp16" and device.type == "cuda")
    mixer = BatchMixup(args.mixup_alpha, args.mixup_prob, args.seed) if args.mixup else None
    sam = (SharpnessAwareMinimization(model, optimizer, args.sam_rho, args.sam_adaptive)
           if getattr(args, "sam", False) else None)
    styler = (MixStyleSampler(args.mixstyle_p, args.mixstyle_alpha, args.mixstyle_eps, args.seed,
                              getattr(model, "mixstyle_layers", 0)) if getattr(args, "mixstyle", False) else None)
    train_forward = compile_module(model, args.compile)

    def endless_batches():
        epoch = 0
        while True:
            yield from batches.epoch(epoch)
            epoch += 1

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    model.train()
    stream = endless_batches()
    loss_sum = torch.zeros((), device=device)
    correct = torch.zeros((), device=device)
    first_step_s = warmup_s = warmup_target_s = None
    synchronize(device)
    start = time.perf_counter()
    for i in range(warmup + steps):
        if i == warmup:
            synchronize(device)
            warmup_s = time.perf_counter() - start
            warmup_target_s = target_timer.pop_seconds() if target_timer is not None else None
            start = time.perf_counter()
        x, y = next(stream)
        compile_step_begin(args.compile, target_compile)
        optimizer.zero_grad(set_to_none=True)
        partner_y, lam = None, 1.0
        if mixer is not None:
            x, partner_y, lam, _ = mixer.apply(x, y)
        styles = {} if styler is None else {"mixstyle": styler.draw(y.size(0), device)}
        target = None
        if target_fn is not None:
            if target_timer is not None:
                target_timer.start()
            target = target_fn(x)
            if target_timer is not None:
                target_timer.stop()
        with autocast(device, args.amp):
            logits = train_forward(x, **styles)
            if loss_fn is not None:
                loss, batch_correct = loss_fn(logits, target, y, criterion, partner_y, lam)
            elif mixer is None:
                loss = criterion(logits, y)
                batch_correct = None
            else:
                loss, batch_correct = mixup_loss_and_correct(logits, y, partner_y, lam, criterion)
        if not torch.isfinite(loss).item():
            raise FloatingPointError(f"Nonfinite loss at benchmark step {i+1}")
        scaler.scale(loss).backward()
        if sam is not None:  # E only: same batch and draws at w + e (fp16 is rejected by validate_sam_mixstyle_args)
            def perturbed_loss():
                with autocast(device, args.amp):
                    out = train_forward(x, **styles)
                    return (criterion(out, y) if mixer is None
                            else mixup_loss_and_correct(out, y, partner_y, lam, criterion)[0])
            grad_norm, second_loss = sam.second_pass(perturbed_loss)
            if not torch.isfinite(grad_norm + second_loss).item():
                raise FloatingPointError(f"Nonfinite SAM gradient norm or second-pass loss at benchmark step {i+1}")
        old_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        if not scaler.is_enabled() or scaler.get_scale() >= old_scale:
            scheduler.step()
            if ema is not None:
                ema.update(model)
        loss_sum += loss.detach().float() * y.size(0)
        correct += (logits.argmax(1) == y).sum() if batch_correct is None else batch_correct
        if i == 0:
            synchronize(device)
            first_step_s = time.perf_counter() - start
    synchronize(device)
    timed_s = time.perf_counter() - start
    validation_s = None
    if len(val_idx):
        val_x, val_y = images[val_idx], labels[val_idx]
        eval_kw = dict(mean=mean, std=std, device=device, batch_size=args.eval_batch_size, channels_last=args.channels_last)
        nets = [model] + ([ema.module] if ema is not None else [])
        for net in nets:  # untimed pass: cuDNN autotuning for the evaluation shapes
            evaluate(net, val_x, val_y, **eval_kw)
        synchronize(device)
        t0 = time.perf_counter()
        for net in nets:
            evaluate(net, val_x, val_y, **eval_kw)
        synchronize(device)
        validation_s = time.perf_counter() - t0
    per_step = timed_s / steps
    train_epoch_s = per_step * batches.steps_per_epoch
    result = {"model": args.model, "compile": args.compile, "device": device_name(device), "torch": torch.__version__,
              "amp": args.amp, "channels_last": bool(args.channels_last), "batch_size": args.batch_size,
              "steps": steps, "warmup_steps": warmup, "first_step_s": first_step_s, "warmup_s": warmup_s,
              "timed_s": timed_s, "steps_per_s": steps / timed_s, "ms_per_step": 1000 * per_step,
              "steps_per_epoch": batches.steps_per_epoch, "train_epoch_s": train_epoch_s,
              "validation_s": validation_s, "epoch_s": train_epoch_s + (validation_s or 0.0),
              "peak_vram_mib": torch.cuda.max_memory_allocated(device) / 2**20 if device.type == "cuda" else None,
              "mean_train_loss": loss_sum.item() / (y.size(0) * (warmup + steps)),
              "sam": sam is not None, "mixstyle": styler is not None}
    print(f"[benchmark] {args.model} | COMPILE {args.compile!r} | {result['device']} | torch {torch.__version__} | "
          f"amp {args.amp} | channels_last {_on(args.channels_last)} | batch {args.batch_size}"
          + (" | SAM on (two passes per step)" if sam is not None else "")
          + (" | MixStyle on" if styler is not None else ""), flush=True)
    print(f"[benchmark] warm-up: {warmup} steps in {warmup_s:.1f}s (first step {first_step_s:.1f}s, includes compilation)")
    print(f"[benchmark] {steps} timed steps: {result['steps_per_s']:.2f} steps/s = {result['ms_per_step']:.1f} ms/step "
          f"-> training {train_epoch_s:.1f}s/epoch ({batches.steps_per_epoch} steps)"
          + (f" + validation {validation_s:.1f}s (model + EMA, uncompiled) = about {result['epoch_s']:.1f}s/epoch"
             if validation_s is not None else ""))
    if target_timer is not None:
        target_s = target_timer.pop_seconds()
        result.update(teacher_compile=target_compile, teacher_warmup_s=warmup_target_s,
                      teacher_ms_per_step=1000 * target_s / steps, teacher_share=target_s / timed_s)
        print(f"[benchmark] teacher forward (KD_COMPILE_TEACHERS {target_compile!r}): "
              f"{result['teacher_ms_per_step']:.1f} ms/step = {result['teacher_share']:.0%} of the timed steps "
              f"(warm-up {warmup_target_s:.1f}s)")
    print("[benchmark] " + (f"peak CUDA allocation {result['peak_vram_mib']:.0f} MiB; " if device.type == "cuda" else "")
          + "nothing was saved; the benchmark weights are discarded")
    return result


## 5b. Knowledge distillation (T05)

**Target.** For a training batch `x` (random crop + flip + Cutout, drawn once with the same generators as
`E_pyramid.ipynb`), every teacher computes logits on the identical tensor `x` and, with `KD_FLIP_AVERAGE=True`, on its
horizontal mirror in the same forward pass. The target is the mean of the teachers' softmax(logits / T) over teachers
and views, formed in log space (logsumexp) for numerical stability. Teachers run frozen in eval mode (BatchNorm uses its
running statistics, which never change), under `torch.no_grad`, with bf16 autocast and channels-last on CUDA, like
the student. Nothing is precomputed. Teachers trained on the same 45k split share the student's normalization
statistics exactly; otherwise (only in smoke tests with `TRAIN_SUBSET`) the student's normalized tensor is mapped to the
teacher's normalization of the same pixels.

**Loss.** `L = α · T² · KL(p_teacher^T ‖ p_student^T) + (1 − α) · CE(student logits, label)` with
`KD_ALPHA = α = 0.9` and `KD_TEMPERATURE = T = 2` (Hinton et al., 2015). The KL term uses the student's log-softmax
at temperature T and is summed over classes and averaged over the batch (`reduction="batchmean"`); T² keeps its
gradient scale comparable to CE. CE has no label smoothing (R0's 0.1 is replaced by the soft target). With
`MIXUP=True` the mixed batch goes to teachers and student, the KD target is the teachers' output on the mixed image,
and the hard term is the usual λ-weighted CE of both labels with the shared, checkpointed Mixup RNG. α = 0 reduces
exactly to E's training with plain CE (checked by the tests).

**Where teachers are found.** A teacher without its own `root` is looked up in `KD_TEACHER_ROOT` (default
`RESULT_DIR`), then in each read-only `KD_EXTRA_TEACHER_ROOTS` folder, for example H.W.'s shared folder through the
MyDrive shortcut `HW_DL_Project1` (`/content/drive/MyDrive/HW_DL_Project1/result`); on Colab a missing shortcut is
awaited for `KD_EXTRA_ROOT_WAIT_S` seconds. The checkpoint comes from the first folder that holds it, the recorded
submission and stored accuracies from the first folder that has them; the folder used is recorded with each teacher
check. Folders are runtime settings, so a resume may use another mount path; the checkpoint SHA-256 must not change.

**Identity and resume.** The resolved teachers (run ID, family, checkpoint role, epoch, raw/EMA weights, checkpoint
SHA-256, parameter count, model fingerprint, architecture and normalization) together with α, T, flip averaging, label
smoothing and the Mixup settings are saved as `distillation` in the run configuration, every checkpoint and the
training report; their SHA-256 is the recipe fingerprint. Resume rebuilds the teachers from `KD_TEACHERS`, re-checks
them and refuses to continue unless the fingerprint matches. The student's own model fingerprint stays E's, so its
checkpoints remain loadable by E's evaluation code and Q08.

**Timing.** Each epoch line prints the teacher-forward time and its share of the epoch (CUDA events, so no extra
synchronization per step); the CSV log adds `kd_soft_loss`, `kd_hard_loss`, `kd_teacher_s` and `kd_teacher_share`, and
the report compares the mean epoch time with E ver-1/T18's 32.9 s on the Colab A100 (estimate for E ver-1/T18 + C ver-2
with flip averaging: ≈1.5–1.9× E, about 4–5 hours for 300 epochs).

In [ ]:
# Verbatim copy of A_resnet.ipynb section 4: its model cell, then its parameter-budget cell.
# Used only to rebuild teachers of this family. Do not edit: tests/test_distill.py fails when this text
# differs from the training notebook; after a model-cell change there, copy the new cells here unchanged.
TEACHER_MODEL_SOURCES = {}  # family -> (model cell, parameter-budget cell), filled by the six cells

TEACHER_MODEL_SOURCES['A'] = (
r'''MODEL = "A"
MODEL_DESCRIPTION = "ResNet BasicBlock [64,128,256] x [3,5,3]"
EXPECTED_PARAMS = 4_918_602


class ECAModule(nn.Module):
    """Spatial average -> shared 1D convolution across channels -> sigmoid channel gates."""
    def __init__(self, kernel_size=3):
        super().__init__()
        self.conv = nn.Conv1d(1, 1, kernel_size, padding=kernel_size // 2, bias=False)

    def forward(self, x):
        weights = torch.sigmoid(self.conv(x.mean(dim=(2, 3)).unsqueeze(1))).squeeze(1)
        return x * weights[:, :, None, None]


class BasicBlock(nn.Module):
    """conv-BN-ReLU-conv-BN, added to the shortcut, then ReLU (He et al., 2016)."""

    def __init__(self, in_ch, out_ch, stride=1, channel_attention="none", eca_kernel_size=3):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.eca = ECAModule(eca_kernel_size) if channel_attention == "eca" else None
        if stride == 1 and in_ch == out_ch:
            self.shortcut = nn.Identity()
        else:  # 1x1 projection shortcut where the shape changes
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, bias=False), nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.eca is not None:
            out = self.eca(out)
        return F.relu(out + self.shortcut(x))


class PreActBasicBlock(nn.Module):
    """Full pre-activation: raw identity; projections use BN-ReLU input; no post-add ReLU."""
    def __init__(self, in_ch, out_ch, stride=1, channel_attention="none", eca_kernel_size=3):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.eca = ECAModule(eca_kernel_size) if channel_attention == "eca" else None
        self.shortcut = (nn.Identity() if stride == 1 and in_ch == out_ch else
                         nn.Conv2d(in_ch, out_ch, 1, stride, bias=False))

    def forward(self, x):
        pre = F.relu(self.bn1(x))
        shortcut = x if isinstance(self.shortcut, nn.Identity) else self.shortcut(pre)
        out = self.conv1(pre)
        out = self.conv2(F.relu(self.bn2(out)))
        if self.eca is not None:
            out = self.eca(out)
        return shortcut + out


class ResNet(nn.Module):
    """3x3 stem (no max-pool), one stage per entry of `channels`, global average pool, linear head."""

    def __init__(self, channels=(64, 128, 256), blocks=(3, 5, 3), num_classes=10, channel_attention="none", eca_kernel_size=3, activation_order="post"):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels[0], 3, 1, 1, bias=False), nn.BatchNorm2d(channels[0]), nn.ReLU(inplace=True)
        )
        if activation_order == "pre":
            # Preserve the original post-activation modules/state keys when the switch is off.
            self.stem = nn.Sequential(nn.Conv2d(3, channels[0], 3, 1, 1, bias=False))
            self.final_norm = nn.Sequential(nn.BatchNorm2d(channels[-1]), nn.ReLU())
        block = PreActBasicBlock if activation_order == "pre" else BasicBlock
        stages, in_ch = [], channels[0]
        for i, (ch, n) in enumerate(zip(channels, blocks)):
            layers = []
            for j in range(n):
                stride = 2 if (i > 0 and j == 0) else 1
                layers.append(block(in_ch, ch, stride, channel_attention, eca_kernel_size))
                in_ch = ch
            stages.append(nn.Sequential(*layers))
        self.stages = nn.Sequential(*stages)
        self.fc = nn.Linear(in_ch, num_classes)

    def forward(self, x):
        x = self.stages(self.stem(x))
        if hasattr(self, "final_norm"):
            x = self.final_norm(x)
        return self.fc(torch.flatten(F.adaptive_avg_pool2d(x, 1), 1))


ARCH_DEFAULTS = {"implementation_version": 1, "channel_attention": "none", "eca_kernel_size": 3}


ARCH_OPTION_DEFAULTS = {"depth_preset": "baseline", "capacity_preset": "baseline", "activation_order": "post"}  # Omitted baseline key preserves old checkpoint identities.


def architecture_from_config():
    return {"channel_attention": CHANNEL_ATTENTION, "eca_kernel_size": ECA_KERNEL_SIZE,
            "depth_preset": DEPTH_PRESET, "capacity_preset": CAPACITY_PRESET, "activation_order": ACTIVATION_ORDER}


def architecture_channels(options):
    return (64, 160, 224) if options.get("capacity_preset") == "mid" else (64, 128, 256)


def architecture_blocks(options):
    if options.get("capacity_preset") == "mid":
        return (3, 4, 3)
    return (4, 5, 3) if options.get("depth_preset", "baseline") == "early" else (3, 5, 3)


def validate_architecture(options):
    if options.get("activation_order", "post") not in ("post", "pre"):
        raise ValueError("ACTIVATION_ORDER must be post or pre")
    capacity = options.get("capacity_preset", "baseline")
    if capacity not in ("baseline", "mid"):
        raise ValueError("CAPACITY_PRESET must be baseline or mid")
    if capacity == "mid" and (options.get("depth_preset", "baseline") != "baseline" or options["channel_attention"] != "none"):
        raise ValueError("T26 mid capacity requires DEPTH_PRESET=baseline and CHANNEL_ATTENTION=none")
    depth = options.get("depth_preset", "baseline")
    if depth not in ("baseline", "early"):
        raise ValueError("DEPTH_PRESET must be baseline or early")
    if options["channel_attention"] not in ("none", "eca"):
        raise ValueError("CHANNEL_ATTENTION must be 'none' or 'eca'")
    k = options["eca_kernel_size"]
    if type(k) is not int or not 1 <= k <= 63 or k % 2 != 1:
        raise ValueError("ECA_KERNEL_SIZE must be a positive odd integer <= 63; T48 uses 3")


def describe_architecture(options):
    if options.get("activation_order", "post") == "pre":
        return (f"PreAct ResNet {list(architecture_channels(options))} x {list(architecture_blocks(options))}; "
                "BN-ReLU-conv branches, raw identity / preactivated projection, final BN-ReLU" +
                (f"; ECA k={options['eca_kernel_size']}" if options['channel_attention'] == 'eca' else ''))
    if options.get("capacity_preset") == "mid":
        return "ResNet BasicBlock [64,160,224] x [3,4,3]"
    if options.get("depth_preset") == "early":
        return "ResNet BasicBlock [64,128,256] x [4,5,3]" + (f"; ECA k={options['eca_kernel_size']} after each bn2" if options["channel_attention"] == "eca" else "")
    return MODEL_DESCRIPTION + (f"; ECA k={options['eca_kernel_size']} after each bn2" if options["channel_attention"] == "eca" else "")


def architecture_feature(options):
    if options.get("activation_order", "post") == "pre":
        return "preact"
    if options.get("capacity_preset") == "mid":
        return "mid"
    if options.get("depth_preset") == "early":
        return "early-eca" if options["channel_attention"] == "eca" else "early"
    return "eca" if options["channel_attention"] == "eca" else None


def expected_param_count(options):
    if options.get("capacity_preset") == "mid":
        return 4_563_594 - (2 * sum(architecture_channels(options)[1:]) if options.get("activation_order", "post") == "pre" else 0)
    base = 4_992_586 if options.get("depth_preset") == "early" else EXPECTED_PARAMS
    eca_params = sum(architecture_blocks(options)) * options["eca_kernel_size"] if options["channel_attention"] == "eca" else 0
    return base + eca_params - (2 * sum(architecture_channels(options)[1:]) if options.get("activation_order", "post") == "pre" else 0)


def make_model(num_classes=10, architecture=None):
    options = resolve_architecture(architecture)
    return ResNet(architecture_channels(options), architecture_blocks(options), num_classes=num_classes,
                  channel_attention=options["channel_attention"], eca_kernel_size=options["eca_kernel_size"],
                  activation_order=options.get("activation_order", "post"))''',
r'''PARAM_BUDGET = 5_000_000


def init_weights(model):
    """Kaiming-normal (fan_out) convs and unit/zero BN, as in torchvision's ResNet."""
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def count_params(model):
    """Number of trainable parameters (conv/linear weights and biases, BN gamma/beta)."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


ARCHITECTURE_IMPLEMENTATION_VERSION = 1  # Bump when the meaning of a switch/forward pass changes.


def resolve_architecture(options=None):
    """None reads UI switches; an explicit dict (including {}) uses saved/baseline options."""
    options = architecture_from_config() if options is None else options
    if not isinstance(options, dict):
        raise ValueError("Architecture options must be a dictionary")
    unknown = set(options) - (set(ARCH_DEFAULTS) | set(ARCH_OPTION_DEFAULTS))
    if unknown:
        raise ValueError(f"Unknown architecture options for model {MODEL}: {sorted(unknown)}")
    resolved = {**ARCH_DEFAULTS, **ARCH_OPTION_DEFAULTS, **options}
    version = resolved["implementation_version"]
    if type(version) is not int or version != ARCHITECTURE_IMPLEMENTATION_VERSION:
        raise ValueError("Unsupported architecture implementation_version; use the matching notebook code")
    validate_architecture(resolved)
    # Canonical omission preserves fingerprints/topology records for all old options.
    return {k: v for k, v in resolved.items() if k not in ARCH_OPTION_DEFAULTS or v != ARCH_OPTION_DEFAULTS[k]}


def saved_architecture(args):
    """Missing legacy fields mean baseline; malformed saved options must never read UI globals."""
    options = args.get("architecture", ARCH_DEFAULTS)
    if not isinstance(options, dict):
        raise ValueError("Saved architecture options must be a dictionary")
    return resolve_architecture(options)


def architecture_details(options):
    """Resolved topology accompanies options in configs, checkpoints and evaluation reports."""
    details = resolve_architecture(options)
    if MODEL not in ("A", "B", "C"):  # E/F/G define architecture_topology next to their model (section 4)
        return dict(details, model=MODEL, **architecture_topology(details), num_classes=NUM_CLASSES)
    channels = (WIDTH_PRESETS[details["width_preset"]] if MODEL == "B" else
                architecture_channels(details) if MODEL == "A" else (64, 128, 256))
    return dict(details, model=MODEL, stem_channels=16 if MODEL == "B" else 64,
                stage_channels=list(channels), blocks=list(architecture_blocks(details)),
                num_classes=NUM_CLASSES)


def build_model(num_classes=NUM_CLASSES, architecture=None):
    """Build this notebook's model, check the 5M budget, and return (model, n_trainable_params)."""
    options = resolve_architecture(architecture)
    model = init_weights(make_model(num_classes, architecture=options))
    # Plain attributes do not alter legacy state_dict keys or baseline repr/fingerprints.
    model._project_architecture = options
    model._project_description = describe_architecture(options)
    n_params = count_params(model)
    assert n_params <= PARAM_BUDGET, f"model {MODEL} has {n_params:,} trainable params > {PARAM_BUDGET:,}"
    return model, n_params


_model, N_PARAMS = build_model()
print(f"Model {MODEL}: {_model._project_description}")
print(f"Architecture options: {_model._project_architecture}")
print(f"Trainable parameters: {N_PARAMS:,} (budget {PARAM_BUDGET:,})")
assert N_PARAMS <= PARAM_BUDGET
assert N_PARAMS == expected_param_count(_model._project_architecture), "Unexpected architecture parameter count"
if SHOW_SUMMARY:
    try:
        from torchinfo import summary
    except ImportError:
        print("torchinfo is not installed; layer summary skipped (the parameter count above is complete)")
    else:
        print(summary(_model, input_size=(1, 3, 32, 32), depth=3, col_names=("output_size", "num_params"),
                      device="cpu", verbose=0))
del _model''',
)


In [ ]:
# Verbatim copy of B_wrn.ipynb section 4: its model cell, then its parameter-budget cell.
# Used only to rebuild teachers of this family. Do not edit: tests/test_distill.py fails when this text
# differs from the training notebook; after a model-cell change there, copy the new cells here unchanged.

TEACHER_MODEL_SOURCES['B'] = (
r'''MODEL = "B"
MODEL_DESCRIPTION = "WRN-22-4 (pre-activation, widths 16/64/128/256)"
EXPECTED_PARAMS = 4_298_970


class PreActBlock(nn.Module):
    """Pre-activation block BN-ReLU-conv-BN-ReLU-conv (He et al., 2016b), as used in WRN."""

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        # WRN uses a plain 1x1 conv (no BN) on the pre-activated input when the shape changes.
        self.shortcut = None if (stride == 1 and in_ch == out_ch) else nn.Conv2d(in_ch, out_ch, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = x if self.shortcut is None else self.shortcut(out)
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class WideResNet(nn.Module):
    """WRN-depth-k: widths 16 / 16k / 32k / 64k, (depth-4)/6 blocks per group."""

    def __init__(self, depth=22, widen=4, num_classes=10, stage_channels=None):
        super().__init__()
        assert (depth - 4) % 6 == 0, "WRN depth must be 6n+4"
        n = (depth - 4) // 6
        widths = [16, *(stage_channels if stage_channels is not None else (16 * widen, 32 * widen, 64 * widen))]
        self.stem = nn.Conv2d(3, widths[0], 3, 1, 1, bias=False)
        groups, in_ch = [], widths[0]
        for g in range(3):
            layers = []
            for j in range(n):
                stride = 2 if (g > 0 and j == 0) else 1
                layers.append(PreActBlock(in_ch, widths[g + 1], stride))
                in_ch = widths[g + 1]
            groups.append(nn.Sequential(*layers))
        self.groups = nn.Sequential(*groups)
        self.bn = nn.BatchNorm2d(in_ch)
        self.fc = nn.Linear(in_ch, num_classes)

    def forward(self, x):
        x = F.relu(self.bn(self.groups(self.stem(x))))
        return self.fc(torch.flatten(F.adaptive_avg_pool2d(x, 1), 1))


WIDTH_PRESETS = {"baseline": (64, 128, 256), "wide68": (68, 136, 272), "deep56": (56, 112, 224), "balanced": (64, 144, 272), "shallow80": (80, 160, 320)}
WIDTH_DEPTHS = {"baseline": 22, "wide68": 22, "deep56": 28, "balanced": 22, "shallow80": 16}
ARCH_OPTION_DEFAULTS = {}
ARCH_DEFAULTS = {"implementation_version": 1, "width_preset": "baseline"}


def architecture_from_config():
    return {"width_preset": WIDTH_PRESET}


def architecture_blocks(options):
    return ((WIDTH_DEPTHS[options["width_preset"]] - 4) // 6,) * 3


def validate_architecture(options):
    if not isinstance(options["width_preset"], str) or options["width_preset"] not in WIDTH_PRESETS:
        raise ValueError("WIDTH_PRESET must be baseline, wide68, deep56, balanced or shallow80")


def describe_architecture(options):
    if options["width_preset"] == "shallow80":
        return "WRN-16-5 (pre-activation, widths 16/80/160/320, blocks 2/2/2)"
    if options["width_preset"] == "balanced":
        return "WRN-22 (pre-activation, widths 16/64/144/272, blocks 3/3/3)"
    if options["width_preset"] == "deep56":
        return "WRN-28 (pre-activation, widths 16/56/112/224, blocks 4/4/4)"
    return MODEL_DESCRIPTION if options["width_preset"] == "baseline" else "WRN-22 (pre-activation, widths 16/68/136/272, blocks 3/3/3)"


def architecture_feature(options):
    return {"baseline": None, "wide68": "wide", "deep56": "deep", "balanced": "balanced", "shallow80": "shallow"}[options["width_preset"]]


def expected_param_count(options):
    return {"baseline": EXPECTED_PARAMS, "wide68": 4_851_866, "deep56": 4_480_474, "balanced": 4_949_754, "shallow80": 4_289_754}[options["width_preset"]]


def make_model(num_classes=10, architecture=None):
    options = resolve_architecture(architecture)
    return WideResNet(depth=WIDTH_DEPTHS[options["width_preset"]], num_classes=num_classes,
                      stage_channels=WIDTH_PRESETS[options["width_preset"]])''',
r'''PARAM_BUDGET = 5_000_000


def init_weights(model):
    """Kaiming-normal (fan_out) convs and unit/zero BN, as in torchvision's ResNet."""
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def count_params(model):
    """Number of trainable parameters (conv/linear weights and biases, BN gamma/beta)."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


ARCHITECTURE_IMPLEMENTATION_VERSION = 1  # Bump when the meaning of a switch/forward pass changes.


def resolve_architecture(options=None):
    """None reads UI switches; an explicit dict (including {}) uses saved/baseline options."""
    options = architecture_from_config() if options is None else options
    if not isinstance(options, dict):
        raise ValueError("Architecture options must be a dictionary")
    unknown = set(options) - (set(ARCH_DEFAULTS) | set(ARCH_OPTION_DEFAULTS))
    if unknown:
        raise ValueError(f"Unknown architecture options for model {MODEL}: {sorted(unknown)}")
    resolved = {**ARCH_DEFAULTS, **ARCH_OPTION_DEFAULTS, **options}
    version = resolved["implementation_version"]
    if type(version) is not int or version != ARCHITECTURE_IMPLEMENTATION_VERSION:
        raise ValueError("Unsupported architecture implementation_version; use the matching notebook code")
    validate_architecture(resolved)
    # Canonical omission preserves fingerprints/topology records for all old options.
    return {k: v for k, v in resolved.items() if k not in ARCH_OPTION_DEFAULTS or v != ARCH_OPTION_DEFAULTS[k]}


def saved_architecture(args):
    """Missing legacy fields mean baseline; malformed saved options must never read UI globals."""
    options = args.get("architecture", ARCH_DEFAULTS)
    if not isinstance(options, dict):
        raise ValueError("Saved architecture options must be a dictionary")
    return resolve_architecture(options)


def architecture_details(options):
    """Resolved topology accompanies options in configs, checkpoints and evaluation reports."""
    details = resolve_architecture(options)
    if MODEL not in ("A", "B", "C"):  # E/F/G define architecture_topology next to their model (section 4)
        return dict(details, model=MODEL, **architecture_topology(details), num_classes=NUM_CLASSES)
    channels = (WIDTH_PRESETS[details["width_preset"]] if MODEL == "B" else
                architecture_channels(details) if MODEL == "A" else (64, 128, 256))
    return dict(details, model=MODEL, stem_channels=16 if MODEL == "B" else 64,
                stage_channels=list(channels), blocks=list(architecture_blocks(details)),
                num_classes=NUM_CLASSES)


def build_model(num_classes=NUM_CLASSES, architecture=None):
    """Build this notebook's model, check the 5M budget, and return (model, n_trainable_params)."""
    options = resolve_architecture(architecture)
    model = init_weights(make_model(num_classes, architecture=options))
    # Plain attributes do not alter legacy state_dict keys or baseline repr/fingerprints.
    model._project_architecture = options
    model._project_description = describe_architecture(options)
    n_params = count_params(model)
    assert n_params <= PARAM_BUDGET, f"model {MODEL} has {n_params:,} trainable params > {PARAM_BUDGET:,}"
    return model, n_params


_model, N_PARAMS = build_model()
print(f"Model {MODEL}: {_model._project_description}")
print(f"Architecture options: {_model._project_architecture}")
print(f"Trainable parameters: {N_PARAMS:,} (budget {PARAM_BUDGET:,})")
assert N_PARAMS <= PARAM_BUDGET
assert N_PARAMS == expected_param_count(_model._project_architecture), "Unexpected architecture parameter count"
if SHOW_SUMMARY:
    try:
        from torchinfo import summary
    except ImportError:
        print("torchinfo is not installed; layer summary skipped (the parameter count above is complete)")
    else:
        print(summary(_model, input_size=(1, 3, 32, 32), depth=3, col_names=("output_size", "num_params"),
                      device="cpu", verbose=0))
del _model''',
)

In [ ]:
# Verbatim copy of C_seresnet.ipynb section 4: its model cell, then its parameter-budget cell.
# Used only to rebuild teachers of this family. Do not edit: tests/test_distill.py fails when this text
# differs from the training notebook; after a model-cell change there, copy the new cells here unchanged.

TEACHER_MODEL_SOURCES['C'] = (
r'''MODEL = "C"
MODEL_DESCRIPTION = "SE-ResNet BasicBlock [64,128,256] x [3,5,3], SE r=16"
EXPECTED_PARAMS = 4_956_654


class SEModule(nn.Module):
    """Squeeze-and-Excitation channel gating (Hu et al., 2018)."""

    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)

    def forward(self, x):
        w = x.mean(dim=(2, 3))
        w = torch.sigmoid(self.fc2(F.relu(self.fc1(w))))
        return x * w[:, :, None, None]


class InstanceBatchNorm2d(nn.Module):
    """IBN-a: fixed half-channel affine IN + BN; IN uses per-image stats in train and eval.

    Stage placement is controlled by SEResNet, not by train/eval mode.
    Reference: https://github.com/XingangPan/IBN-Net/blob/master/ibnnet/modules.py
    """

    def __init__(self, channels):
        super().__init__()
        if type(channels) is not int or channels < 2 or channels % 2:
            raise ValueError("Half-channel IBN requires a positive even channel count >= 2")
        self.half = channels // 2
        self.instance = nn.InstanceNorm2d(self.half, affine=True, track_running_stats=False)
        self.batch = nn.BatchNorm2d(self.half)

    def forward(self, x):
        left, right = x.split(self.half, dim=1)
        return torch.cat((self.instance(left.contiguous()), self.batch(right.contiguous())), dim=1)


class SEBasicBlock(nn.Module):
    """conv-(BN or IBN)-ReLU-conv-BN-SE, shortcut addition, then ReLU."""

    def __init__(self, in_ch, out_ch, stride=1, reduction=16, ibn=False):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn1 = InstanceBatchNorm2d(out_ch) if ibn else nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.se = SEModule(out_ch, reduction)
        if stride == 1 and in_ch == out_ch:
            self.shortcut = nn.Identity()
        else:  # 1x1 projection shortcut where the shape changes
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, bias=False), nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.se(self.bn2(self.conv2(out)))
        return F.relu(out + self.shortcut(x))


class SEResNet(nn.Module):
    """3x3 stem (no max-pool), one stage of SE blocks per entry of `channels`, global average pool, linear head."""

    def __init__(self, channels=(64, 128, 256), blocks=(3, 5, 3), reduction=16, num_classes=10, ibn_stage1=False):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels[0], 3, 1, 1, bias=False), nn.BatchNorm2d(channels[0]), nn.ReLU(inplace=True)
        )
        stages, in_ch = [], channels[0]
        for i, (ch, n) in enumerate(zip(channels, blocks)):
            layers = []
            for j in range(n):
                stride = 2 if (i > 0 and j == 0) else 1
                layers.append(SEBasicBlock(in_ch, ch, stride, reduction, ibn=ibn_stage1 and i == 0))
                in_ch = ch
            stages.append(nn.Sequential(*layers))
        self.stages = nn.Sequential(*stages)
        self.fc = nn.Linear(in_ch, num_classes)

    def forward(self, x):
        x = self.stages(self.stem(x))
        return self.fc(torch.flatten(F.adaptive_avg_pool2d(x, 1), 1))


ARCH_DEFAULTS = {"implementation_version": 1, "se_reduction": 16}


ARCH_OPTION_DEFAULTS = {"ibn_stage1": False}


def architecture_blocks(options):
    return (3, 5, 3)


def architecture_from_config():
    return {"se_reduction": SE_REDUCTION, "ibn_stage1": IBN_STAGE1}


def validate_architecture(options):
    if type(options["ibn_stage1"]) is not bool:
        raise ValueError("IBN_STAGE1 must be True or False; T57 uses stage 1 only, fixed IN ratio 0.5")
    if type(options["se_reduction"]) is not int or options["se_reduction"] not in (16, 8):
        raise ValueError("SE_REDUCTION must be 16 (baseline) or 8 (T06)")


def describe_architecture(options):
    description = MODEL_DESCRIPTION.replace("r=16", f"r={options['se_reduction']}")
    if options.get("ibn_stage1", False):
        description += "; stage1 conv1 IBN-a (half affine IN/half BN, IN per-image stats)"
    return description


def architecture_feature(options):
    if options.get("ibn_stage1", False):
        return "ibn" if options["se_reduction"] == 16 else "se8ibn"
    return "se8" if options["se_reduction"] == 8 else None


def expected_param_count(options):
    return EXPECTED_PARAMS if options["se_reduction"] == 16 else 4_993_106


def make_model(num_classes=10, architecture=None):
    options = resolve_architecture(architecture)
    return SEResNet((64, 128, 256), (3, 5, 3), reduction=options["se_reduction"],
                    num_classes=num_classes, ibn_stage1=options.get("ibn_stage1", False))''',
r'''PARAM_BUDGET = 5_000_000


def init_weights(model):
    """Kaiming-normal (fan_out) convs and unit/zero affine BN/IN; original BN initialization unchanged."""
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        elif isinstance(m, (nn.BatchNorm2d, nn.InstanceNorm2d)) and m.affine:
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def count_params(model):
    """Number of trainable parameters (conv/linear weights and biases, normalization gamma/beta)."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


ARCHITECTURE_IMPLEMENTATION_VERSION = 1  # Bump when the meaning of a switch/forward pass changes.


def resolve_architecture(options=None):
    """None reads UI switches; an explicit dict (including {}) uses saved/baseline options."""
    options = architecture_from_config() if options is None else options
    if not isinstance(options, dict):
        raise ValueError("Architecture options must be a dictionary")
    unknown = set(options) - (set(ARCH_DEFAULTS) | set(ARCH_OPTION_DEFAULTS))
    if unknown:
        raise ValueError(f"Unknown architecture options for model {MODEL}: {sorted(unknown)}")
    resolved = {**ARCH_DEFAULTS, **ARCH_OPTION_DEFAULTS, **options}
    version = resolved["implementation_version"]
    if type(version) is not int or version != ARCHITECTURE_IMPLEMENTATION_VERSION:
        raise ValueError("Unsupported architecture implementation_version; use the matching notebook code")
    validate_architecture(resolved)
    # Canonical omission preserves fingerprints/topology records for all old options.
    return {k: v for k, v in resolved.items() if k not in ARCH_OPTION_DEFAULTS or v != ARCH_OPTION_DEFAULTS[k]}


def saved_architecture(args):
    """Missing legacy fields mean baseline; malformed saved options must never read UI globals."""
    options = args.get("architecture", ARCH_DEFAULTS)
    if not isinstance(options, dict):
        raise ValueError("Saved architecture options must be a dictionary")
    return resolve_architecture(options)


def architecture_details(options):
    """Resolved topology accompanies options in configs, checkpoints and evaluation reports."""
    details = resolve_architecture(options)
    if MODEL not in ("A", "B", "C"):  # E/F/G define architecture_topology next to their model (section 4)
        return dict(details, model=MODEL, **architecture_topology(details), num_classes=NUM_CLASSES)
    channels = (WIDTH_PRESETS[details["width_preset"]] if MODEL == "B" else
                architecture_channels(details) if MODEL == "A" else (64, 128, 256))
    if MODEL == "C" and details.get("ibn_stage1", False):
        details["ibn"] = {"stages": [1], "placement": "after conv1, before ReLU in each block",
                          "in_ratio": 0.5, "in_affine": True, "in_track_running_stats": False,
                          "implementation_version": 1}
    return dict(details, model=MODEL, stem_channels=16 if MODEL == "B" else 64,
                stage_channels=list(channels), blocks=list(architecture_blocks(details)),
                num_classes=NUM_CLASSES)


def build_model(num_classes=NUM_CLASSES, architecture=None):
    """Build this notebook's model, check the 5M budget, and return (model, n_trainable_params)."""
    options = resolve_architecture(architecture)
    model = init_weights(make_model(num_classes, architecture=options))
    # Plain attributes do not alter legacy state_dict keys or baseline repr/fingerprints.
    model._project_architecture = options
    model._project_description = describe_architecture(options)
    n_params = count_params(model)
    assert n_params <= PARAM_BUDGET, f"model {MODEL} has {n_params:,} trainable params > {PARAM_BUDGET:,}"
    return model, n_params


_model, N_PARAMS = build_model()
print(f"Model {MODEL}: {_model._project_description}")
print(f"Architecture options: {_model._project_architecture}")
print(f"Trainable parameters: {N_PARAMS:,} (budget {PARAM_BUDGET:,})")
assert N_PARAMS <= PARAM_BUDGET
assert N_PARAMS == expected_param_count(_model._project_architecture), "Unexpected architecture parameter count"
if SHOW_SUMMARY:
    try:
        from torchinfo import summary
    except ImportError:
        print("torchinfo is not installed; layer summary skipped (the parameter count above is complete)")
    else:
        print(summary(_model, input_size=(1, 3, 32, 32), depth=3, col_names=("output_size", "num_params"),
                      device="cpu", verbose=0))
del _model''',
)

In [ ]:
# Verbatim copy of E_pyramid.ipynb section 4: its model cell, then its parameter-budget cell.
# Used only to rebuild teachers of this family. Do not edit: tests/test_distill.py fails when this text
# differs from the training notebook; after a model-cell change there, copy the new cells here unchanged.

TEACHER_MODEL_SOURCES['E'] = (
r'''MODEL = "E"
MODEL_DESCRIPTION = "PyramidNet-110 basic, alpha=98 (widths 16->114)"
EXPECTED_PARAMS = 4_963_991


class DropPath(nn.Module):
    """Stochastic depth (Huang et al., 2016) for one residual branch, per sample.

    In training, each image's branch is dropped with probability p and survivors are scaled by 1 / (1 - p);
    in evaluation it is the identity. The mask comes from the global torch RNG of the input's device.
    """

    def __init__(self, p=0.0):
        super().__init__()
        self.p = float(p)

    def forward(self, x):
        if not self.training or self.p == 0.0:
            return x
        keep = 1.0 - self.p
        mask = torch.rand(x.shape[0], 1, 1, 1, device=x.device) < keep
        return x * mask.to(x.dtype) / keep

    def extra_repr(self):
        return f"p={self.p:.4f}"


MIXSTYLE_AFTER_STAGES = (1, 2)  # training-only MixStyle (MIXSTYLE, section 1) mixes the outputs of stages 1 and 2


def mix_feature_styles(x, apply, lam, perm, eps):
    """MixStyle (Zhou et al., ICLR 2021), "random" variant, on one training batch; parameter-free.

    Each image's per-channel mean mu and standard deviation sigma over the spatial positions (unbiased variance + eps,
    computed in float32 and detached, as in the reference code) are replaced by lam * stat + (1 - lam) * stat of its
    partner image perm[i]; the result is cast back to x's dtype. `apply` (0-dim bool tensor) selects the mixed batch
    or exactly x; lam has shape (B, 1, 1, 1). The draws come from the trainer's MixStyleSampler (section 5).
    """
    x32 = x.float()
    mu = x32.mean(dim=(2, 3), keepdim=True).detach()
    sigma = (x32.var(dim=(2, 3), keepdim=True) + eps).sqrt().detach()
    mixed_mu = lam * mu + (1 - lam) * mu[perm]
    mixed_sigma = lam * sigma + (1 - lam) * sigma[perm]
    mixed = ((x32 - mu) / sigma * mixed_sigma + mixed_mu).to(x.dtype)
    return torch.where(apply, mixed, x)


class PyramidBlock(nn.Module):
    """BN-conv-BN-ReLU-conv-BN (Han et al., 2017), added to a parameter-free shortcut; no ReLU after the addition."""

    def __init__(self, in_ch, out_ch, stride=1, drop_path=0.0):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_ch)
        self.drop_path = DropPath(drop_path)
        # Shortcut: identity, 2x2 average pool where the block downsamples, zero padding for the extra channels.
        self.pool = nn.AvgPool2d(2, 2, ceil_mode=True) if stride != 1 else nn.Identity()
        self.extra_ch = out_ch - in_ch

    def forward(self, x):
        out = self.conv1(self.bn1(x))
        out = self.conv2(F.relu(self.bn2(out)))
        out = self.drop_path(self.bn3(out))
        shortcut = self.pool(x)
        if self.extra_ch:
            shortcut = F.pad(shortcut, (0, 0, 0, 0, 0, self.extra_ch))
        return out + shortcut


class PyramidNet(nn.Module):
    """3x3 stem (16 channels, BN), 3 stages of (depth-2)/6 pyramidal blocks, BN-ReLU, global average pool, linear.

    The width grows by alpha / N after each of the N blocks. As in the reference implementation, the width is
    accumulated as a float and each block uses int(round(width)) channels. Block l of N drops its residual branch
    with probability drop_path * l / N (linear decay rule of Huang et al., 2016). A training forward may receive
    MixStyle draws for the points after stages MIXSTYLE_AFTER_STAGES; this adds no module, parameter or buffer.
    """

    mixstyle_layers = len(MIXSTYLE_AFTER_STAGES)  # number of MixStyle draws a training forward accepts

    def __init__(self, depth=110, alpha=98, drop_path=0.0, num_classes=10):
        super().__init__()
        assert (depth - 2) % 6 == 0, "basic-block PyramidNet depth must be 6n+2"
        n = (depth - 2) // 6
        total = 3 * n
        add_rate = alpha / total
        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, 1, 1, bias=False), nn.BatchNorm2d(16))
        stages, in_ch, width, index = [], 16, 16.0, 0
        for i in range(3):
            layers = []
            for j in range(n):
                index += 1
                width += add_rate
                out_ch = int(round(width))
                stride = 2 if (i > 0 and j == 0) else 1
                layers.append(PyramidBlock(in_ch, out_ch, stride, drop_path * index / total))
                in_ch = out_ch
            stages.append(nn.Sequential(*layers))
        self.stages = nn.Sequential(*stages)
        self.bn = nn.BatchNorm2d(in_ch)
        self.fc = nn.Linear(in_ch, num_classes)

    def forward(self, x, mixstyle=None):
        """`mixstyle` = (apply, lam, perm, eps) from MixStyleSampler.draw, or None. Without draws, and always in
        evaluation mode, this is exactly the plain PyramidNet forward."""
        x = self.stem(x)
        if mixstyle is None or not self.training:
            x = self.stages(x)
        else:
            apply, lam, perm, eps = mixstyle
            layer = 0
            for index, stage in enumerate(self.stages, 1):
                x = stage(x)
                if index in MIXSTYLE_AFTER_STAGES:
                    x = mix_feature_styles(x, apply[layer], lam[layer], perm[layer], eps)
                    layer += 1
        x = F.relu(self.bn(x))
        return self.fc(torch.flatten(F.adaptive_avg_pool2d(x, 1), 1))


ARCH_DEFAULTS = {"implementation_version": 1, "drop_path": 0.0}


def architecture_from_config():
    return {"drop_path": DROP_PATH}


def validate_architecture(options):
    rate = options["drop_path"]
    if isinstance(rate, bool) or not isinstance(rate, (int, float)) or not 0 <= rate < 1:
        raise ValueError(f"DROP_PATH = {rate!r}; choose a rate in [0, 1) (0 = no stochastic depth)")


def describe_architecture(options):
    # The rate is saved with the options and enters the architecture fingerprint: resuming with another rate is
    # rejected, and evaluation rebuilds the saved rate whatever the current DROP_PATH.
    return f"{MODEL_DESCRIPTION}, stochastic depth DROP_PATH={float(options['drop_path'])!r}"


def architecture_feature(options):
    return "stochdepth" if options["drop_path"] > 0 else None


def expected_param_count(options):
    return EXPECTED_PARAMS  # stochastic depth adds no parameters


def architecture_topology(options):
    """Stem width, output channels of each stage (the width grows by alpha/54 in every block) and blocks per stage."""
    return {"stem_channels": 16, "stage_channels": [49, 81, 114], "blocks": [18, 18, 18]}


def make_model(num_classes=10, architecture=None):
    options = resolve_architecture(architecture)
    return PyramidNet(depth=110, alpha=98, drop_path=options["drop_path"], num_classes=num_classes)''',
r'''PARAM_BUDGET = 5_000_000


def init_weights(model):
    """Kaiming-normal (fan_out) convs and unit/zero BN, as in torchvision's ResNet."""
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def count_params(model):
    """Number of trainable parameters (conv/linear weights and biases, BN gamma/beta)."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


ARCHITECTURE_IMPLEMENTATION_VERSION = 1  # Bump when the meaning of a switch/forward pass changes.


def resolve_architecture(options=None):
    """None reads UI switches; an explicit dict (including {}) uses saved/baseline options."""
    options = architecture_from_config() if options is None else options
    if not isinstance(options, dict):
        raise ValueError("Architecture options must be a dictionary")
    unknown = set(options) - set(ARCH_DEFAULTS)
    if unknown:
        raise ValueError(f"Unknown architecture options for model {MODEL}: {sorted(unknown)}")
    resolved = dict(ARCH_DEFAULTS, **options)
    version = resolved["implementation_version"]
    if type(version) is not int or version != ARCHITECTURE_IMPLEMENTATION_VERSION:
        raise ValueError("Unsupported architecture implementation_version; use the matching notebook code")
    validate_architecture(resolved)
    return resolved


def saved_architecture(args):
    """Missing legacy fields mean baseline; malformed saved options must never read UI globals."""
    options = args.get("architecture", ARCH_DEFAULTS)
    if not isinstance(options, dict):
        raise ValueError("Saved architecture options must be a dictionary")
    return resolve_architecture(options)


def architecture_details(options):
    """Resolved topology accompanies options in configs, checkpoints and evaluation reports."""
    details = resolve_architecture(options)
    if MODEL not in ("A", "B", "C"):  # E/F/G define architecture_topology next to their model (section 4)
        return dict(details, model=MODEL, **architecture_topology(details), num_classes=NUM_CLASSES)
    channels = WIDTH_PRESETS[details["width_preset"]] if MODEL == "B" else (64, 128, 256)
    return dict(details, model=MODEL, stem_channels=16 if MODEL == "B" else 64,
                stage_channels=list(channels), blocks=[3, 3, 3] if MODEL == "B" else [3, 5, 3],
                num_classes=NUM_CLASSES)


def build_model(num_classes=NUM_CLASSES, architecture=None):
    """Build this notebook's model, check the 5M budget, and return (model, n_trainable_params)."""
    options = resolve_architecture(architecture)
    model = init_weights(make_model(num_classes, architecture=options))
    # Plain attributes do not alter legacy state_dict keys or baseline repr/fingerprints.
    model._project_architecture = options
    model._project_description = describe_architecture(options)
    n_params = count_params(model)
    assert n_params <= PARAM_BUDGET, f"model {MODEL} has {n_params:,} trainable params > {PARAM_BUDGET:,}"
    return model, n_params


_model, N_PARAMS = build_model()
print(f"Model {MODEL}: {_model._project_description}")
print(f"Architecture options: {_model._project_architecture}")
print(f"Trainable parameters: {N_PARAMS:,} (budget {PARAM_BUDGET:,})")
assert N_PARAMS <= PARAM_BUDGET
assert N_PARAMS == expected_param_count(_model._project_architecture), "Unexpected architecture parameter count"
if SHOW_SUMMARY:
    try:
        from torchinfo import summary
    except ImportError:
        print("torchinfo is not installed; layer summary skipped (the parameter count above is complete)")
    else:
        print(summary(_model, input_size=(1, 3, 32, 32), depth=3, col_names=("output_size", "num_params"),
                      device="cpu", verbose=0))
del _model''',
)

In [ ]:
# Verbatim copy of F_shake.ipynb section 4: its model cell, then its parameter-budget cell.
# Used only to rebuild teachers of this family. Do not edit: tests/test_distill.py fails when this text
# differs from the training notebook; after a model-cell change there, copy the new cells here unchanged.

TEACHER_MODEL_SOURCES['F'] = (
r'''MODEL = "F"
MODEL_DESCRIPTION = "Shake-Shake-26 2x40d (widths 40/80/160), Shake-Shake-Image: forward and backward per image"
EXPECTED_PARAMS = 4_561_354


class ShakeShake(torch.autograd.Function):
    """Forward alpha * a + (1 - alpha) * b; backward passes beta and 1 - beta of the gradient (Gastaldi, 2017)."""

    @staticmethod
    def forward(ctx, a, b, alpha, beta):
        ctx.save_for_backward(beta)
        return torch.lerp(b, a, alpha.to(a.dtype))  # = alpha * a + (1 - alpha) * b

    @staticmethod
    def backward(ctx, grad):
        (beta,) = ctx.saved_tensors
        return grad * beta.to(grad.dtype), grad * (1 - beta).to(grad.dtype), None, None


def shake_shake(a, b, training):
    """Combine two residual branches: independent per-image U(0,1) coefficients for the forward (alpha) and
    backward (beta) pass in training, drawn from the global torch RNG of the input's device; 0.5 in evaluation."""
    if not training:
        return 0.5 * (a + b)
    alpha, beta = torch.rand(2, a.shape[0], 1, 1, 1, device=a.device)
    return ShakeShake.apply(a, b, alpha, beta)


def shake_branch(in_ch, out_ch, stride):
    """ReLU-conv3x3-BN-ReLU-conv3x3-BN residual branch."""
    return nn.Sequential(
        nn.ReLU(), nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False), nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True), nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False), nn.BatchNorm2d(out_ch)
    )


class ShakeShortcut(nn.Module):
    """Shortcut where the shape changes: ReLU, then two 1x1 convs (out/2 channels each) after a stride-s 1x1
    average pool, the second on the input shifted by one pixel so that it sees the other pixels; concat, BN."""

    def __init__(self, in_ch, out_ch, stride):
        super().__init__()
        self.stride = stride
        self.conv1 = nn.Conv2d(in_ch, out_ch // 2, 1, bias=False)
        self.conv2 = nn.Conv2d(in_ch, out_ch // 2, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        x = F.relu(x)
        y1 = self.conv1(F.avg_pool2d(x, 1, self.stride))
        y2 = self.conv2(F.avg_pool2d(F.pad(x[:, :, 1:, 1:], (0, 1, 0, 1)), 1, self.stride))
        return self.bn(torch.cat([y1, y2], 1))

    def extra_repr(self):
        return f"stride={self.stride}"


class ShakeBlock(nn.Module):
    """Two residual branches combined by shake_shake and added to the shortcut; no ReLU after the addition."""

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.branch1 = shake_branch(in_ch, out_ch, stride)
        self.branch2 = shake_branch(in_ch, out_ch, stride)
        if stride == 1 and in_ch == out_ch:
            self.shortcut = nn.Identity()
        else:
            self.shortcut = ShakeShortcut(in_ch, out_ch, stride)

    def forward(self, x):
        return self.shortcut(x) + shake_shake(self.branch1(x), self.branch2(x), self.training)


class ShakeShakeNet(nn.Module):
    """3x3 stem (16 channels, BN), 3 stages of (depth-2)/6 Shake-Shake blocks with widths k/2k/4k (k even),
    ReLU, global average pool, linear."""

    def __init__(self, depth=26, width=40, num_classes=10):
        super().__init__()
        assert (depth - 2) % 6 == 0, "Shake-Shake depth must be 6n+2"
        if isinstance(width, bool) or not isinstance(width, int) or width < 2 or width % 2:
            raise ValueError(f"Shake-Shake width = {width!r}; choose an even positive integer: the first stage's "
                             "two-path shortcut concatenates two halves of width/2 channels")
        n = (depth - 2) // 6
        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, 1, 1, bias=False), nn.BatchNorm2d(16))
        stages, in_ch = [], 16
        for i, ch in enumerate((width, 2 * width, 4 * width)):
            layers = []
            for j in range(n):
                stride = 2 if (i > 0 and j == 0) else 1
                layers.append(ShakeBlock(in_ch, ch, stride))
                in_ch = ch
            stages.append(nn.Sequential(*layers))
        self.stages = nn.Sequential(*stages)
        self.fc = nn.Linear(in_ch, num_classes)

    def forward(self, x):
        x = F.relu(self.stages(self.stem(x)))
        return self.fc(torch.flatten(F.adaptive_avg_pool2d(x, 1), 1))


ARCH_DEFAULTS = {"implementation_version": 1}  # scheme F has no architecture switches


def architecture_from_config():
    return {}


def validate_architecture(options):
    pass  # resolve_architecture already rejects unknown options and other implementation versions


def describe_architecture(options):
    return MODEL_DESCRIPTION


def architecture_feature(options):
    return None


def expected_param_count(options):
    return EXPECTED_PARAMS


def architecture_topology(options):
    """Stem width, output channels and number of blocks of each stage (saved with the architecture options)."""
    return {"stem_channels": 16, "stage_channels": [40, 80, 160], "blocks": [4, 4, 4]}


def make_model(num_classes=10, architecture=None):
    resolve_architecture(architecture)
    return ShakeShakeNet(depth=26, width=40, num_classes=num_classes)''',
r'''PARAM_BUDGET = 5_000_000


def init_weights(model):
    """Kaiming-normal (fan_out) convs and unit/zero BN, as in torchvision's ResNet."""
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def count_params(model):
    """Number of trainable parameters (conv/linear weights and biases, BN gamma/beta)."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


ARCHITECTURE_IMPLEMENTATION_VERSION = 1  # Bump when the meaning of a switch/forward pass changes.


def resolve_architecture(options=None):
    """None reads UI switches; an explicit dict (including {}) uses saved/baseline options."""
    options = architecture_from_config() if options is None else options
    if not isinstance(options, dict):
        raise ValueError("Architecture options must be a dictionary")
    unknown = set(options) - set(ARCH_DEFAULTS)
    if unknown:
        raise ValueError(f"Unknown architecture options for model {MODEL}: {sorted(unknown)}")
    resolved = dict(ARCH_DEFAULTS, **options)
    version = resolved["implementation_version"]
    if type(version) is not int or version != ARCHITECTURE_IMPLEMENTATION_VERSION:
        raise ValueError("Unsupported architecture implementation_version; use the matching notebook code")
    validate_architecture(resolved)
    return resolved


def saved_architecture(args):
    """Missing legacy fields mean baseline; malformed saved options must never read UI globals."""
    options = args.get("architecture", ARCH_DEFAULTS)
    if not isinstance(options, dict):
        raise ValueError("Saved architecture options must be a dictionary")
    return resolve_architecture(options)


def architecture_details(options):
    """Resolved topology accompanies options in configs, checkpoints and evaluation reports."""
    details = resolve_architecture(options)
    if MODEL not in ("A", "B", "C"):  # E/F/G define architecture_topology next to their model (section 4)
        return dict(details, model=MODEL, **architecture_topology(details), num_classes=NUM_CLASSES)
    channels = WIDTH_PRESETS[details["width_preset"]] if MODEL == "B" else (64, 128, 256)
    return dict(details, model=MODEL, stem_channels=16 if MODEL == "B" else 64,
                stage_channels=list(channels), blocks=[3, 3, 3] if MODEL == "B" else [3, 5, 3],
                num_classes=NUM_CLASSES)


def build_model(num_classes=NUM_CLASSES, architecture=None):
    """Build this notebook's model, check the 5M budget, and return (model, n_trainable_params)."""
    options = resolve_architecture(architecture)
    model = init_weights(make_model(num_classes, architecture=options))
    # Plain attributes do not alter legacy state_dict keys or baseline repr/fingerprints.
    model._project_architecture = options
    model._project_description = describe_architecture(options)
    n_params = count_params(model)
    assert n_params <= PARAM_BUDGET, f"model {MODEL} has {n_params:,} trainable params > {PARAM_BUDGET:,}"
    return model, n_params


_model, N_PARAMS = build_model()
print(f"Model {MODEL}: {_model._project_description}")
print(f"Architecture options: {_model._project_architecture}")
print(f"Trainable parameters: {N_PARAMS:,} (budget {PARAM_BUDGET:,})")
assert N_PARAMS <= PARAM_BUDGET
assert N_PARAMS == expected_param_count(_model._project_architecture), "Unexpected architecture parameter count"
if SHOW_SUMMARY:
    try:
        from torchinfo import summary
    except ImportError:
        print("torchinfo is not installed; layer summary skipped (the parameter count above is complete)")
    else:
        print(summary(_model, input_size=(1, 3, 32, 32), depth=3, col_names=("output_size", "num_params"),
                      device="cpu", verbose=0))
del _model''',
)

In [ ]:
# Verbatim copy of G_resnext.ipynb section 4: its model cell, then its parameter-budget cell.
# Used only to rebuild teachers of this family. Do not edit: tests/test_distill.py fails when this text
# differs from the training notebook; after a model-cell change there, copy the new cells here unchanged.

TEACHER_MODEL_SOURCES['G'] = (
r'''MODEL = "G"
MODEL_DESCRIPTION = "ResNeXt-29 32x4d (cardinality 32, group widths 128/256/512, outputs 256/512/1024)"
EXPECTED_PARAMS = 4_775_754


class ResNeXtBlock(nn.Module):
    """1x1 conv-BN-ReLU, grouped 3x3 conv-BN-ReLU, 1x1 conv (2x channels)-BN, added to the shortcut, then ReLU."""

    expansion = 2

    def __init__(self, in_ch, cardinality=32, bottleneck_width=4, stride=1):
        super().__init__()
        group_width = cardinality * bottleneck_width
        out_ch = self.expansion * group_width
        self.conv1 = nn.Conv2d(in_ch, group_width, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(group_width)
        self.conv2 = nn.Conv2d(group_width, group_width, 3, stride, 1, groups=cardinality, bias=False)
        self.bn2 = nn.BatchNorm2d(group_width)
        self.conv3 = nn.Conv2d(group_width, out_ch, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_ch)
        if stride == 1 and in_ch == out_ch:
            self.shortcut = nn.Identity()
        else:  # 1x1 projection shortcut where the shape changes
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, bias=False), nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        return F.relu(out + self.shortcut(x))


class ResNeXt(nn.Module):
    """3x3 stem (64 channels, no max-pool), one stage of bottleneck blocks per entry of `blocks`; the bottleneck
    width doubles after each stage. Global average pool, linear head."""

    def __init__(self, blocks=(3, 3, 3), cardinality=32, bottleneck_width=4, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True)
        )
        stages, in_ch, width = [], 64, bottleneck_width
        for i, n in enumerate(blocks):
            layers = []
            for j in range(n):
                stride = 2 if (i > 0 and j == 0) else 1
                layers.append(ResNeXtBlock(in_ch, cardinality, width, stride))
                in_ch = ResNeXtBlock.expansion * cardinality * width
            stages.append(nn.Sequential(*layers))
            width *= 2
        self.stages = nn.Sequential(*stages)
        self.fc = nn.Linear(in_ch, num_classes)

    def forward(self, x):
        x = self.stages(self.stem(x))
        return self.fc(torch.flatten(F.adaptive_avg_pool2d(x, 1), 1))


ARCH_DEFAULTS = {"implementation_version": 1}  # scheme G has no architecture switches


def architecture_from_config():
    return {}


def validate_architecture(options):
    pass  # resolve_architecture already rejects unknown options and other implementation versions


def describe_architecture(options):
    return MODEL_DESCRIPTION


def architecture_feature(options):
    return None


def expected_param_count(options):
    return EXPECTED_PARAMS


def architecture_topology(options):
    """Stem width, output channels and number of blocks of each stage (saved with the architecture options)."""
    return {"stem_channels": 64, "stage_channels": [256, 512, 1024], "blocks": [3, 3, 3]}


def make_model(num_classes=10, architecture=None):
    resolve_architecture(architecture)
    return ResNeXt((3, 3, 3), cardinality=32, bottleneck_width=4, num_classes=num_classes)''',
r'''PARAM_BUDGET = 5_000_000


def init_weights(model):
    """Kaiming-normal (fan_out) convs and unit/zero BN, as in torchvision's ResNet."""
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def count_params(model):
    """Number of trainable parameters (conv/linear weights and biases, BN gamma/beta)."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


ARCHITECTURE_IMPLEMENTATION_VERSION = 1  # Bump when the meaning of a switch/forward pass changes.


def resolve_architecture(options=None):
    """None reads UI switches; an explicit dict (including {}) uses saved/baseline options."""
    options = architecture_from_config() if options is None else options
    if not isinstance(options, dict):
        raise ValueError("Architecture options must be a dictionary")
    unknown = set(options) - set(ARCH_DEFAULTS)
    if unknown:
        raise ValueError(f"Unknown architecture options for model {MODEL}: {sorted(unknown)}")
    resolved = dict(ARCH_DEFAULTS, **options)
    version = resolved["implementation_version"]
    if type(version) is not int or version != ARCHITECTURE_IMPLEMENTATION_VERSION:
        raise ValueError("Unsupported architecture implementation_version; use the matching notebook code")
    validate_architecture(resolved)
    return resolved


def saved_architecture(args):
    """Missing legacy fields mean baseline; malformed saved options must never read UI globals."""
    options = args.get("architecture", ARCH_DEFAULTS)
    if not isinstance(options, dict):
        raise ValueError("Saved architecture options must be a dictionary")
    return resolve_architecture(options)


def architecture_details(options):
    """Resolved topology accompanies options in configs, checkpoints and evaluation reports."""
    details = resolve_architecture(options)
    if MODEL not in ("A", "B", "C"):  # E/F/G define architecture_topology next to their model (section 4)
        return dict(details, model=MODEL, **architecture_topology(details), num_classes=NUM_CLASSES)
    channels = WIDTH_PRESETS[details["width_preset"]] if MODEL == "B" else (64, 128, 256)
    return dict(details, model=MODEL, stem_channels=16 if MODEL == "B" else 64,
                stage_channels=list(channels), blocks=[3, 3, 3] if MODEL == "B" else [3, 5, 3],
                num_classes=NUM_CLASSES)


def build_model(num_classes=NUM_CLASSES, architecture=None):
    """Build this notebook's model, check the 5M budget, and return (model, n_trainable_params)."""
    options = resolve_architecture(architecture)
    model = init_weights(make_model(num_classes, architecture=options))
    # Plain attributes do not alter legacy state_dict keys or baseline repr/fingerprints.
    model._project_architecture = options
    model._project_description = describe_architecture(options)
    n_params = count_params(model)
    assert n_params <= PARAM_BUDGET, f"model {MODEL} has {n_params:,} trainable params > {PARAM_BUDGET:,}"
    return model, n_params


_model, N_PARAMS = build_model()
print(f"Model {MODEL}: {_model._project_description}")
print(f"Architecture options: {_model._project_architecture}")
print(f"Trainable parameters: {N_PARAMS:,} (budget {PARAM_BUDGET:,})")
assert N_PARAMS <= PARAM_BUDGET
assert N_PARAMS == expected_param_count(_model._project_architecture), "Unexpected architecture parameter count"
if SHOW_SUMMARY:
    try:
        from torchinfo import summary
    except ImportError:
        print("torchinfo is not installed; layer summary skipped (the parameter count above is complete)")
    else:
        print(summary(_model, input_size=(1, 3, 32, 32), depth=3, col_names=("output_size", "num_params"),
                      device="cpu", verbose=0))
del _model''',
)

In [ ]:
# ---- knowledge distillation (T05): teacher families, teacher loading and checks, KD target, loss, identity
import contextlib
import types

KD_IMPLEMENTATION_VERSION = 1  # bump when the KD target, loss or teacher input changes meaning
TEACHER_NOTEBOOKS = {"A": "A_resnet.ipynb", "B": "B_wrn.ipynb", "C": "C_seresnet.ipynb",
                     "E": "E_pyramid.ipynb", "F": "F_shake.ipynb", "G": "G_resnext.ipynb"}
# Copied checkpoint helpers whose global names (MODEL, build_model, ...) must resolve inside a teacher family.
TEACHER_BOUND_HELPERS = ("model_fingerprint", "validate_checkpoint", "load_model")
TEACHER_KEYS = ("run", "root", "path", "checkpoint", "weights")
# Colab A100 bf16 reference for the slowdown estimate: E ver-1/T18 used 164.357 compute minutes for 300 epochs.
E_REFERENCE_RUN = "10-30-21-05_E_pyramid-ver-1-T18-basic110-ema"
E_REFERENCE_EPOCH_S = 164.357 * 60 / 300
# The competition cache and the 5k split shared by every A/B/C/E/F/G run (as checked in Q08_shiftval.ipynb).
COMPETITION_CACHE_FINGERPRINT = "42749a486a13184f0f96dab13293950f0120c357ddf1e6b1ef6a3bd3a9ce5fde"
EXPECTED_VAL_INDICES_SHA256 = "7957d060fcc34aefa5a43c70e31433927db55526494f67a9f7da553059be6767"
_TEACHER_NAMESPACES = {}


# Verbatim copy of section 7's load_model (the prediction cell runs after training, but teachers are needed before
# it). tests/test_distill.py checks that both definitions are identical; section 7 redefines the same function.
def load_model(ckpt, weights, device):
    options = saved_architecture(ckpt["args"])
    model, n_params = build_model(architecture=options)
    if "architecture" in ckpt and ckpt["architecture"] != architecture_details(options):
        raise ValueError("Checkpoint resolved architecture differs from its saved options")
    if "n_params" in ckpt and ckpt["n_params"] != n_params:
        raise ValueError("Checkpoint parameter count differs from its saved architecture")
    if ckpt.get("model_fingerprint") != model_fingerprint(model) or ckpt["args"]["model"] != MODEL:
        raise ValueError("Checkpoint architecture differs from this notebook")
    model.load_state_dict(ckpt["ema"]["module"] if weights == "ema" else ckpt["model"])
    return model.to(device).eval(), n_params


def _rebind(function, namespace):
    bound = types.FunctionType(function.__code__, namespace, function.__name__, function.__defaults__,
                               function.__closure__)
    bound.__kwdefaults__ = function.__kwdefaults__
    bound.__doc__ = function.__doc__
    return bound


def teacher_family(model):
    """Run the verbatim copy of `model`'s section 4 in a private namespace (cached), as its training notebook does.

    Teachers are always rebuilt from their saved architecture options; the copied cell's own baseline build uses
    the family defaults instead of UI switches, so this notebook needs none of the other families' switches.
    """
    if model in _TEACHER_NAMESPACES:
        return _TEACHER_NAMESPACES[model]
    if model not in TEACHER_MODEL_SOURCES:
        raise ValueError(f"Unknown teacher model family {model!r}; expected one of {sorted(TEACHER_MODEL_SOURCES)}")
    shared = {k: v for k, v in globals().items() if not k.startswith("_")}
    ns = dict(shared, __name__=f"kd_teacher_family_{model}", SHOW_SUMMARY=False)
    model_cell, budget_cell = TEACHER_MODEL_SOURCES[model]
    with contextlib.redirect_stdout(io.StringIO()):
        exec(compile(model_cell, f"{TEACHER_NOTEBOOKS[model]}:section4-model", "exec"), ns)
        ns["architecture_from_config"] = lambda: {}  # baseline options for the copied cell's own build only
        exec(compile(budget_cell, f"{TEACHER_NOTEBOOKS[model]}:section4-build", "exec"), ns)
    for name in TEACHER_BOUND_HELPERS:
        ns[name] = _rebind(globals()[name], ns)
    if ns["MODEL"] != model:
        raise RuntimeError(f"TEACHER_MODEL_SOURCES[{model!r}] defines model {ns['MODEL']!r}")
    _TEACHER_NAMESPACES[model] = ns
    return ns


@contextlib.contextmanager
def preserved_global_rng():
    """Teacher construction draws initial weights; restore Python/NumPy/torch RNGs so the student stream is unchanged."""
    state = get_rng_state()
    try:
        yield
    finally:
        set_rng_state(state)


def teacher_config_entries(entries):
    """JSON-ready copy of KD_TEACHERS (paths as strings); validated later by teacher_request."""
    if not isinstance(entries, (list, tuple)):
        raise ValueError("KD_TEACHERS must be a list of teacher run IDs or dicts")
    return [{k: str(v) if isinstance(v, Path) else v for k, v in e.items()} if isinstance(e, dict) else e
            for e in entries]


def teacher_request(entry):
    """Normalize one KD_TEACHERS entry (a run ID string or a dict with TEACHER_KEYS)."""
    if isinstance(entry, str):
        entry = {"run": entry}
    if not isinstance(entry, dict):
        raise ValueError(f"KD_TEACHERS entries must be run ID strings or dicts, got {entry!r}")
    unknown = set(entry) - set(TEACHER_KEYS)
    if unknown:
        raise ValueError(f"Unknown KD_TEACHERS keys {sorted(unknown)}; allowed {TEACHER_KEYS}")
    request = {"run": entry.get("run"), "root": entry.get("root"), "path": entry.get("path"),
               "checkpoint": entry.get("checkpoint", "submission"), "weights": entry.get("weights", "submission")}
    if not request["run"] and not request["path"]:
        raise ValueError(f"KD_TEACHERS entry {entry!r} needs a 'run' ID or an explicit checkpoint 'path'")
    if request["run"] is not None:
        if not isinstance(request["run"], str):
            raise ValueError("Teacher 'run' must be a run ID string")
        request["run"] = canonical_run_id(request["run"])
    if request["checkpoint"] not in ("submission", "best", "last"):
        raise ValueError("Teacher 'checkpoint' must be 'submission', 'best' or 'last'")
    if request["path"] is not None and "checkpoint" in entry:
        raise ValueError("An explicit teacher 'path' already names its checkpoint file; omit 'checkpoint'")
    if request["weights"] not in ("submission", "model", "ema"):
        raise ValueError("Teacher 'weights' must be 'submission', 'model' (raw) or 'ema'")
    return request


def _artifact_folders(result_root, run_id, category):
    """Canonical and legacy-alias folders of one artifact category for a run.

    `result_root` may be a list of roots in search order: the folders of the first root that has any are returned.
    """
    if isinstance(result_root, (list, tuple)):
        return next((found for root in result_root if (found := _artifact_folders(root, run_id, category))), [])
    if result_root is None:
        return []
    canonical = canonical_run_id(run_id)
    names = [canonical] + [legacy for legacy, target in RUN_ID_ALIASES.items() if target == canonical]
    parent = getattr(run_paths(result_root, canonical), category).parent
    return [parent / name for name in names if (parent / name).is_dir()]


def _read_json(path):
    try:
        return json.loads(Path(path).read_text())
    except (OSError, ValueError):
        print(f"Warning: unreadable {path}; ignored")
        return None


def _number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value)


def _checkpoint_role(name):
    name = str(name)
    return "last" if name.endswith("_last-ckpt.pt") else "best" if name.endswith("_best-ckpt.pt") else None


def recorded_submissions(result_root, run_id):
    """Distinct (checkpoint role, epoch, weights) of the run's saved prediction reports (submission CSVs)."""
    found = set()
    for folder in _artifact_folders(result_root, run_id, "test"):
        for path in sorted(folder.glob("*_prediction-report*.json")):
            data = _read_json(path)
            if data and data.get("weights") in ("model", "ema"):
                found.add((_checkpoint_role(data.get("checkpoint", "")), data.get("epoch"), data["weights"]))
    return sorted(found, key=str)


def recorded_clean_accuracies(result_root, run_id, ckpt):
    """Stored clean validation accuracies for this checkpoint's epoch: its training-log row, *-validation.json
    exports and prediction reports (the last two only when the run's result folders are available)."""
    refs = []
    metrics = ckpt.get("metrics") or {}
    for weights, key in (("model", "val_acc"), ("ema", "ema_val_acc")):
        if _number(metrics.get(key)):
            refs.append({"source": f"checkpoint metrics (epoch {ckpt['epoch']})", "weights": weights,
                         "tta": False, "accuracy_pct": float(metrics[key])})
    sources = [(path, "metrics") for folder in _artifact_folders(result_root, run_id, "validation")
               for path in sorted(folder.glob("*-validation.json"))]
    sources += [(path, "val") for folder in _artifact_folders(result_root, run_id, "test")
                for path in sorted(folder.glob("*_prediction-report*.json"))]
    for path, field in sources:
        data = _read_json(path)
        if (not data or data.get("epoch") != ckpt["epoch"]
                or data.get("cache_fingerprint", ckpt["cache_fingerprint"]) != ckpt["cache_fingerprint"]):
            continue
        for weights, values in (data.get(field) or {}).items():
            for mode, tta in (("no_tta", False), ("flip_tta", True)):
                if isinstance(values, dict) and _number(values.get(mode)):
                    refs.append({"source": path.name, "weights": weights, "tta": tta,
                                 "accuracy_pct": float(values[mode])})
    return refs


def available_extra_roots(extra_roots, wait_s=0.0):
    """The existing KD_EXTRA_TEACHER_ROOTS folders (read-only), in order. On Colab a MyDrive shortcut can appear a few
    seconds after mounting, so a missing folder is polled for up to `wait_s` seconds; it is then reported and skipped."""
    found = []
    for root in map(Path, extra_roots or []):
        deadline, waiting = time.monotonic() + max(0.0, float(wait_s)), False
        while True:
            try:
                if root.is_dir():
                    found.append(root)
                    print(f"KD extra teacher root (read-only): {root}")
                    break
            except OSError as error:
                print(f"KD: cannot access {root} yet: {error}")
            if time.monotonic() >= deadline:
                print(f"KD WARNING: extra teacher root {root} was not found; teachers that exist only there will not "
                      "be found. On Colab check the MyDrive shortcut name, or edit KD_EXTRA_TEACHER_ROOTS.", flush=True)
                break
            if not waiting:
                print(f"KD: waiting up to {wait_s:g}s for {root} (a Drive shortcut may appear after mounting)", flush=True)
                waiting = True
            time.sleep(min(3.0, max(0.05, deadline - time.monotonic())))
    return found


def open_teacher(entry, index, result_root, cache_fingerprint, train_idx, val_idx, device, channels_last=False,
                 default_root=None, extra_roots=()):
    """Load, identify and freeze one teacher exactly as resume/evaluation rebuild a checkpoint (no accuracy check).

    A run without its own "root" is searched in `default_root` (else `result_root`), then in `extra_roots`; its
    checkpoint comes from the first root that holds one, its recorded submission and stored accuracies from the first
    root that has them. The root used is recorded (`teacher.root`); roots are only read.
    """
    request = teacher_request(entry)
    if request["root"] is not None:
        roots = [Path(request["root"])]
    else:
        primary = default_root or result_root
        roots = [Path(r) for r in ([primary] if primary is not None else []) + list(extra_roots or [])]
    root = roots[0] if roots else None
    submissions = None
    if request["path"] is not None:
        path = Path(request["path"])
        role = _checkpoint_role(path.name) or "explicit"
        root = (path.parents[2] if path.parent.parent.name == "checkpoints" else None) if request["root"] is None else root
        roots = [root] if root is not None else []
    else:
        role = request["checkpoint"]
        holding = [r for r in roots if any(checkpoint_path(run_paths(r, request["run"]).checkpoints, x).is_file()
                                           for x in ("best", "last"))]
        if holding:
            root = holding[0]
        if role == "submission":
            submissions = recorded_submissions(roots, request["run"])
            roles = {s[:2] for s in submissions}
            if len(roles) != 1 or next(iter(roles))[0] is None:
                raise ValueError(f"Teacher {request['run']}: {'no' if not roles else 'ambiguous'} recorded submission "
                                 f"checkpoint under {', '.join(map(str, roots))} ({sorted(roles, key=str)}); "
                                 "set 'checkpoint' to 'best' or 'last'")
            role = next(iter(roles))[0]
        path = checkpoint_path(run_paths(root, request["run"]).checkpoints, role)
    if not path.is_file():
        searched = ", ".join(map(str, roots)) or "no result folder"
        raise FileNotFoundError(f"Teacher checkpoint not found: {path} (searched {searched}); set KD_TEACHER_ROOT, "
                                "KD_EXTRA_TEACHER_ROOTS (e.g. ['/content/drive/MyDrive/HW_DL_Project1/result'] for "
                                "H.W.'s shared folder), the entry's 'root', or an explicit 'path' (a teammate's Drive "
                                "folder needs a shortcut in MyDrive)")
    ckpt = load_checkpoint(path)
    if not isinstance(ckpt, dict) or "args" not in ckpt:
        raise ValueError(f"{path} is not a project training checkpoint")
    run_id = canonical_run_id(ckpt["args"]["run_name"])
    if request["run"] is not None and request["run"] != run_id:
        raise ValueError(f"Teacher checkpoint {path.name} belongs to {run_id}, not {request['run']}")
    family = ckpt["args"]["model"]
    fam = teacher_family(family)
    fam["validate_checkpoint"](ckpt, cache_fingerprint, run_id=run_id)
    weights = request["weights"]
    if weights == "submission":
        submissions = recorded_submissions(root, run_id) if submissions is None else submissions
        matching = sorted({s[2] for s in submissions if s[0] in (None, role) and s[1] in (None, ckpt["epoch"])})
        if len(matching) != 1:
            raise ValueError(f"Teacher {run_id}: {'no' if not matching else 'ambiguous'} recorded submission weights "
                             f"for its {role} checkpoint (epoch {ckpt['epoch']}); set 'weights' to 'model' or 'ema'")
        weights = matching[0]
    if request["checkpoint"] == "submission" and request["path"] is None:
        epochs = {s[1] for s in submissions if s[0] == role}
        if epochs != {ckpt["epoch"]}:
            raise ValueError(f"Teacher {run_id}: recorded submission epochs {sorted(epochs, key=str)} differ from the "
                             f"{role} checkpoint epoch {ckpt['epoch']}")
    if weights == "ema" and ckpt.get("ema") is None:
        raise ValueError(f"Teacher {run_id} has no EMA weights")
    # Teachers must not have trained on this student's validation images, and need a split for the accuracy check.
    teacher_val = ckpt["val_idx"].numpy()
    if not len(teacher_val):
        raise ValueError(f"Teacher {run_id} is a full-data refit: it trained on the validation split and its stored "
                         "accuracy cannot be reproduced; choose a teacher trained on the 45k split")
    if len(val_idx) and not np.array_equal(teacher_val, val_idx):
        raise ValueError(f"Teacher {run_id} uses another validation split (VAL_SIZE/SPLIT_SEED differ)")
    if len(val_idx) and np.intersect1d(ckpt["train_idx"].numpy(), val_idx).size:
        raise ValueError(f"Teacher {run_id} trained on this student's validation images")
    model, n_params = fam["load_model"](ckpt, weights, device)
    options = fam["saved_architecture"](ckpt["args"])
    expected = fam["expected_param_count"](options)
    if n_params != ckpt.get("n_params", n_params) or n_params != expected:
        raise ValueError(f"Teacher {run_id}: {n_params:,} parameters differ from the saved/expected count {expected:,}")
    model.eval()
    model.requires_grad_(False)
    if channels_last:
        model.to(memory_format=torch.channels_last)
    identity = {"run_id": run_id, "model": family, "checkpoint_role": role, "epoch": ckpt["epoch"], "weights": weights,
                "sha256": sha256_file(path), "n_params": n_params, "model_fingerprint": ckpt["model_fingerprint"],
                "architecture": options, "normalization": {"mean": list(ckpt["norm"]["mean"]), "std": list(ckpt["norm"]["std"])}}
    selection = {"checkpoint": "recorded submission" if request["checkpoint"] == "submission" and request["path"] is None
                 else "explicit", "weights": "recorded submission" if request["weights"] == "submission" else "explicit"}
    label = f"teacher {index + 1} ({family}) {run_id} {role} epoch {ckpt['epoch']} {weights}"
    # The checkpoint's own root first, then the remaining roots, for the stored clean accuracies.
    reference_roots = ([root] if root is not None else []) + [r for r in roots if r != root]
    return SimpleNamespace(model=model, family=family, identity=identity, selection=selection, label=label, path=path,
                           root=root, val_idx=teacher_val, norm=ckpt["norm"],
                           references=recorded_clean_accuracies(reference_roots, run_id, ckpt),
                           description=fam["describe_architecture"](options), scale=None, shift=None)


def load_teachers(args, result_root, cache_fingerprint, train_idx, val_idx, device):
    """Every configured teacher, frozen in eval mode; the global RNG state is left exactly as it was."""
    if cache_fingerprint == COMPETITION_CACHE_FINGERPRINT and (args.val_size, args.split_seed) == (5000, 0):
        digest = hashlib.sha256(np.ascontiguousarray(val_idx, dtype="<i8").tobytes()).hexdigest()
        if digest != EXPECTED_VAL_INDICES_SHA256:
            raise RuntimeError("The 5k validation split differs from the split shared by the other notebooks")
    extras = getattr(args, "kd_extra_teacher_roots", None) or []
    if extras:
        try:
            colab = detect_env() == "colab"
        except ValueError:
            colab = False
        extras = available_extra_roots(extras, getattr(args, "kd_extra_root_wait_s", 0.0) if colab else 0.0)
    with preserved_global_rng():
        teachers = [open_teacher(entry, i, result_root, cache_fingerprint, train_idx, val_idx, device,
                                 bool(args.channels_last), args.kd_teacher_root, extras)
                    for i, entry in enumerate(args.kd_teachers)]
    keys = [(t.identity["sha256"], t.identity["weights"]) for t in teachers]
    if len(set(keys)) != len(keys):
        raise ValueError("The same teacher checkpoint and weights are listed twice in KD_TEACHERS")
    return teachers


def attach_student_normalization(teachers, mean, std, device):
    """Map the student's normalized batch to each teacher's normalization of the same pixels.

    Teachers trained on the same 45k split have exactly the student's statistics; the map is then skipped and the
    teacher receives the student's tensor itself. Cutout's zeros (student mean colour) stay the same pixel colour.
    """
    for t in teachers:
        t_mean, t_std = list(t.norm["mean"]), list(t.norm["std"])
        if t_mean == list(mean) and t_std == list(std):
            t.scale = t.shift = None
        else:
            s_mean, s_std = torch.tensor(mean, device=device), torch.tensor(std, device=device)
            tm, ts = torch.tensor(t_mean, device=device), torch.tensor(t_std, device=device)
            t.scale = (s_std / ts).view(1, 3, 1, 1)
            t.shift = ((s_mean - tm) / ts).view(1, 3, 1, 1)
    return teachers


def assert_teachers_frozen(teachers):
    for t in teachers:
        if t.model.training or any(p.requires_grad for p in t.model.parameters()):
            raise RuntimeError(f"{t.label} is not frozen in eval mode")


def compile_teachers(teachers, mode):
    """KD_COMPILE_TEACHERS: each teacher's training-step forward (`teacher.compiled`); "off" keeps the module itself.

    Checks, KD-target accuracy and freezing tests always use the uncompiled `teacher.model`.
    """
    for t in teachers:
        t.compiled = compile_module(t.model, mode)
    return teachers


def teacher_log_probs(teachers, x, temperature, flip_average, device, amp, channels_last=False, compiled=False):
    """log of the mean teacher softmax(logits / T) over teachers (and the flipped view): float32 (B, classes), no grad.

    Every teacher sees the student's exact (augmented, possibly mixed) tensor `x`; with flip averaging also its
    horizontal mirror, in one batch. The mean is formed in log space (logsumexp) for numerical stability.
    compiled=True (training step only) calls each teacher's compile_teachers() wrapper when one is attached.
    """
    with torch.no_grad():
        views = torch.cat([x, x.flip(3)]) if flip_average else x
        if channels_last:
            views = views.contiguous(memory_format=torch.channels_last)
        logs = []
        for t in teachers:
            inp = views if t.scale is None else views * t.scale + t.shift
            net = getattr(t, "compiled", None) if compiled else None
            with autocast(device, amp):
                logits = (t.model if net is None else net)(inp)
            logs.append(F.log_softmax(logits.float() / temperature, dim=1).view(-1, x.shape[0], logits.shape[1]))
        stacked = torch.cat(logs)
        return torch.logsumexp(stacked, 0) - math.log(stacked.shape[0])


def distillation_loss(student_logits, teacher_log_p, y, criterion, alpha, temperature, partner_y=None, lam=1.0):
    """alpha * T^2 * KL(p_teacher^T || p_student^T) + (1 - alpha) * CE(student, hard labels).

    KL is summed over classes and averaged over the batch (reduction="batchmean") on log-softmax inputs. CE is the
    notebook's criterion (no label smoothing in distillation); with Mixup (partner_y given) it is the usual
    lambda-weighted CE of both labels. Returns (loss, soft KL, hard CE, correct-prediction count).
    """
    log_p_student = F.log_softmax(student_logits.float() / temperature, dim=1)
    soft = F.kl_div(log_p_student, teacher_log_p, reduction="batchmean", log_target=True)
    if partner_y is None:
        hard = criterion(student_logits, y)
        correct = (student_logits.argmax(1) == y).sum()
    else:
        hard, correct = mixup_loss_and_correct(student_logits, y, partner_y, lam, criterion)
    loss = alpha * temperature ** 2 * soft + (1.0 - alpha) * hard
    return loss, soft.detach(), hard.detach(), correct


def verify_teachers(teachers, images, labels, args, device):
    """Each teacher's clean validation accuracy (fp32, plain and flip) must reproduce its stored values."""
    checks = []
    for t in teachers:
        vx, vy = images[t.val_idx], labels[t.val_idx]
        kw = dict(batch_size=args.eval_batch_size, channels_last=bool(args.channels_last))
        mean, std = t.norm["mean"], t.norm["std"]
        plain = F.softmax(predict_logits(t.model, vx, mean, std, device, **kw), 1).numpy()
        flipped = F.softmax(predict_logits(t.model, vx, mean, std, device, flip=True, **kw), 1).numpy()
        measured = {False: float((plain.argmax(1) == vy).mean() * 100),
                    True: float(((plain + flipped).argmax(1) == vy).mean() * 100)}
        rows = [dict(ref, measured_pct=measured[ref["tta"]], difference_pp=measured[ref["tta"]] - ref["accuracy_pct"])
                for ref in t.references if ref["weights"] == t.identity["weights"]]
        if not rows:
            raise RuntimeError(f"{t.label}: no stored clean accuracy for this epoch/weights; cannot verify the teacher")
        worst = max(abs(r["difference_pp"]) for r in rows)
        if worst > args.kd_repro_tolerance_pp + 1e-9:
            details = "; ".join(f"{r['source']} {'flip' if r['tta'] else 'plain'} {r['accuracy_pct']:.2f}% vs "
                                f"{r['measured_pct']:.2f}%" for r in rows)
            raise RuntimeError(f"{t.label}: clean validation does not reproduce the stored accuracy within "
                               f"{args.kd_repro_tolerance_pp}pp ({details}); check the checkpoint, split and model code")
        checks.append({"teacher": t.identity["run_id"], "checkpoint_role": t.identity["checkpoint_role"],
                       "epoch": t.identity["epoch"], "weights": t.identity["weights"], "selection": t.selection,
                       "path": str(t.path), "root": None if t.root is None else str(t.root),
                       "description": t.description, "n_params": t.identity["n_params"],
                       "plain_pct": measured[False], "flip_pct": measured[True], "max_abs_difference_pp": worst,
                       "tolerance_pp": args.kd_repro_tolerance_pp, "references": rows, "status": "reproduced"})
    return checks


def kd_target_accuracy(teachers, images_u8, labels, mean, std, args, device):
    """Accuracy of the KD target's argmax on clean (unaugmented) validation images, computed as in training."""
    mean_t = torch.tensor(mean, device=device).view(1, 3, 1, 1)
    std_t = torch.tensor(std, device=device).view(1, 3, 1, 1)
    correct = 0
    for i in range(0, len(images_u8), args.eval_batch_size):
        xb = torch.as_tensor(np.asarray(images_u8[i:i + args.eval_batch_size])).to(device)
        x = to_float_nchw(xb, mean_t, std_t, bool(args.channels_last))
        log_p = teacher_log_probs(teachers, x, args.kd_temperature, args.kd_flip_average, device, args.amp,
                                  bool(args.channels_last))
        correct += int((log_p.argmax(1).cpu() == torch.as_tensor(labels[i:i + args.eval_batch_size])).sum())
    return 100.0 * correct / len(labels)


def validate_distillation_args(args):
    alpha, temperature = args.kd_alpha, args.kd_temperature
    if isinstance(alpha, bool) or not isinstance(alpha, (int, float)) or not math.isfinite(alpha) or not 0 <= alpha <= 1:
        raise ValueError("KD_ALPHA must be a number in [0, 1]")
    if (isinstance(temperature, bool) or not isinstance(temperature, (int, float)) or not math.isfinite(temperature)
            or temperature <= 0):
        raise ValueError("KD_TEMPERATURE must be a finite positive number")
    if not isinstance(args.kd_flip_average, bool):
        raise ValueError("KD_FLIP_AVERAGE must be True or False")
    if args.label_smoothing != 0.0:
        raise ValueError("Distillation replaces R0's label smoothing with the teacher target: set LABEL_SMOOTHING=0.0")
    if not isinstance(args.kd_teachers, (list, tuple)) or not args.kd_teachers:
        raise ValueError("KD_TEACHERS must list at least one teacher (section 1b)")
    for entry in args.kd_teachers:
        teacher_request(entry)
    tol = args.kd_repro_tolerance_pp
    if isinstance(tol, bool) or not isinstance(tol, (int, float)) or not math.isfinite(tol) or tol < 0:
        raise ValueError("KD_REPRO_TOLERANCE_PP must be a nonnegative number")
    if type(args.kd_implementation_version) is not int or args.kd_implementation_version != KD_IMPLEMENTATION_VERSION:
        raise ValueError("Unsupported distillation implementation version; use the matching notebook code")
    if not hasattr(args, "kd_compile_teachers"):
        args.kd_compile_teachers = "off"
    if not isinstance(args.kd_compile_teachers, str) or args.kd_compile_teachers not in COMPILE_MODES:
        raise ValueError(f"KD_COMPILE_TEACHERS = {args.kd_compile_teachers!r}; choose one of {COMPILE_MODES}")


def distillation_run_label(args):
    """Feature words of a KD run: `distill` plus mixup / the architecture feature / ema; RUN_NAME must start with it."""
    if args.run_name:
        words = re.split(r"[-_\s]+", str(args.run_name).strip())
        if words[0].lower() not in ("distill", "kd"):
            raise ValueError("Distillation runs must be marked: RUN_NAME must start with 'distill' or 'kd'")
        return args.run_name
    second = "mixup" if args.mixup else architecture_feature(args.architecture) or ("ema" if args.ema else None)
    return "distill" + (f"-{second}" if second else "")


def distillation_identity(args, teachers):
    """Scientific identity of the distillation recipe; its SHA-256 is the recipe fingerprint checked on resume."""
    identity = {"implementation_version": KD_IMPLEMENTATION_VERSION, "alpha": float(args.kd_alpha),
                "temperature": float(args.kd_temperature), "flip_average": bool(args.kd_flip_average),
                "label_smoothing": float(args.label_smoothing),
                "mixup": {key: getattr(args, key) for key in ("mixup", "mixup_alpha", "mixup_prob")},
                "teachers": [dict(t.identity) for t in teachers]}
    identity["fingerprint"] = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    return identity


def check_distillation_resume(saved_args, args):
    """Resume only the same distillation recipe: identical teachers (SHA-256, checkpoint, weights) and settings."""
    saved = saved_args.get("distillation")
    if not isinstance(saved, dict) or "fingerprint" not in saved:
        raise ValueError("Checkpoint is not a distillation run; resume it with its own notebook")
    current = args.distillation
    if saved["fingerprint"] == current["fingerprint"]:
        return
    differences = [f"{key}: requested {current.get(key)!r}, saved {saved.get(key)!r}"
                   for key in sorted(set(saved) | set(current)) if key not in ("teachers", "fingerprint")
                   and saved.get(key) != current.get(key)]
    old, new = saved.get("teachers", []), current["teachers"]
    if len(old) != len(new):
        differences.append(f"teacher count: requested {len(new)}, saved {len(old)}")
    for i, (a, b) in enumerate(zip(new, old)):
        for key in sorted(set(a) | set(b)):
            if a.get(key) != b.get(key):
                differences.append(f"teacher {i + 1} {key}: requested {a.get(key)!r}, saved {b.get(key)!r}")
    raise ValueError("Teacher set or distillation settings differ from the checkpoint; refusing to resume. "
                     + "; ".join(differences or ["recipe fingerprint differs"]))


class DistillCSVLogger(CSVLogger):
    """The notebook's CSV log plus the KD loss terms and the teacher-forward time of every epoch."""

    FIELDS = CSVLogger.FIELDS + ["kd_soft_loss", "kd_hard_loss", "kd_teacher_s", "kd_teacher_share"]


class TeacherTimer:
    """Teacher-forward time per epoch: CUDA events on CUDA (no extra host sync per step), else synchronize + clock."""

    def __init__(self, device):
        self.device, self.cuda = device, device.type == "cuda"
        self.pairs, self.seconds, self.t0 = [], 0.0, None

    def start(self):
        if self.cuda:
            event = torch.cuda.Event(enable_timing=True)
            event.record()
            self.pairs.append([event, None])
        else:
            synchronize(self.device)
            self.t0 = time.perf_counter()

    def stop(self):
        if self.cuda:
            event = torch.cuda.Event(enable_timing=True)
            event.record()
            self.pairs[-1][1] = event
        else:
            synchronize(self.device)
            self.seconds += time.perf_counter() - self.t0

    def pop_seconds(self):
        """Total since the last call; synchronizes once on CUDA."""
        if self.cuda:
            torch.cuda.synchronize(self.device)
            self.seconds += sum(start.elapsed_time(end) for start, end in self.pairs) / 1000.0
            self.pairs = []
        seconds, self.seconds = self.seconds, 0.0
        return seconds


def distillation_summary(state):
    """Identity, checks and teacher-forward timing for the training report."""
    args = state["args"]
    compute, teacher = state["elapsed_s"], state.get("kd_teacher_elapsed_s", 0.0)
    share = teacher / compute if compute > 0 else None
    mean_epoch = compute / max(1, state["epoch"])
    return {"identity": args.get("distillation"), "checks": state.get("distillation_checks"),
            "kd_target_val_accuracy_pct": state.get("kd_target_val_accuracy_pct"),
            "teacher_forward_s": round(teacher, 3), "teacher_forward_share": share,
            "mean_epoch_s": round(mean_epoch, 3),
            "slowdown_vs_teacher_free_loop": (1.0 / (1.0 - share)) if share is not None and share < 1 else None,
            "e_reference": {"run_id": E_REFERENCE_RUN, "mean_epoch_s": round(E_REFERENCE_EPOCH_S, 3),
                            "note": "Colab A100 40GB bf16; compare only on the same hardware"},
            "slowdown_vs_e_reference": round(mean_epoch / E_REFERENCE_EPOCH_S, 3)}


def distillation_report(run_dir, submission, data_dir, cache_dir, device, batch_size=1000):
    """Pre-declared T05 headline (final epoch, EMA, flip TTA) plus the full best/final x raw/EMA x plain/flip table."""
    artifacts = paths_for_run(run_dir)
    results = json.loads(artifact_path(artifacts, "logs", "report", ".json").read_text())
    table = {}
    for role in ("best", "last"):
        path = checkpoint_path(run_dir, role)
        if not path.exists():
            continue
        ck = load_checkpoint(path)
        if not len(ck["val_idx"]):
            continue
        same = ck["epoch"] == submission["epoch"] and Path(submission["checkpoint"]).name == path.name
        report = submission["val"] if same and submission["val"] else val_accuracy(ck, data_dir, cache_dir, device, batch_size)
        table[role] = {"epoch": ck["epoch"], **{f"{w}_{m}": report[w][m] for w in report for m in ("no_tta", "flip_tta")}}
    headline = None
    if "last" in table:
        weights = "ema" if "ema_flip_tta" in table["last"] else "model"
        headline = {"rule": "pre-declared: final epoch (last checkpoint), EMA weights (raw if EMA is off), flip TTA",
                    "epoch": table["last"]["epoch"], "weights": weights,
                    "accuracy_pct": table["last"][f"{weights}_flip_tta"]}
    summary = {"run_id": artifacts.run_id, "headline": headline, "table": table,
               "submission": {"file": submission["submission"], "checkpoint": Path(submission["checkpoint"]).name,
                              "epoch": submission["epoch"], "weights": submission["weights"], "tta": submission["tta"]},
               "distillation": results.get("distillation")}
    out = artifact_path(artifacts, "validation", "kd-summary", ".json", event=True, unique=True)
    write_json_atomic(summary, out)
    if headline:
        print(f"T05 HEADLINE | final epoch {headline['epoch']}, {headline['weights']}, flip TTA: "
              f"{headline['accuracy_pct']:.2f}% ({headline['rule']})")
    else:
        print("No validation split: no headline accuracy (full-data refit)")
    for role, row in table.items():
        print(f"  {role:4} epoch {row['epoch']}: " + ", ".join(f"{k} {v:.2f}%" for k, v in row.items() if k != "epoch"))
    kd = results.get("distillation") or {}
    if kd.get("teacher_forward_share") is not None:
        print(f"Teacher forward {kd['teacher_forward_s'] / 60:.1f} min ({kd['teacher_forward_share']:.1%} of compute); "
              f"mean epoch {kd['mean_epoch_s']:.1f}s = x{kd['slowdown_vs_e_reference']:.2f} of E ver-1/T18's "
              f"{kd['e_reference']['mean_epoch_s']:.1f}s on Colab A100")
    print(f"KD summary saved: {out}")
    return summary

In [ ]:
# Distillation-aware copy of E_pyramid.ipynb's trainer cell. Only RUNTIME_ARGS, make_args, write_results and train
# differ (marked "KD:"); tests/test_distill.py checks that every other definition is identical to E's.
# Runtime settings may change on resume; scientific training settings may not.
RUNTIME_ARGS = {"resume_run", "workers", "device", "log_interval", "eval_batch_size",
                "out_dir", "data_dir", "cache_dir", "run_name", "amp", "channels_last",
                "cudnn_benchmark", "pin_memory", "ckpt_every", "model_version", "task_id",
                "compile", "benchmark_steps",  # speed-only switches (README "Compilation")
                # KD: teacher locations and the check tolerance may change; the resolved identity may not.
                "kd_teachers", "kd_teacher_root", "kd_extra_teacher_roots", "kd_extra_root_wait_s",
                "kd_repro_tolerance_pp", "kd_compile_teachers"}


def make_args():
    amp = {True: "on", False: "off", None: "auto"}.get(AMP, AMP)
    for name, value, allowed in (("AUG", AUG, ("basic", "cutout", "autoaug")),
                                ("AUTOAUG_ERASE", AUTOAUG_ERASE, ("cutout", "erasing", "none")),
                                ("AMP", amp, ("auto", "on", "off", "bf16", "fp16")),
                                ("CHECKPOINT", CHECKPOINT, ("best", "last")),
                                ("COMPILE", COMPILE, COMPILE_MODES),
                                ("WEIGHTS", WEIGHTS, ("auto", "model", "ema"))):
        if value not in allowed:
            raise ValueError(f"{name} = {value!r}; choose one of {allowed}")
    if RESUME_RUN and RUN_NAME:
        raise ValueError("Set only RESUME_RUN for resume; leave RUN_NAME = None")
    if VAL_SIZE == 0 and WEIGHTS == "auto":
        raise ValueError("Full-data training needs WEIGHTS='model' or 'ema', chosen on earlier validation runs")
    if WEIGHTS == "ema" and not EMA:
        raise ValueError("WEIGHTS='ema' requires EMA=True")
    return SimpleNamespace(
        model=MODEL, architecture=resolve_architecture(), model_version=MODEL_VERSION, task_id=TASK_ID, run_name=RUN_NAME, resume_run=RESUME_RUN, out_dir=RESULT_DIR, seed=SEED,
        epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, momentum=MOMENTUM, nesterov=NESTEROV,
        weight_decay=WEIGHT_DECAY, wd_skip_bn_bias=WD_SKIP_BN_BIAS, warmup_epochs=WARMUP_EPOCHS,
        label_smoothing=LABEL_SMOOTHING, mixup=MIXUP, mixup_alpha=MIXUP_ALPHA, mixup_prob=MIXUP_PROB,
        mixup_implementation_version=1, ema=EMA, ema_decay=EMA_DECAY, data_dir=DATA_DIR, cache_dir=CACHE_DIR,
        val_size=VAL_SIZE, split_seed=SPLIT_SEED, train_subset=TRAIN_SUBSET, aug=AUG, cutout_size=CUTOUT_SIZE,
        autoaug_erase=AUTOAUG_ERASE, erasing_p=ERASING_P, workers=WORKERS, device=DEVICE,
        channels_last=CHANNELS_LAST, amp=amp, cudnn_benchmark=CUDNN_BENCHMARK, pin_memory=PIN_MEMORY,
        ckpt_every=CKPT_EVERY, eval_batch_size=EVAL_BATCH_SIZE, log_interval=LOG_INTERVAL,
        compile=COMPILE, benchmark_steps=BENCHMARK_STEPS,
        # KD: requested teachers and loss settings; train() resolves the teachers into `distillation`.
        kd_teachers=teacher_config_entries(KD_TEACHERS), kd_teacher_root=None if KD_TEACHER_ROOT is None else str(KD_TEACHER_ROOT),
        kd_extra_teacher_roots=[str(r) for r in (KD_EXTRA_TEACHER_ROOTS or [])], kd_extra_root_wait_s=KD_EXTRA_ROOT_WAIT_S,
        kd_alpha=KD_ALPHA,
        kd_temperature=KD_TEMPERATURE, kd_flip_average=KD_FLIP_AVERAGE, kd_repro_tolerance_pp=KD_REPRO_TOLERANCE_PP,
        kd_implementation_version=KD_IMPLEMENTATION_VERSION, kd_compile_teachers=KD_COMPILE_TEACHERS,
        distillation=None)


def default_run_name(args):
    """Each new experiment gets the next version for its notebook, independent of its minute."""
    notebook = {"A": "A_resnet", "B": "B_wrn", "C": "C_seresnet", "E": "E_pyramid", "F": "F_shake", "G": "G_resnext"}[args.model]
    feature = architecture_feature(args.architecture)
    if getattr(args, "mixup", False):
        default_label = f"{feature}-mixup" if feature else "mixup" + ("-ema" if args.ema else "")
    else:
        default_label = (feature or args.aug) + ("-ema" if args.ema else "")
    label = args.run_name or default_label
    words = re.split(r"[-_\s]+", label.strip())
    if not 1 <= len(words) <= 2 or any(not re.fullmatch(r"[A-Za-z0-9]+", word) for word in words):
        raise ValueError("RUN_NAME must contain one or two feature words (letters/digits), separated by space, - or _")
    features = '-'.join(word.lower() for word in words)
    root = Path(args.out_dir)
    used = set()
    pattern = re.compile(r"\d{2}-\d{2}-\d{2}-\d{2}_" + re.escape(notebook) + r"-ver-(\d+)-")
    for category in ('checkpoints', 'configs', 'logs', 'progress', 'plots', 'exports', 'manifests',
                     'evaluations/validation', 'evaluations/test'):
        for child in (root / category).glob('*'):
            match = pattern.match(canonical_run_id(child.name))
            if match:
                used.add(int(match.group(1)))
    version = getattr(args, 'model_version', None)
    if version is None:
        version = max(used, default=0) + 1
    if not isinstance(version, int) or isinstance(version, bool) or version < 1:
        raise ValueError("MODEL_VERSION must be a positive integer or None")
    if version in used:
        raise FileExistsError(f"{notebook} version {version} already exists; use MODEL_VERSION=None or RESUME_RUN")
    args.model_version = version
    task = getattr(args, "task_id", "T00")
    if not isinstance(task, str) or not re.fullmatch(r"T\d{2,}", task):
        raise ValueError("TASK_ID must be T followed by at least two digits, e.g. T26 or T00")
    return f"{run_timestamp()}_{notebook}-ver-{version}-{task}-{features}"


def validate_training_args(args):
    for key, default in MIXUP_DEFAULTS.items():
        if not hasattr(args, key):
            setattr(args, key, default)
    for key, default in PERFORMANCE_DEFAULTS.items():  # speed-only switches, recorded but never part of the recipe
        if not hasattr(args, key):
            setattr(args, key, default)
    if not isinstance(args.compile, str) or args.compile not in COMPILE_MODES:
        raise ValueError(f"COMPILE = {args.compile!r}; choose one of {COMPILE_MODES}")
    if not isinstance(args.benchmark_steps, int) or isinstance(args.benchmark_steps, bool) or args.benchmark_steps < 0:
        raise ValueError("BENCHMARK_STEPS must be a nonnegative integer (0 = train normally)")
    if args.benchmark_steps and getattr(args, "resume_run", None):
        raise ValueError("BENCHMARK_STEPS times a fresh model and never touches a run: set RESUME_RUN = None")
    if not isinstance(args.mixup, bool):
        raise ValueError("MIXUP must be True or False")
    if type(args.mixup_implementation_version) is not int or args.mixup_implementation_version != 1:
        raise ValueError("Unsupported Mixup implementation version")
    BatchMixup(args.mixup_alpha, args.mixup_prob, args.seed)  # validates without touching global RNG
    if args.mixup and (args.aug != "basic" or args.label_smoothing != 0.0):
        raise ValueError('The supported Mixup-only recipe requires explicit AUG="basic" and LABEL_SMOOTHING=0.0')
    args.architecture = resolve_architecture(getattr(args, "architecture", ARCH_DEFAULTS))
    for name in ("epochs", "batch_size", "eval_batch_size"):
        value = getattr(args, name)
        if not isinstance(value, int) or isinstance(value, bool) or value <= 0:
            raise ValueError(f"{name} must be a positive integer")
    for name in ("seed", "split_seed", "workers", "log_interval"):
        value = getattr(args, name)
        if value is not None and (not isinstance(value, int) or isinstance(value, bool) or value < 0):
            raise ValueError(f"{name} must be a nonnegative integer")
    if args.ckpt_every is not None and (not isinstance(args.ckpt_every, int) or args.ckpt_every <= 0):
        raise ValueError("ckpt_every must be a positive integer or None")
    _per_class(args.val_size, "val_size")
    _per_class(args.train_subset, "train_subset")
    if not math.isfinite(args.lr) or args.lr <= 0 or args.weight_decay < 0:
        raise ValueError("Learning rate must be positive and weight decay nonnegative")
    if not 0 <= args.label_smoothing < 1 or not 0 <= args.ema_decay < 1:
        raise ValueError("Label smoothing and EMA decay must be in [0,1)")
    if not 0 <= args.warmup_epochs <= args.epochs:
        raise ValueError("Set WARMUP_EPOCHS between 0 and EPOCHS (use 0 or 1 for a two-epoch smoke test)")
    if not 0 <= args.erasing_p <= 1 or not isinstance(args.cutout_size, int) or not 0 <= args.cutout_size <= 32:
        raise ValueError("Invalid erasing probability or Cutout size")
    if args.nesterov and args.momentum <= 0:
        raise ValueError("Nesterov requires positive momentum")


def fmt(x, spec=".2f"):
    return "-" if x is None or x == "" else format(x, spec)


def fmt_duration(seconds):
    return f"{seconds / 60:.0f}m" if seconds < 5940 else f"{seconds / 3600:.1f}h"


def write_results(artifacts, state, wall_elapsed=None):
    args = state["args"]
    result = {"run_id": artifacts.run_id, "model": args["model"], "n_params": state["n_params"],
              "architecture": architecture_details(saved_architecture(args)), "mixup": mixup_metadata(args),
              "model_description": describe_architecture(saved_architecture(args)),
              "epochs": state["epoch"], "planned_epochs": args["epochs"],
              "n_train": state["n_train"], "n_val": len(state["val_idx"]),
              "compute_time_min": round(state["elapsed_s"] / 60, 3),
              "wall_time_s": wall_elapsed if wall_elapsed is not None else state["wall_elapsed_s"],
              "wall_time_recovered_from_checkpoint": wall_elapsed is None,
              "peak_vram_mib": state["peak_vram_mib"], "final": state["metrics"], "best": state["best"],
              "cache_fingerprint": state["cache_fingerprint"], "model_fingerprint": state["model_fingerprint"],
              "compile": args.get("compile", "off"), "kd_compile_teachers": args.get("kd_compile_teachers", "off"),
              "distillation": distillation_summary(state)}  # KD: identity, teacher checks and timing
    write_json_atomic(result, artifact_path(artifacts, "logs", "report", ".json"))
    write_json_atomic({"status": "completed", "epoch": state["epoch"], "run_id": artifacts.run_id,
                       "updated_at": datetime.now(ZoneInfo("America/New_York")).isoformat()},
                      artifact_path(artifacts, "progress", "status", ".json"))
    return result


def train(args):
    validate_training_args(args)
    validate_distillation_args(args)  # KD
    session_start = time.perf_counter()
    paths = resolve_paths(args.data_dir, args.cache_dir, args.out_dir)
    args.data_dir, args.cache_dir, args.out_dir = str(paths.data_dir), str(paths.cache_dir), str(paths.result_dir)
    if args.resume_run:
        args.resume_run = canonical_run_id(args.resume_run)
        args.task_id = run_task_id(args.resume_run)
    else:
        args.run_name = distillation_run_label(args)  # KD: feature words start with distill/kd
    args.run_name = args.resume_run or default_run_name(args)
    if args.resume_run:
        version_match = re.search(r"-ver-(\d+)-", args.resume_run)
        if version_match:
            args.model_version = int(version_match.group(1))
    artifacts = run_paths(args.out_dir, args.run_name)
    run_dir = artifacts.checkpoints
    if not args.resume_run and any(p.exists() for k, p in vars(artifacts).items() if k not in ("root", "run_id")):
        raise FileExistsError("Run ID already exists; use RESUME_RUN or start a fresh timestamped run")
    if args.resume_run and not run_dir.is_dir():
        raise FileNotFoundError(f"Resume run does not exist: {run_dir}")

    cache_dir = ensure_cache(args.data_dir, args.cache_dir)
    fingerprint = verify_cache(cache_dir)["fingerprint"]
    images, labels, _ = load_train(args.data_dir, cache_dir)
    train_idx, val_idx = stratified_split(labels, args.val_size, args.split_seed)
    if args.train_subset:
        train_idx = stratified_subset(train_idx, labels, args.train_subset, args.split_seed)
    if len(train_idx) < args.batch_size:
        raise ValueError(f"{len(train_idx)} training images cannot form a batch of {args.batch_size}")
    device = get_device(args.device)
    resolve_hardware(args, device)
    args.kd_compile_teachers = resolve_compile(args.kd_compile_teachers, device, "KD_COMPILE_TEACHERS")  # KD
    args.ckpt_every = args.ckpt_every or default_ckpt_every(paths.env)
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = bool(args.cudnn_benchmark)
        torch.cuda.reset_peak_memory_stats(device)
    set_seed(args.seed)
    model, n_params = build_model(architecture=args.architecture)
    model_hash = model_fingerprint(model)
    # KD: rebuild, identify and freeze the teachers (global RNG preserved); their identity enters the recipe fingerprint.
    teachers = load_teachers(args, paths.result_dir, fingerprint, train_idx, val_idx, device)
    args.distillation = distillation_identity(args, teachers)
    if args.benchmark_steps:  # timing only, teacher forward included (README "Compilation"); no run folder or file
        mean, std = channel_stats(images[train_idx])
        attach_student_normalization(teachers, mean, std, device)
        compile_teachers(teachers, args.kd_compile_teachers)

        def kd_benchmark_loss(logits, target, y, criterion, partner_y, lam):
            loss, _, _, correct = distillation_loss(logits, target, y, criterion, args.kd_alpha, args.kd_temperature,
                                                    partner_y, lam)
            return loss, correct

        raise BenchmarkFinished(benchmark_training(
            args, model, images, labels, train_idx, val_idx, mean, std, device, cache_dir / "train_images.npy",
            target_fn=lambda x: teacher_log_probs(teachers, x, args.kd_temperature, args.kd_flip_average, device,
                                                  args.amp, bool(args.channels_last), compiled=True),
            loss_fn=kd_benchmark_loss, target_timer=TeacherTimer(device), target_compile=args.kd_compile_teachers))
    ckpt = recover_checkpoints(run_dir, fingerprint, model_hash) if args.resume_run else None
    if ckpt is not None:
        saved_args = dict(ckpt["args"])
        check_distillation_resume(saved_args, args)  # KD: same teachers, checkpoints, weights, alpha, T, flip, Mixup
        saved_args["architecture"] = saved_architecture(saved_args)
        for key, default in MIXUP_DEFAULTS.items():
            saved_args.setdefault(key, default)  # pre-Mixup checkpoints mean disabled, not the current UI value
        differences = [f"{k}: requested {getattr(args, k, None)!r}, saved {v!r}"
                       for k, v in saved_args.items()
                       if k not in RUNTIME_ARGS and getattr(args, k, None) != v]
        if differences:
            raise ValueError("Resume cannot change training settings; start a new run. " + "; ".join(differences))
        if not np.array_equal(val_idx, ckpt["val_idx"].numpy()) or not np.array_equal(train_idx, ckpt["train_idx"].numpy()):
            raise ValueError("Checkpoint split/subset differs from the current split")
        mean, std = ckpt["norm"]["mean"], ckpt["norm"]["std"]
    else:
        mean, std = channel_stats(images[train_idx])
    mixer = BatchMixup(args.mixup_alpha, args.mixup_prob, args.seed) if args.mixup else None
    if ckpt is not None:
        if mixer is not None:
            mixer.load_state_dict(ckpt.get("mixup_state"))
        elif ckpt.get("mixup_state") is not None:
            raise ValueError("Disabled Mixup conflicts with checkpoint Mixup state")
    # KD: every teacher must reproduce its stored clean validation accuracy before any run folder is created.
    distillation_checks = verify_teachers(teachers, images, labels, args, device)
    attach_student_normalization(teachers, mean, std, device)
    kd_target_acc = (kd_target_accuracy(teachers, images[val_idx], labels[val_idx], mean, std, args, device)
                     if len(val_idx) else None)
    artifacts = run_paths(args.out_dir, args.run_name, create=True)

    def say(message=""):
        print(message, flush=True)
        with (artifact_path(artifacts, "logs", "stdout", ".log")).open("a") as f:
            f.write(message + "\n")

    say(f"Run ID: {args.run_name}")
    say(f"To resume, set RESUME_RUN = {args.run_name!r} and RUN_NAME = None")
    say(summary_line(paths, device, args))
    if ckpt is not None and ckpt["epoch"] >= args.epochs:
        # Validate first; repair an absent final last-ckpt.pt if the final best was published before interruption.
        # The selected complete checkpoint is already validated; replace even an unreadable last-ckpt.pt.
        atomic_save(ckpt, checkpoint_path(run_dir, "last"))
        if not (artifact_path(artifacts, "logs", "report", ".json")).exists():
            write_results(artifacts, ckpt)
        say(f"Completed run recovered at epoch {ckpt['epoch']}; no further training")
        return run_dir

    for teacher, check in zip(teachers, distillation_checks):
        say(f"KD {teacher.label} | {check['n_params']:,} params | sha256 {teacher.identity['sha256'][:12]} | "
            f"clean val plain {check['plain_pct']:.2f}%, flip {check['flip_pct']:.2f}% | stored values reproduced "
            f"(max |diff| {check['max_abs_difference_pp']:.2f}pp <= {check['tolerance_pp']}pp, "
            f"{len(check['references'])} references)")
    say(f"KD alpha {args.kd_alpha} | T {args.kd_temperature} | teacher flip averaging {_on(args.kd_flip_average)} | "
        f"Mixup {_on(args.mixup)} | label smoothing {args.label_smoothing} | recipe fingerprint "
        f"{args.distillation['fingerprint'][:12]}"
        + (f" | KD-target accuracy on clean validation {kd_target_acc:.2f}%" if kd_target_acc is not None else ""))

    model.to(device)
    if args.channels_last:
        model.to(memory_format=torch.channels_last)
    ema = ModelEMA(model, args.ema_decay) if args.ema else None
    batches = TrainBatches(images, labels, train_idx, args, mean, std, device,
                           images_path=cache_dir / "train_images.npy")
    optimizer = build_optimizer(model, args)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, warmup_cosine_lambda(
        args.epochs * batches.steps_per_epoch, round(args.warmup_epochs * batches.steps_per_epoch)))
    criterion = nn.CrossEntropyLoss(label_smoothing=args.label_smoothing)
    scaler = torch.amp.GradScaler("cuda", enabled=args.amp == "fp16" and device.type == "cuda")
    start_epoch, elapsed, prior_wall = 0, 0.0, 0.0
    best = {"acc": -1.0, "epoch": None, "source": None}
    peak_vram = 0.0
    kd_teacher_elapsed = 0.0  # KD: cumulative teacher-forward seconds
    if ckpt is not None:
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        if ckpt["scaler"] and scaler.is_enabled():
            scaler.load_state_dict(ckpt["scaler"])
        if ema is not None:
            ema.load_state_dict(ckpt["ema"])
        set_rng_state(ckpt["rng"])
        start_epoch, elapsed, prior_wall, best = ckpt["epoch"], ckpt["elapsed_s"], ckpt["wall_elapsed_s"], ckpt["best"]
        peak_vram = ckpt.get("peak_vram_mib", 0.0)
        kd_teacher_elapsed = ckpt.get("kd_teacher_elapsed_s", 0.0)
    hardware = {"env": paths.env, "device": device_name(device), "amp": args.amp,
                "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
                "channels_last": args.channels_last, "workers": args.workers, "ckpt_every": args.ckpt_every,
                "compile": args.compile, "kd_compile_teachers": args.kd_compile_teachers}  # KD: teachers' mode
    config_path = artifact_path(artifacts, "configs", "config", ".json")
    if ckpt is None or not config_path.exists():
        config = {"args": vars(args), "model_description": model._project_description, "architecture": architecture_details(args.architecture), "n_params": n_params,
                  "mixup": mixup_metadata(vars(args)), "param_budget": PARAM_BUDGET, "n_train": len(train_idx), "n_val": len(val_idx),
                  "normalization": {"mean": mean, "std": std, "computed_on": "training split only"},
                  "versions": {"python": platform.python_version(), "torch": torch.__version__,
                               "torchvision": torchvision.__version__, "numpy": np.__version__,
                               "pandas": pd.__version__, "platform": platform.platform()},
                  "hardware": hardware, "cache_fingerprint": fingerprint, "model_fingerprint": model_hash,
                  "started": datetime.now(ZoneInfo("America/New_York")).isoformat(), "resumed": []}
        config["distillation"] = {"identity": args.distillation, "checks": distillation_checks,  # KD
                                  "kd_target_val_accuracy_pct": kd_target_acc,
                                  "objective": "alpha*T^2*KL(p_teacher^T || p_student^T) + (1-alpha)*CE(hard labels); "
                                               "KL batchmean on log-softmax; CE without label smoothing",
                                  "target": "mean of teacher softmax(logits/T) over teachers" +
                                            (" and the horizontally flipped view" if args.kd_flip_average else ""),
                                  "teacher_input": "the student's exact augmented (and, with Mixup, mixed) batch; "
                                                   "frozen eval-mode teachers, no grad, on the fly"}
    else:
        config = json.loads(config_path.read_text())
        config.setdefault("resumed", []).append({"at": datetime.now(ZoneInfo("America/New_York")).isoformat(),
                                                  "from_epoch": start_epoch, "hardware": hardware,
                                                  "distillation_checks": distillation_checks})  # KD
    write_json_atomic(config, config_path)
    (artifact_path(artifacts, "configs", "architecture", ".txt")).write_text(model._project_description + "\n" + json.dumps(architecture_details(args.architecture), sort_keys=True) + "\n" + repr(model) + "\n")
    say(f"Architecture: {model._project_description} | options {args.architecture}")
    say(f"Parameters {n_params:,} | train {len(train_idx):,} | validation {len(val_idx):,} | cache {fingerprint[:12]}")
    say(f"Mixup: {mixup_metadata(vars(args))}")
    train_label = "train weighted" if args.mixup else "train"
    logger = DistillCSVLogger(artifact_path(artifacts, "logs", "log", ".csv"),
                       resume_epoch=start_epoch if ckpt is not None else None, mixup=args.mixup)
    val_x, val_y = images[val_idx], labels[val_idx]
    eval_kw = dict(mean=mean, std=std, device=device, batch_size=args.eval_batch_size, channels_last=args.channels_last)
    train_forward = compile_module(model, args.compile)  # `model` itself when COMPILE = "off"; eval/saving use `model`
    compile_teachers(teachers, args.kd_compile_teachers)  # KD: only the training-step teacher forward is compiled
    epoch_timer = EpochTimer(f"compile {args.compile}, teachers {args.kd_compile_teachers}",
                             args.compile != "off" or args.kd_compile_teachers != "off")
    teacher_timer = TeacherTimer(device)  # KD
    for epoch in range(start_epoch, args.epochs):
        model.train()
        assert_teachers_frozen(teachers)  # KD
        t0 = time.perf_counter()
        loss_sum = torch.zeros((), device=device)
        soft_sum, hard_sum = torch.zeros((), device=device), torch.zeros((), device=device)  # KD
        correct = torch.zeros((), device=device, dtype=torch.float32 if args.mixup else torch.long)
        seen, skipped, mixup_batches, lambda_sum = 0, 0, 0, 0.0
        for step, (x, y) in enumerate(batches.epoch(epoch)):
            compile_step_begin(args.compile, args.kd_compile_teachers)
            optimizer.zero_grad(set_to_none=True)
            partner_y, lam = None, 1.0
            if mixer is not None:
                x, partner_y, lam, applied = mixer.apply(x, y)
                mixup_batches += int(applied)
                lambda_sum += lam if applied else 0.0
            # KD: the teachers see exactly the tensor the student sees (augmented once; mixed when Mixup triggers).
            teacher_timer.start()
            target = teacher_log_probs(teachers, x, args.kd_temperature, args.kd_flip_average, device, args.amp,
                                       bool(args.channels_last), compiled=True)
            teacher_timer.stop()
            with autocast(device, args.amp):
                logits = train_forward(x)
                loss, soft, hard, batch_correct = distillation_loss(logits, target, y, criterion, args.kd_alpha,
                                                                    args.kd_temperature, partner_y, lam)
            if not torch.isfinite(loss).item():
                raise FloatingPointError(f"Nonfinite loss at epoch {epoch+1}, step {step+1}")
            scaler.scale(loss).backward()
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            updated = not scaler.is_enabled() or scaler.get_scale() >= old_scale
            if updated:
                scheduler.step()
                if ema is not None:
                    ema.update(model)
            else:
                skipped += 1
            loss_sum += loss.detach().float() * y.size(0)
            soft_sum += soft * y.size(0)
            hard_sum += hard.float() * y.size(0)
            correct += batch_correct
            seen += y.size(0)
            if args.log_interval and (step + 1) % args.log_interval == 0:
                say(f"  step {step+1}/{batches.steps_per_epoch}, loss {loss_sum.item()/seen:.4f}")
        if skipped == batches.steps_per_epoch:
            raise FloatingPointError("Every fp16 optimizer update was skipped; use bf16 or fp32")
        row = {"epoch": epoch+1, "lr": optimizer.param_groups[0]["lr"],
               "train_loss": loss_sum.item()/seen, "train_acc": correct.item()/seen*100, "skipped_updates": skipped,
               "kd_soft_loss": soft_sum.item()/seen, "kd_hard_loss": hard_sum.item()/seen}
        if mixer is not None:
            row.update(mixup_batches=mixup_batches, mixup_fraction=mixup_batches/batches.steps_per_epoch,
                       mixup_lambda_mean=lambda_sum/mixup_batches if mixup_batches else 1.0)
        if len(val_idx):
            row["val_loss"], row["val_acc"] = evaluate(model, val_x, val_y, **eval_kw)
            if ema is not None:
                row["ema_val_loss"], row["ema_val_acc"] = evaluate(ema.module, val_x, val_y, **eval_kw)
        synchronize(device)
        epoch_time = time.perf_counter() - t0
        elapsed += epoch_time
        row.update(epoch_time_s=round(epoch_time, 3), elapsed_min=round(elapsed/60, 3))
        teacher_s = teacher_timer.pop_seconds()  # KD: teacher-forward time inside this epoch
        kd_teacher_elapsed += teacher_s
        row.update(kd_teacher_s=round(teacher_s, 3), kd_teacher_share=round(teacher_s/epoch_time, 4))
        if device.type == "cuda":
            peak_vram = max(peak_vram, torch.cuda.max_memory_allocated(device)/2**20)
        is_best = False
        if len(val_idx):
            score, selected = row["val_acc"], "model"
            if ema is not None and row["ema_val_acc"] > score:
                score, selected = row["ema_val_acc"], "ema"
            if score > best["acc"]:
                best, is_best = {"acc": score, "epoch": epoch+1, "source": selected}, True
        logger.log(row)
        state = {"format": CHECKPOINT_FORMAT, "epoch": epoch+1, "args": vars(args), "n_params": n_params,
                 "architecture": architecture_details(args.architecture),
                 "model": model.state_dict(), "ema": ema.state_dict() if ema is not None else None,
                 "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                 "norm": {"mean": mean, "std": std}, "train_idx": torch.from_numpy(train_idx),
                 "val_idx": torch.from_numpy(val_idx), "n_train": len(train_idx), "metrics": row, "best": best,
                 "rng": get_rng_state(), "mixup_state": mixer.state_dict() if mixer is not None else None, "elapsed_s": elapsed,
                 "wall_elapsed_s": prior_wall + time.perf_counter()-session_start, "peak_vram_mib": peak_vram,
                 "cache_fingerprint": fingerprint, "model_fingerprint": model_hash,
                 "distillation_checks": distillation_checks, "kd_target_val_accuracy_pct": kd_target_acc,
                 "kd_teacher_elapsed_s": kd_teacher_elapsed}  # KD (the identity is in args["distillation"])
        save_epoch_checkpoints(state, run_dir, is_best,
                               (epoch+1) % args.ckpt_every == 0 or epoch+1 == args.epochs)
        write_json_atomic({"status": "running", "run_id": artifacts.run_id, "epoch": epoch+1,
                           "best": best, "updated_at": datetime.now(ZoneInfo("America/New_York")).isoformat()},
                          artifact_path(artifacts, "progress", "status", ".json"))
        say(f"ep {epoch+1:3}/{args.epochs} | loss {row['train_loss']:.4f} | {train_label} {row['train_acc']:.2f}% | "
            f"val {fmt(row.get('val_acc'))}% | EMA {fmt(row.get('ema_val_acc'))}% | "
            f"{epoch_time:.1f}s (teachers {teacher_s:.1f}s, {teacher_s/epoch_time:.0%})"
            f"{' * best saved' if is_best else ''}")
        timing = epoch_timer.add(epoch_time)
        if timing:
            say(timing)
    if epoch_timer.final():
        say(epoch_timer.final())
    summary = write_results(artifacts, state, prior_wall + time.perf_counter()-session_start)["distillation"]
    say(f"KD timing: teacher forward {summary['teacher_forward_s']/60:.1f} min = {summary['teacher_forward_share']:.1%} "
        f"of compute; mean epoch {summary['mean_epoch_s']:.1f}s, x{summary['slowdown_vs_e_reference']:.2f} of E ver-1/T18's "
        f"{summary['e_reference']['mean_epoch_s']:.1f}s (Colab A100 bf16)")
    say(f"Finished. Checkpoints: {run_dir}; logs: {artifacts.logs}")
    return run_dir

In [ ]:
RUN_DIR = train(make_args())

## 6. Training curves

Loss and accuracy come from `result/logs/<RUN_ID>/<RUN_ID>_log.csv`; the plot is saved under `result/plots/<RUN_ID>`.
Training uses augmented images and the configured objective; Mixup curves are labeled **train weighted**.
Their accuracy is lambda-weighted and should not be compared directly with clean validation accuracy.
Validation uses unaugmented images and ordinary cross-entropy.


In [ ]:
# Categorical colours in fixed order (one per run, or one per metric for a single run).
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, INK_MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"


def load_log(run_dir):
    df = pd.read_csv(artifact_path(paths_for_run(run_dir), "logs", "log", ".csv"))
    return df.apply(pd.to_numeric, errors="coerce")


def _style(ax, ylabel):
    ax.set_xlabel("epoch", color=INK_MUTED)
    ax.set_ylabel(ylabel, color=INK_MUTED)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(INK_MUTED)
    ax.tick_params(colors=INK_MUTED)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))


def _end_label(ax, df, col, color):
    s = df[["epoch", col]].dropna()
    if len(s):
        x, y = s.iloc[-1]
        ax.plot([x], [y], "o", color=color, markersize=4)
        ax.annotate(f"{y:.2f}%", (x, y), xytext=(4, 0), textcoords="offset points",
                    va="center", fontsize=8, color=INK)


def plot_runs(run_dirs, labels=None, out=None, title=None):
    """Two panels (loss, accuracy). One run: colour = metric. Several runs: colour = run,
    line style = metric (val solid, EMA val dashed, train dotted)."""
    run_dirs = [Path(r) for r in run_dirs]
    if len(run_dirs) > len(SERIES):
        raise ValueError(f"at most {len(SERIES)} runs per figure; split them into several figures")
    labels = labels or [r.name for r in run_dirs]
    single = len(run_dirs) == 1
    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
    losses = []

    for i, (run, name) in enumerate(zip(run_dirs, labels)):
        df = load_log(run)
        losses += [df[c].dropna().to_numpy() for c in ("train_loss", "val_loss", "ema_val_loss")]
        has_ema = df["ema_val_acc"].notna().any()
        if single:
            c_train, c_val, c_ema = SERIES[0], SERIES[1], SERIES[2]
            ls_train, prefix = "-", ""
        else:
            c_train = c_val = c_ema = SERIES[i]
            ls_train, prefix = ":", f"{name} "
        lw = dict(linewidth=2)
        train_label = "train weighted" if "mixup_batches" in df.columns else "train"
        ax_loss.plot(df["epoch"], df["train_loss"], ls_train, color=c_train, label=f"{prefix}{train_label}", **lw)
        ax_acc.plot(df["epoch"], df["train_acc"], ls_train, color=c_train, label=f"{prefix}{train_label}", **lw)
        if df["val_acc"].notna().any():
            ax_loss.plot(df["epoch"], df["val_loss"], "-", color=c_val, label=f"{prefix}val", **lw)
            ax_acc.plot(df["epoch"], df["val_acc"], "-", color=c_val, label=f"{prefix}val", **lw)
            _end_label(ax_acc, df, "val_acc", c_val)
        if has_ema:
            ax_loss.plot(df["epoch"], df["ema_val_loss"], "--", color=c_ema, label=f"{prefix}EMA val", **lw)
            ax_acc.plot(df["epoch"], df["ema_val_acc"], "--", color=c_ema, label=f"{prefix}EMA val", **lw)
            if single:
                _end_label(ax_acc, df, "ema_val_acc", c_ema)

    _style(ax_loss, "loss (training objective; validation CE)")
    _style(ax_acc, "accuracy (%)")
    ax_loss.set_title("Loss", color=INK, loc="left")
    ax_acc.set_title("Accuracy", color=INK, loc="left")
    loss_values = np.concatenate(losses)
    if len(loss_values) > 5:  # keep a few huge early-epoch losses from flattening the curves
        ax_loss.set_ylim(0, min(loss_values.max(), 1.2 * np.percentile(loss_values, 95)) * 1.05)
    handles, names = ax_acc.get_legend_handles_labels()
    fig.legend(handles, names, loc="outside lower center", ncol=3 if not single else len(names),
               frameon=False, fontsize=8)
    if title:
        fig.suptitle(title, color=INK)
    if out is not None:
        fig.savefig(out, dpi=150)
    return fig


fig = plot_runs([RUN_DIR], out=artifact_path(paths_for_run(RUN_DIR), "plots", "curves", ".png"), title=f"Model {MODEL}: {RUN_DIR.name}")
plt.show()
print(f"Saved {artifact_path(paths_for_run(RUN_DIR), 'plots', 'curves', '.png')}")

## 7. Prediction and submission

The checkpoint's dataset and architecture identities are checked before evaluation. For a validation run,
`WEIGHTS="auto"` selects raw or EMA weights using the requested inference mode: plain accuracy when `TTA=False`,
flip-TTA accuracy when `TTA=True`. This choice is among weights at the selected checkpoint; training selects
the best epoch using non-TTA validation. Full-data refits require explicit weights and use the completed `<RUN_ID>_last-ckpt.pt`.

Timestamped validation reports contain all four raw/EMA × plain/flip comparisons, per-class recall,
confusion matrices and per-example probabilities. Selected-model error plots are saved in `plots`.
Test predictions, format checks, checkpoint hashes and submission CSVs go in `evaluations/test`.
The CSV follows the sample's exact ID order; hidden-test accuracy remains unknown until Kaggle returns it.


In [ ]:
def resolve_checkpoint(run_dir, which="best"):
    run_dir = Path(run_dir)
    last = checkpoint_path(run_dir, "last")
    if last.exists() and load_checkpoint(last)["args"]["val_size"] == 0:
        return last  # A full-data run always uses its agreed final epoch.
    path = checkpoint_path(run_dir, which)
    if not path.exists():
        raise FileNotFoundError(f"Requested checkpoint missing: {path}; resume the run to repair/recover it")
    return path


def choose_weights(ckpt, requested, tta=False, val_report=None):
    if requested not in ("auto", "model", "ema"):
        raise ValueError("weights must be auto, model or ema")
    if requested != "auto":
        if requested == "ema" and ckpt["ema"] is None:
            raise ValueError("Checkpoint has no EMA weights")
        return requested
    if not len(ckpt["val_idx"]):
        raise ValueError("No validation split: explicitly use the weights selected in earlier validation experiments")
    if not val_report:
        raise ValueError("Automatic selection requires a fresh validation report")
    metric = "flip_tta" if tta else "no_tta"
    # Ties favor raw weights, consistently with training's best-checkpoint convention.
    return max(val_report, key=lambda name: (val_report[name][metric], name == "model"))


def load_model(ckpt, weights, device):
    options = saved_architecture(ckpt["args"])
    model, n_params = build_model(architecture=options)
    if "architecture" in ckpt and ckpt["architecture"] != architecture_details(options):
        raise ValueError("Checkpoint resolved architecture differs from its saved options")
    if "n_params" in ckpt and ckpt["n_params"] != n_params:
        raise ValueError("Checkpoint parameter count differs from its saved architecture")
    if ckpt.get("model_fingerprint") != model_fingerprint(model) or ckpt["args"]["model"] != MODEL:
        raise ValueError("Checkpoint architecture differs from this notebook")
    model.load_state_dict(ckpt["ema"]["module"] if weights == "ema" else ckpt["model"])
    return model.to(device).eval(), n_params


def validate_submission(path, sample_path):
    sub, sample = pd.read_csv(path), pd.read_csv(sample_path)
    if list(sub.columns) != ["id", "label"] or len(sub) != len(sample):
        raise ValueError("Submission columns or row count differ from the sample")
    if not sub["id"].is_unique or not np.array_equal(sub["id"], sample["id"]):
        raise ValueError("Submission IDs are duplicated or differ from sample order")
    if not pd.api.types.is_integer_dtype(sub["label"]) or not sub["label"].between(0,9).all():
        raise ValueError("Submission labels must be integers in 0..9")
    return len(sub)


def classification_metrics(probs, labels):
    labels = np.asarray(labels)
    probs = np.asarray(probs)
    pred = probs.argmax(1)
    cm = np.bincount(labels * NUM_CLASSES + pred, minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)
    support = cm.sum(1)
    recall = np.divide(cm.diagonal(), support, out=np.zeros(NUM_CLASSES, dtype=float), where=support > 0)
    return {"accuracy_pct": float((pred == labels).mean()*100), "correct": int((pred == labels).sum()),
            "n": len(labels), "cross_entropy": float(-np.log(probs[np.arange(len(labels)), labels].clip(1e-12)).mean()),
            "per_class_recall": {name: float(value) if count else None
                                 for name, value, count in zip(CLASSES, recall, support)},
            "confusion_matrix": cm.tolist()}


def val_accuracy(ckpt, data_dir, cache_dir, device, batch_size=1000, save_prefix=None):
    fingerprint = verify_cache(ensure_cache(data_dir, cache_dir))["fingerprint"]
    validate_checkpoint(ckpt, fingerprint)
    val_idx = ckpt["val_idx"].numpy()
    if not len(val_idx):
        return {}
    images, labels, ids = load_train(data_dir, cache_dir)
    vx, vy = images[val_idx], labels[val_idx]
    mean, std = ckpt["norm"]["mean"], ckpt["norm"]["std"]
    report, arrays = {}, {"ids": ids[val_idx], "indices": val_idx, "labels": vy}
    for weights in (["model", "ema"] if ckpt["ema"] is not None else ["model"]):
        model = load_model(ckpt, weights, device)[0]
        plain = F.softmax(predict_logits(model, vx, mean, std, device, batch_size), 1).numpy()
        flipped = F.softmax(predict_logits(model, vx, mean, std, device, batch_size, flip=True), 1).numpy()
        report[weights] = {"details": {}}
        for name, probs in (("no_tta", plain), ("flip_tta", 0.5*(plain+flipped))):
            metrics = classification_metrics(probs, vy)
            report[weights][name] = metrics["accuracy_pct"]
            report[weights]["details"][name] = metrics
            arrays[f"{weights}_{name}_probabilities"] = probs
            arrays[f"{weights}_{name}_predictions"] = probs.argmax(1)
    if save_prefix is not None:
        save_prefix = Path(save_prefix)
        with save_prefix.with_suffix(".npz").open("wb") as f:
            np.savez_compressed(f, **arrays)
        write_json_atomic({"epoch": ckpt["epoch"], "cache_fingerprint": fingerprint,
                           "normalization": ckpt["norm"], "metrics": report}, save_prefix.with_suffix(".json"))
    return report


def plot_validation_diagnostics(ckpt, weights, tta, report, arrays_path, data_dir, cache_dir, out_prefix):
    mode = "flip_tta" if tta else "no_tta"
    cm = np.array(report[weights]["details"][mode]["confusion_matrix"])
    fig, ax = plt.subplots(figsize=(8,7), layout="constrained")
    ax.imshow(cm, cmap="Blues")
    ax.set(xticks=range(10), yticks=range(10), xticklabels=CLASSES, yticklabels=CLASSES,
           xlabel="Predicted label", ylabel="True label", title=f"Validation confusion matrix: {weights}, {mode}")
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    for i in range(10):
        for j in range(10):
            ax.text(j, i, str(cm[i,j]), ha="center", va="center", fontsize=8,
                    color="white" if cm[i,j] > cm.max()/2 else "black")
    fig.savefig(str(out_prefix)+"-confusion.png", dpi=150)
    plt.close(fig)
    with np.load(arrays_path, allow_pickle=False) as arrays:
        labels = arrays["labels"]
        predictions = arrays[f"{weights}_{mode}_predictions"]
        indices = arrays["indices"]
        wrong = np.flatnonzero(predictions != labels)[:12]
        if len(wrong):
            images = load_train(data_dir, cache_dir)[0]
            fig, axes = plt.subplots(3,4,figsize=(10,8),layout="constrained")
            for ax in axes.flat:
                ax.axis("off")
            for ax, pos in zip(axes.flat, wrong):
                ax.imshow(images[indices[pos]])
                ax.set_title(f"true: {CLASSES[labels[pos]]}\npred: {CLASSES[predictions[pos]]}", fontsize=9)
            fig.savefig(str(out_prefix)+"-errors.png", dpi=150)
            plt.close(fig)


def predict_run(run_dir, which="best", weights="auto", tta=True, data_dir=None, cache_dir=None, device=None,
                batch_size=1000):
    device = device or get_device(DEVICE)
    fingerprint = verify_cache(ensure_cache(data_dir, cache_dir))["fingerprint"]
    artifacts = paths_for_run(run_dir)
    ckpt_path = resolve_checkpoint(run_dir, which)
    ckpt = load_checkpoint(ckpt_path)
    validate_checkpoint(ckpt, fingerprint, run_id=artifacts.run_id)
    if ckpt["args"]["val_size"] == 0 and (ckpt["epoch"] != ckpt["args"]["epochs"] or ckpt["args"]["train_subset"]):
        raise ValueError("Full-data submission requires a completed run without a training subset")
    checkpoint_role = "last" if ckpt["args"]["val_size"] == 0 else which
    val_prefix = artifact_path(artifacts, "validation", f"{checkpoint_role}-validation", "", event=True, unique=True)
    val_report = val_accuracy(ckpt, data_dir, cache_dir, device, batch_size, save_prefix=val_prefix)
    weights = choose_weights(ckpt, weights, tta, val_report)
    model, n_params = load_model(ckpt, weights, device)
    mean, std = ckpt["norm"]["mean"], ckpt["norm"]["std"]
    print(f"Checkpoint {ckpt_path.name}, epoch {ckpt['epoch']}; weights {weights}; flip TTA {tta}; params {n_params:,}")
    for name, metrics in val_report.items():
        print(f"Validation {name}: plain {metrics['no_tta']:.2f}%, flip {metrics['flip_tta']:.2f}%")
    if val_report:
        plot_validation_diagnostics(ckpt, weights, tta, val_report, val_prefix.with_suffix(".npz"),
                                    data_dir, cache_dir, artifact_path(artifacts, "plots", checkpoint_role, "", event=True, unique=True))
    test_images, test_ids = load_test(data_dir, cache_dir)
    probs = predict_probs(model, test_images, mean, std, device, batch_size, tta=tta).numpy()
    sample_path = Path(cache_dir) / "sample_submission.csv"
    sample = pd.read_csv(sample_path)
    row_of = {int(value): i for i, value in enumerate(test_ids)}
    if len(row_of) != len(test_ids) or set(sample["id"]) != set(row_of):
        raise ValueError("Test IDs differ from sample or contain duplicates")
    order = np.array([row_of[int(value)] for value in sample["id"]])
    ordered = probs[order]
    out = artifact_path(artifacts, "test", "submission", ".csv", event=True, unique=True)
    pd.DataFrame({"id": sample["id"], "label": ordered.argmax(1).astype(np.int64)}).to_csv(out,index=False)
    n = validate_submission(out, sample_path)
    np.savez_compressed(out.with_name(out.stem.replace("_submission", "_probabilities") + ".npz"), ids=sample["id"].to_numpy(), probabilities=ordered)
    if detect_env() == "kaggle":
        shutil.copyfile(out, work_dir() / "submission.csv")
    report = {"checkpoint": str(ckpt_path), "checkpoint_sha256": sha256_file(ckpt_path),
              "epoch": ckpt["epoch"], "epochs": ckpt["args"]["epochs"], "model": MODEL, "n_params": n_params,
              "architecture": architecture_details(saved_architecture(ckpt["args"])),
              "model_description": model._project_description,
              "n_train": ckpt["n_train"], "n_val": len(ckpt["val_idx"]), "weights": weights, "tta": tta,
              "val": val_report, "submission": str(out), "rows": n, "format_check": "passed",
              "cache_fingerprint": fingerprint, "validation_artifact_prefix": str(val_prefix) if val_report else None,
              "kaggle_public_accuracy": None, "kaggle_private_accuracy": None}
    write_json_atomic(report, out.with_name(out.stem.replace("_submission", "_prediction-report") + ".json"))
    print(f"Submission saved: {out}. Hidden-test accuracy is not available locally.")
    return report


def export_run(run_dir, include_checkpoints=False):
    artifacts = paths_for_run(run_dir)
    records = []
    for name, folder in vars(artifacts).items():
        if name in ("root", "run_id", "exports", "manifests"):
            continue
        for path in sorted(folder.glob("*")):
            if path.is_file() and not path.name.endswith(".tmp"):
                records.append({"path": str(path.relative_to(artifacts.root)), "bytes": path.stat().st_size,
                                "sha256": sha256_file(path), "included_in_bundle": name != "checkpoints" or include_checkpoints})
    manifest = artifact_path(artifacts, "manifests", "manifest", ".json", event=True, unique=True)
    write_json_atomic({"run_id": artifacts.run_id, "files": records}, manifest)
    bundle = artifact_path(artifacts, "exports", "review-bundle", ".zip", event=True, unique=True)
    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
        for record in records:
            if record["included_in_bundle"]:
                z.write(artifacts.root / record["path"], record["path"])
        z.write(manifest, str(manifest.relative_to(artifacts.root)))
    print(f"Review bundle: {bundle}")
    print(f"Keep full checkpoints separately in {artifacts.checkpoints}")
    return bundle


In [ ]:
SUBMISSION = predict_run(RUN_DIR, CHECKPOINT, WEIGHTS, TTA, PATHS.data_dir, CACHE, DEV, EVAL_BATCH_SIZE)

# Compare the final and selected best epochs without confusing their identities.
VAL_RESULTS = {}
for key, name in (("final", "last-ckpt.pt"), ("best", "best-ckpt.pt")):
    path = checkpoint_path(RUN_DIR, key if key == "best" else "last")
    if not path.exists():
        continue
    ck = load_checkpoint(path)
    if ck["epoch"] == SUBMISSION["epoch"]:
        acc = SUBMISSION["val"]
    else:
        prefix = artifact_path(paths_for_run(RUN_DIR), "validation", f"{key}-validation", "", event=True, unique=True)
        acc = val_accuracy(ck, PATHS.data_dir, CACHE, DEV, EVAL_BATCH_SIZE, save_prefix=prefix)
    if acc:
        weights = choose_weights(ck, WEIGHTS, TTA, acc)
        VAL_RESULTS[key] = {"epoch": ck["epoch"], "weights": weights,
                            "no_tta": acc[weights]["no_tta"], "flip_tta": acc[weights]["flip_tta"]}
write_json_atomic(VAL_RESULTS, artifact_path(paths_for_run(RUN_DIR), "validation", "epoch-comparison", ".json", event=True, unique=True))


### 7b. T05 headline

The pre-declared result is the **final epoch with EMA weights and flip TTA** (raw weights only if `EMA=False`). The cell
below also tabulates best/final × raw/EMA × plain/flip, and saves a dated `kd-summary.json` (headline, table,
distillation identity, teacher checks and timing) in `evaluations/validation/<RUN_ID>/`, which the review bundle
includes. Compare it with E ver-1/T18 under the same rule (final300 EMA+flip 96.84%), not with T18's best epoch.

In [ ]:
# T05 headline under the pre-declared rule (final epoch, EMA weights, flip TTA) plus the
# best/final x raw/EMA x plain/flip table; saved before the review bundle is exported.
KD_SUMMARY = distillation_report(RUN_DIR, SUBMISSION, PATHS.data_dir, CACHE, DEV, EVAL_BATCH_SIZE)

## 8. Final results and review bundle

The last cell creates a compact ZIP with configuration, logs, evaluations, plots and a checksum manifest.
Full checkpoints stay separately on Drive and are excluded from this review bundle by default. Download the
bundle for each experiment after training. These bundles and actual Kaggle scores will support the next review
and the later `Archive/improvement.md`; no model refinements are inferred from unrun experiments.


In [ ]:
artifacts = paths_for_run(RUN_DIR)
results = json.loads((artifact_path(artifacts, "logs", "report", ".json")).read_text())
print(f"FINAL RESULTS | scheme {MODEL}: {results.get('model_description', MODEL_DESCRIPTION)}")
print(f"Run ID: {artifacts.run_id}")
print(f"Parameters: {results['n_params']:,} / {PARAM_BUDGET:,}")
print(f"Training images: {results['n_train']:,}; validation images: {results['n_val']:,}")
print(f"Epochs: {results['epochs']}; compute time: {results['compute_time_min']:.2f} min; "
      f"wall time: {results['wall_time_s']/60:.2f} min; peak CUDA allocation: {results['peak_vram_mib']:.0f} MiB")
for key, value in VAL_RESULTS.items():
    print(f"{key}: epoch {value['epoch']}, {value['weights']}; plain {value['no_tta']:.2f}%, flip {value['flip_tta']:.2f}%")
if not VAL_RESULTS:
    print("No validation score for this full-data refit; use the earlier validation runs for model selection.")
print(f"Submission: {SUBMISSION['submission']}")
print(f"Format check: {SUBMISSION['format_check']}; rows: {SUBMISSION['rows']}")
print("Hidden-test accuracy: record the actual score returned by Kaggle separately.")
REVIEW_BUNDLE = export_run(RUN_DIR, include_checkpoints=False)
